# Business Entity Resolution: v6cd → unified re-ranker (US/India) + LB-0.988 France

One notebook, end to end, for a SageMaker **GPU** instance (recommended: `ml.g5.4xlarge` or larger — A10G 24 GB,
64 GB RAM, ≥ 200 GB disk). Everything that learns from labels is cross-fitted; nothing is shipped unless it beats the
current submission (v6cd) on **held-out** validation with a 95% bootstrap CI above zero and no country worse.

**What changes vs the 0.988 submission (US/India only; France rows are copied verbatim from the 0.988 file):**

| Loss seen in `results/headroom_v6cd.json` | Fix in this notebook |
|---|---|
| Dense retrieval finds 7,995 true val pairs, the 6-feature dense stacker accepts only 4,337; decision FNs at p<0.05 went 1,584 → 3,517 | **One re-ranker over base + dense candidates** (stage-2 p, CE, bi-encoder cosine, lexical, per-S1 context, per-target competition, anchor features) |
| The cross-encoder was trained only on base candidates, never on dense-style (renamed / re-addressed) pairs | **CE-u**: xlm-roberta-base trained on FIT fold-0 base pairs **plus** fold-0 dense pairs |
| 73–81% of decision FNs have a house number on one side only; p is calibrated there, so rules cannot help | **Street-without-number** and **anchor** features (does the target agree with the S1's most confident other match?) |
| Bi-encoder trained on all FIT positives → cosine is in-sample on fold 1, where the dense stacker is trained | Bi-encoder trained on **fold-0** positives only (honest cosine on fold 1 / val / test) |
| Empty / singleton errors: 320 S1 = 21% of the remaining loss; `P(empty)=∏(1-p)` assumes independence | **Learned per-S1 P(no match)** (optional variant, gated) |

**Leakage protocol:** bi-encoder + CE-u train on FIT fold 0 → re-ranker trains on FIT fold 1 (2-fold cross-fit inside
fold 1 → out-of-fold p) → decision tuned on val → held-out-half gate on val → test.

**Inputs** (edit the paths in the next cell):
1. `DATA_DIR` — competition data: `train/train_source{1,2,3}.tsv`, `train/train_ground_truth.tsv`, `test/test_source{1,2,3}.tsv`
2. `V6_WORK` — the final2 work dir restored with `artifacts/final2/restore.py` (+ `er.run prep`):
   `{train,test}_s{1,2,3}.parquet`, `train_roles.parquet`, `runs/v6/{fit_oof,val_scores,test_scores}.parquet`,
   `runs/v6/calibrators.pkl`, `runs/v6ce/{val,test}_scores_ce.parquet`; **optional** `runs/v6cd/{val,test}_scores_ce.parquet`
   (then the gate compares against your exact submitted v6cd)
3. `FR_0988` — `matching_results_0.988_balaji` (tsv or zip): France rows are taken from it unchanged
4. `VALIDATOR` — `student_resource/utils/validate_submission.py` (optional but recommended)

Every stage writes checkpoints under `BASE/nb/`; re-running the notebook skips finished stages (safe after a restart).
Approximate wall-clock on one A10G: bi-encoder 45 min · dense retrieval 45 min · CE-u training 2–2.5 h ·
CE scoring 1–1.5 h · features + re-ranker + gate 1 h → **≈ 6 h**.

In [ ]:
# ============================== CONFIG (edit paths here) ==============================
import os
from pathlib import Path

BASE = Path(os.environ.get("NB_BASE", "/home/ec2-user/SageMaker/er_opt"))
DATA_DIR = Path(os.environ.get("NB_DATA", str(BASE / "dataset")))           # contains train/ and test/
V6_WORK = Path(os.environ.get("NB_V6_WORK", str(BASE / "work_v6")))         # restored final2 artifacts
FR_0988 = Path(os.environ.get("NB_FR_0988", str(BASE / "matching_results_0.988_balaji")))
VALIDATOR = Path(os.environ.get("NB_VALIDATOR", str(BASE / "validate_submission.py")))
NB = Path(os.environ.get("NB_OUT", "/kaggle/working/nb" if Path("/kaggle/working").exists() else str(BASE / "nb")))
# On Kaggle the next-but-one cell finds DATA_DIR / V6_WORK / FR_0988 / VALIDATOR under /kaggle/input automatically.

BI_MODEL_DIR = os.environ.get("NB_BI_MODEL") or None   # set to an already trained bi-encoder dir to skip training
CE_MODEL_DIR = os.environ.get("NB_CE_MODEL") or None   # set to an already trained CE-u dir to skip training
BUILD_BASE_FROM_SCRATCH = os.environ.get("NB_FROM_SCRATCH") == "1"   # only if you do NOT have the v6/v6ce artifacts

COUNTRIES = ("US", "India")
SEED = 0
# bi-encoder (dense retrieval)
BI_BASE = os.environ.get("NB_BI_BASE", "intfloat/multilingual-e5-small")
BI_TRAIN_FOLDS = (0,)            # fold-0 positives only -> cosine is out-of-sample on fold 1 / val / test
BI_MAX_PAIRS = 1_200_000
BI_EPOCHS, BI_BS, BI_LR, BI_MAXLEN, BI_SCALE = 2, 128, 2e-5, 96, 20.0
DENSE_K, DENSE_KEEP = 10, 0.95   # top-K per S1 and source; keep 95% of new VAL true pairs (the v6cd recipe)
# cross-encoder (CE-u)
CE_BASE = os.environ.get("NB_CE_BASE", "xlm-roberta-base")
CE_EPOCHS, CE_BS, CE_LR, CE_MAXLEN = 2, 64, 2e-5, 128
CE_DENSE_POS_REPEAT, CE_DENSE_NEG_PCT = 3, 15   # dense fold-0 pairs added to CE training: positives x3, 15% of negatives
# re-ranker
P_MIN = 1e-3                     # base candidates below this stage-2 p are not re-scored (never true in practice)
RR_ROUNDS, RR_EARLY = 4000, 100
SMOKE = os.environ.get("NB_SMOKE") == "1"   # tiny synthetic run (used to test this notebook); leave False
if SMOKE:
    BI_EPOCHS, BI_BS, CE_EPOCHS, CE_BS, RR_ROUNDS = 1, 16, 1, 16, 200
    DENSE_K = 5

In [ ]:
%pip install -q "polars>=1.30,<2" "lightgbm>=4.3" "rapidfuzz>=3.6" scikit-learn scipy anyascii pyarrow "transformers>=4.40" sentencepiece

In [ ]:
# ============================== KAGGLE: find the inputs under /kaggle/input automatically ==============================
# Attach (Add Input) the competition data and the dataset(s) holding the final2 artifacts; any folder layout works.
# The artifacts are linked into the layout the code expects (work/..., work/runs/<run>/...). Run folders may be named
# "v6" / "runs_v6", "v6ce" / "runs_v6ce", "v6cd" / "runs_v6cd". Printed below: what was found and what is missing.
KAGGLE_IN = Path(os.environ.get("NB_KAGGLE_INPUT", "/kaggle/input"))
if KAGGLE_IN.exists() and not os.environ.get("NB_V6_WORK"):
    import pyarrow.parquet as pq

    def _cols(p):
        try:
            return set(pq.read_schema(p).names)
        except Exception:
            return set()

    def _run(p):
        n = p.parent.name
        return n[5:] if n.startswith("runs_") else n

    def _find(pattern, ok=lambda p: True):
        c = sorted((p for p in KAGGLE_IN.rglob(pattern) if p.is_file() and ok(p)), key=lambda p: len(str(p)))
        return c[0] if c else None

    gt = _find("train_ground_truth.tsv")
    if gt is not None:
        DATA_DIR = gt.parent.parent
    FULL = {"idx", "entity_id", "country", "business_name", "business_address", "name_norm", "addr_norm", "hno"}
    links = {f"{s}_s{k}.parquet": _find(f"{s}_s{k}.parquet", lambda p: FULL <= _cols(p)) for s in ("train", "test") for k in (1, 2, 3)}
    links["train_roles.parquet"] = _find("train_roles.parquet")
    for run_, files in (("v6", ("fit_oof.parquet", "calibrators.pkl", "val_scores.parquet", "test_scores.parquet")),
                        ("v6ce", ("val_scores_ce.parquet", "test_scores_ce.parquet")),
                        ("v6cd", ("val_scores_ce.parquet", "test_scores_ce.parquet"))):
        for f in files:
            links[f"runs/{run_}/{f}"] = _find(f, lambda p, r=run_: _run(p) == r)
    V6_WORK = NB / "work_links"
    for rel, src in links.items():
        dst = V6_WORK / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.is_symlink() or dst.exists():
            dst.unlink()
        if src is not None:
            try:
                dst.symlink_to(src)
            except OSError:            # no symlink permission: copy instead
                import shutil
                shutil.copyfile(src, dst)
    FR_0988 = _find("matching_results_0.988*") or FR_0988
    VALIDATOR = _find("validate_submission.py") or VALIDATOR
    print("DATA_DIR  :", DATA_DIR, "(found)" if gt else "(NOT FOUND: add the competition data as an input)")
    for rel, src in links.items():
        tag = "optional" if rel.startswith("runs/v6cd") else "REQUIRED"
        print(f"{rel:32s} <- {src}" if src else f"{rel:32s}    NOT FOUND ({tag})")
    print("FR_0988   :", FR_0988, FR_0988.exists())
    print("VALIDATOR :", VALIDATOR, VALIDATOR.exists())

## 1. The `er` package (final2 code, unchanged) is written next to this notebook

In [ ]:
import os
os.makedirs('er_pkg/code/business_entity_resolution/src/er', exist_ok=True)

In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/__init__.py
# (empty module)


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/admin.py
"""Label-free learning of administrative address parts for countries without a state lexicon.

Unseen countries (France) carry region / department parts that US/India lexicons do not know.
Left in place they occupy the locality slots used by blocking keys and distort locality features
(S1 writes the region, S2/S3 often the department or only the city). Two structural rules:

  R1 hierarchy : a part implied (P(part | city) >= 0.3) by >= 4 distinct frequent parts is an
                 administrative area above the city level (regions).
  R2 source-specific : a frequent S2/S3 part that never appears in the clean S1 reference and is
                 not a spelling variant of any S1 part (edit ratio < 80) is an admin variant
                 (departments replacing the region).

python -m er.admin learn --split test --out work_v4/aux/admin_parts.json
python -m er.admin renorm --split test --country France      (ER_WORK=<work dir>, ER_ADMIN_PARTS set)
"""
from __future__ import annotations

import argparse
import json
import logging
import os
from pathlib import Path

import polars as pl
from rapidfuzz import fuzz, process

from . import config as C
from .aliases import norm_part
from .normalize import STATE_MAPS

log = logging.getLogger(__name__)


def _parts(split: str, src: str, country: str) -> pl.DataFrame:
    d = (pl.scan_parquet(C.WORK / f"{split}_s{src}.parquet").filter(pl.col("country") == country)
           .select("business_address").collect().with_row_index("r"))
    return (d.select("r", pl.col("business_address").fill_null("").str.split(",").alias("p")).explode("p")
             .with_columns(pl.col("p").map_elements(norm_part, return_dtype=pl.Utf8))
             .filter((pl.col("p") != "") & ~pl.col("p").str.contains(r"\d")).unique())


def learn_admin_parts(split: str, country: str, min_freq: int = 1000, implied_by: int = 4,
                      p_implied: float = 0.3, variant_ratio: float = 80.0) -> list[str]:
    per_src = {s: _parts(split, s, country).with_columns((pl.col("r").cast(pl.Int64) * 4 + int(s)).alias("r"))
               for s in ("1", "2", "3")}
    parts = pl.concat(per_src.values())
    freq = parts.group_by("p").len("f")
    big = freq.filter(pl.col("f") >= min_freq)
    co = (parts.join(big.select("p"), on="p").rename({"p": "y"})
               .join(parts.rename({"p": "x"}), on="r").filter(pl.col("x") != pl.col("y"))
               .group_by("x", "y").len("n").join(freq.rename({"p": "y", "f": "fy"}), on="y")
               .filter(pl.col("n") / pl.col("fy") >= p_implied))
    r1 = set(co.group_by("x").len("k").filter(pl.col("k") >= implied_by)["x"].to_list())
    f1 = per_src["1"].group_by("p").len("f1")
    f23 = pl.concat([per_src["2"], per_src["3"]]).group_by("p").len("f23")
    s1_parts = f1.filter(pl.col("f1") >= 50)["p"].to_list()
    cand = (f23.filter(pl.col("f23") >= min_freq).join(f1, on="p", how="left").with_columns(pl.col("f1").fill_null(0))
               .filter(pl.col("f1") <= 0.01 * pl.col("f23")))
    r2 = set()
    for p in cand["p"].to_list():
        best = process.extractOne(p, s1_parts, scorer=fuzz.ratio) if s1_parts else None
        if best is None or best[1] < variant_ratio:
            r2.add(p)
    out = sorted(r1 | r2)
    log.info("admin parts %s: R1=%s R2=%s", country, sorted(r1), sorted(r2))
    return out


def learn(split: str, out: Path) -> dict:
    countries = pl.scan_parquet(C.WORK / f"{split}_s1.parquet").select("country").unique().collect()["country"].to_list()
    res = {c: learn_admin_parts(split, c) for c in countries if c not in STATE_MAPS}
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(res, indent=1))
    return res


def renorm(split: str, country: str, srcs: tuple[str, ...] = ("1", "2", "3"), workers: int = 8) -> None:
    """Re-normalise one country's rows (other rows untouched, row order / idx preserved) and replace
    the split files atomically (breaks hard links, so other work dirs keep their originals)."""
    from .prep import normalize_df
    for src in srcs:
        path = C.WORK / f"{split}_s{src}.parquet"
        df = pl.read_parquet(path)
        m = df["country"] == country
        raw = df.filter(m).select("entity_id", "business_name", "business_address", "country", "idx")
        new = normalize_df(raw, workers=workers).select(df.columns)
        out = pl.concat([df.filter(~m), new]).sort("idx")
        assert out.height == df.height and (out["idx"] == df["idx"]).all()
        tmp = path.with_suffix(".tmp")
        out.write_parquet(tmp)
        os.replace(tmp, path)
        log.info("renorm %s s%s: %d %s rows", split, src, new.height, country)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("what", choices=["learn", "renorm"])
    ap.add_argument("--split", default="test")
    ap.add_argument("--country", default="France")
    ap.add_argument("--out", default=None)
    ap.add_argument("--src", default="123")
    a = ap.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s")
    if a.what == "learn":
        print(json.dumps(learn(a.split, Path(a.out) if a.out else C.WORK / "aux" / "admin_parts.json"), indent=1))
    else:
        renorm(a.split, a.country, tuple(a.src))


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/aliases.py
"""Alias mining (README N4) and name-structure repair, learned only from the provided data.

- split_alias_markers: S3 contains about 105k names of the form "<pseudo-word> d/b/a <real name>"
  (also f/k/a, formerly:, DBA:, FKA). The identity-bearing name is the part after the marker.
- mine_token_aliases: target token -> S1 token, e.g. transliteration artifacts and systematic
  misspellings ("kansalting" -> "consulting", "calcutta" -> "kolkata"). Tokens are aligned inside
  each positive pair by phonetic skeleton and Jaro-Winkler, then kept only if the mapping is
  frequent and pure across pairs.
- mine_part_aliases: whole comma-part aliases in addresses (native-script / abbreviated states,
  city variants) by within-pair co-occurrence purity. No similarity is needed, so it also works
  when the transliteration is far from the English spelling ("tamizhnatu" -> "tamil nadu").
- segment: DP word-break for glued names ("meghanagold" -> ["meghana", "gold"]).
"""
from __future__ import annotations

import math
import re

import numpy as np
import polars as pl
from rapidfuzz import fuzz, process
from rapidfuzz.distance import JaroWinkler

from .text import skeleton, to_ascii

# ---------------------------------------------------------------------------
# (4) alias markers
# ---------------------------------------------------------------------------
_MARKER = re.compile(
    r"\s*(?:\b[df]\s*/\s*[bk]\s*/\s*a\b|\bformerly(?:\s+known\s+as)?\b|\bdoing\s+business\s+as\b"
    r"|\balso\s+known\s+as\b|\btrading\s+as\b|\bdba\b|\bfka\b)\s*:?\s*",
    re.IGNORECASE)
_EDGE = re.compile(r"^[\s\-:;,.(\[]+|[\s\-:;,.)\]]+$")


def split_alias_markers(name_ascii: str | None) -> tuple[str, str | None]:
    """Split "<main> d/b/a <alias>" into (main, alias). The alias is the real business name.

    A split happens only when both sides are non-empty, so real names that *start* with the
    letters ("DBA Organisers Pvt Ltd", "Aka Technologies") are left intact. "aka" is never a
    marker: in this data it only occurs as a genuine name token.
    """
    s = name_ascii or ""
    m = _MARKER.search(s)
    if not m:
        return s, None
    main, alias = _EDGE.sub("", s[:m.start()]), _EDGE.sub("", s[m.end():])
    if not main or not alias:
        return s, None
    return main, alias


# ---------------------------------------------------------------------------
# helpers
# ---------------------------------------------------------------------------
_NON_ALNUM = re.compile(r"[^a-z0-9]+")


def norm_part(part: str | None) -> str:
    return _NON_ALNUM.sub(" ", to_ascii(part or "")).strip()


def _skel_map(tokens) -> dict[str, str]:
    return {t: skeleton(t) for t in tokens}


def _purity_filter(pairs: pl.DataFrame, key: str, val: str, min_count: int, min_purity: float,
                   total: pl.DataFrame | None = None) -> dict[str, str]:
    """pairs: one row per aligned (key, val) occurrence. Keep the best val per key if it is
    frequent and pure. Purity = best count / all occurrences of key (``total`` if given)."""
    c = pairs.group_by(key, val).len("n")
    best = c.sort([key, "n"], descending=[False, True]).group_by(key, maintain_order=True).first()
    tot = total if total is not None else c.group_by(key).agg(pl.col("n").sum().alias("tot"))
    best = best.join(tot, on=key).filter((pl.col("n") >= min_count) &
                                         (pl.col("n") / pl.col("tot") >= min_purity))
    return dict(zip(best[key].to_list(), best[val].to_list()))


# ---------------------------------------------------------------------------
# (1) token aliases
# ---------------------------------------------------------------------------
def mine_token_aliases(pos_s1_core: pl.Series, pos_tgt_core: pl.Series, min_count: int = 5,
                       min_purity: float = 0.6, s1_vocab_freq: dict[str, int] | None = None,
                       max_vocab_freq: int = 2, min_sim: float = 0.80) -> dict[str, str]:
    """Map target tokens that are rare or absent in S1 to their aligned S1 token.

    Alignment within each positive pair: a target token t (not in the pair's S1 token set, and
    with S1-vocabulary frequency <= max_vocab_freq) is paired with the S1 token s of that pair
    (not present in the target) that maximises sim = max(JW(t, s), ratio(skel t, skel s)/100).
    It is kept if sim >= min_sim, or if the skeletons are equal. Across pairs, t -> s must
    occur at least min_count times and with purity >= min_purity, measured over *all*
    occurrences of t as an unmatched token.
    """
    if pos_s1_core.len() != pos_tgt_core.len():
        raise ValueError("inputs must be aligned")
    L = pl.List(pl.Utf8)
    df = pl.DataFrame({"a": pos_s1_core.cast(L), "b": pos_tgt_core.cast(L)}).with_row_index("pid")
    df = df.with_columns(pl.col("a").fill_null(pl.lit([], dtype=L)), pl.col("b").fill_null(pl.lit([], dtype=L)))
    if s1_vocab_freq is None:
        vc = df.select(pl.col("a").explode().alias("t")).drop_nulls().group_by("t").len("f")
    else:
        vc = pl.DataFrame({"t": list(s1_vocab_freq), "f": list(s1_vocab_freq.values())})
    ut = (df.select("pid", pl.col("b").list.set_difference("a").alias("t")).explode("t")
            .drop_nulls().filter(pl.col("t").str.len_chars() >= 3)
            .join(vc, on="t", how="left").filter(pl.col("f").fill_null(0) <= max_vocab_freq).drop("f"))
    us = (df.select("pid", pl.col("a").list.set_difference("b").alias("s")).explode("s")
            .drop_nulls().filter(pl.col("s").str.len_chars() >= 2))
    if ut.height == 0 or us.height == 0:
        return {}
    cand = ut.join(us, on="pid")
    sk = _skel_map(set(cand["t"].to_list()) | set(cand["s"].to_list()))
    t_l, s_l = cand["t"].to_list(), cand["s"].to_list()
    jw = process.cpdist(t_l, s_l, scorer=JaroWinkler.normalized_similarity, workers=-1, dtype=np.float32)
    ks = process.cpdist([sk[t] for t in t_l], [sk[s] for s in s_l], scorer=fuzz.ratio, workers=-1,
                        dtype=np.float32) / 100.0
    eq = np.array([sk[t] == sk[s] and len(sk[t]) >= 2 for t, s in zip(t_l, s_l)])
    cand = cand.with_columns(pl.Series("sim", np.maximum(jw, ks)), pl.Series("eq", eq))
    cand = cand.filter((pl.col("sim") >= min_sim) | pl.col("eq"))
    aligned = (cand.sort("sim", descending=True).group_by("pid", "t", maintain_order=True).first()
                   .select("t", "s"))
    total = ut.group_by("t").len("tot")
    return _purity_filter(aligned, "t", "s", min_count, min_purity, total)


# ---------------------------------------------------------------------------
# (2) comma-part aliases
# ---------------------------------------------------------------------------
def mine_part_aliases(pos_s1_addr_raw: pl.Series, pos_tgt_addr_raw: pl.Series, min_count: int = 5,
                      min_purity: float = 0.6, max_part_tokens: int = 4) -> dict[str, str]:
    """Map a normalised target comma-part to the S1 part it stands for.

    Only digit-free parts with at most max_part_tokens tokens are considered (states, cities,
    districts). For each target part p absent from the pair's S1 parts, every S1 part absent
    from the target is a co-occurrence candidate. p -> q is kept when q co-occurs with p in at
    least min_count pairs and in >= min_purity of p's occurrences. Street-like and random parts
    never reach that purity, while a consistent alias (a state in native script) does.
    """
    if pos_s1_addr_raw.len() != pos_tgt_addr_raw.len():
        raise ValueError("inputs must be aligned")
    def parts(s: pl.Series) -> pl.Series:
        return pl.Series([[q for q in dict.fromkeys(norm_part(x) for x in (v or "").split(","))
                           if q and not any(c.isdigit() for c in q) and len(q.split()) <= max_part_tokens]
                          for v in s.to_list()], dtype=pl.List(pl.Utf8))
    df = pl.DataFrame({"a": parts(pos_s1_addr_raw), "b": parts(pos_tgt_addr_raw)}).with_row_index("pid")
    up = df.select("pid", pl.col("b").list.set_difference("a").alias("p")).explode("p").drop_nulls()
    uq = df.select("pid", pl.col("a").list.set_difference("b").alias("q")).explode("q").drop_nulls()
    if up.height == 0 or uq.height == 0:
        return {}
    co = up.join(uq, on="pid").select("p", "q")
    total = up.group_by("p").len("tot")
    return _purity_filter(co, "p", "q", min_count, min_purity, total)


# ---------------------------------------------------------------------------
# (3) word segmentation
# ---------------------------------------------------------------------------
ALWAYS_OK = {"com", "net", "org", "in", "co", "fr", "llc", "inc", "ltd", "pvt", "llp", "eurl",
             "sarl", "sas", "sasu", "sci", "corp", "pc", "sa"}


def segment(token: str, vocab_freq: dict[str, int], min_freq: int = 3, max_word: int = 20,
            max_unknown: int = 3, min_known_cover: float = 0.75, self_freq: int = 20) -> list[str]:
    """Minimum-cost word break. A known word costs log(N / freq). An unknown piece of length
    <= max_unknown (initials such as "bs", "d2o") costs 12 + 4*len. The split is returned only
    if it has >= 2 pieces, known words cover >= min_known_cover of the characters, and the token
    is not itself a common word (freq >= self_freq). Otherwise the result is [token]."""
    n = len(token)
    if n < 6 or vocab_freq.get(token, 0) >= self_freq:
        return [token]
    total = float(sum(vocab_freq.values()) or 1)
    def cost(w: str) -> tuple[float, bool] | None:
        if w in ALWAYS_OK:
            return 6.0, True
        f = vocab_freq.get(w, 0)
        if f >= min_freq and len(w) >= 2:
            return math.log(total / f), True
        if len(w) <= max_unknown:
            return 12.0 + 4.0 * len(w), False
        return None
    best: list[tuple[float, int, int] | None] = [None] * (n + 1)   # (cost, prev, known_chars)
    best[0] = (0.0, -1, 0)
    for j in range(1, n + 1):
        for i in range(max(0, j - max_word), j):
            if best[i] is None:
                continue
            c = cost(token[i:j])
            if c is None:
                continue
            cand = (best[i][0] + c[0], i, best[i][2] + (j - i if c[1] else 0))
            if best[j] is None or cand[0] < best[j][0]:
                best[j] = cand
    if best[n] is None:
        return [token]
    out, j = [], n
    while j > 0:
        i = best[j][1]
        out.append(token[i:j])
        j = i
    out.reverse()
    if len(out) < 2 or best[n][2] / n < min_known_cover:
        return [token]
    return out


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/analysis.py
"""Validation analysis for a trained run: layer recall, calibration, error taxonomy, segments.

python -m er.analysis --run v1            -> work/runs/<run>/analysis.md (+ printed summary)
"""
from __future__ import annotations

import argparse
import json

import numpy as np
import polars as pl

from . import config as C
from .decide import DecisionParams, select
from .metric import macro_f05
from .pipeline import _files, _parts, id_maps, labels, load, train_roles
from .prep import load_gt


def _val_ids():
    roles = train_roles().filter(pl.col("role") == "val").select(pl.col("idx").alias("s1_idx"), "country")
    return roles


def layer_recall(val: pl.DataFrame) -> pl.DataFrame:
    """Pair recall of true matches for val S1 after blocking (A), stage-1 pruning (P) and final selection."""
    vs = _val_ids()
    lab = labels().join(vs, on="s1_idx")
    a = pl.concat([pl.scan_parquet(_files(d)).select("s1_idx", "src", "t_idx")
                   .join(vs.lazy().select("s1_idx"), on="s1_idx", how="semi").collect(engine="streaming")
                   for d in _parts("train", "A")]).with_columns(pl.lit(1).alias("inA"))
    p = pl.concat([pl.read_parquet(f, columns=["s1_idx", "src", "t_idx"]) for f in _parts("train", "P")]
                  ).join(vs.select("s1_idx"), on="s1_idx", how="semi").with_columns(pl.lit(1).alias("inP"))
    s = val.select("s1_idx", "src", "t_idx").with_columns(pl.lit(1).alias("inS"))
    m = lab.join(a, on=["s1_idx", "src", "t_idx"], how="left").join(p, on=["s1_idx", "src", "t_idx"], how="left") \
           .join(s, on=["s1_idx", "src", "t_idx"], how="left")
    out = m.group_by("country").agg(pl.len().alias("true_pairs"),
                                    pl.col("inA").is_not_null().mean().alias("recall_blocking"),
                                    pl.col("inP").is_not_null().mean().alias("recall_stage1"),
                                    pl.col("inS").is_not_null().mean().alias("recall_scored"))
    vol = pl.DataFrame({"country": ["ALL"], "true_pairs": [m.height],
                        "recall_blocking": [m["inA"].is_not_null().mean()],
                        "recall_stage1": [m["inP"].is_not_null().mean()],
                        "recall_scored": [m["inS"].is_not_null().mean()]})
    cands = {"A_pairs_per_s1": a.height / vs.height, "P_pairs_per_s1": p.height / vs.height}
    return pl.concat([out.sort("country"), vol], how="vertical_relaxed"), cands


def calibration(val: pl.DataFrame, bins: int = 10) -> pl.DataFrame:
    edges = np.linspace(0, 1, bins + 1)
    b = np.clip(np.digitize(val["p"].to_numpy(), edges) - 1, 0, bins - 1)
    return (val.with_columns(pl.Series("bin", b)).group_by("bin").agg(
                pl.len().alias("n"), pl.col("p").mean().alias("mean_p"), pl.col("y").mean().alias("frac_pos"))
              .sort("bin"))


def analyse(run: str) -> dict:
    rd = C.WORK / "runs" / run
    summ = json.loads((rd / "summary.json").read_text())
    best = summ["best"]
    val = pl.read_parquet(rd / "val_scores.parquet")
    sel = select(val, DecisionParams(margin=best["margin"], m0=best["m0"], empty_bias=best["empty_bias"]))
    s1_ids, tg_ids = id_maps("train")
    vs = _val_ids().join(s1_ids.select("s1_idx", "s1"), on="s1_idx")
    truth = load_gt().join(vs.select("s1"), on="s1")
    to_ids = lambda x: x.join(s1_ids.select("s1_idx", "s1"), on="s1_idx").join(tg_ids, on=["t_idx", "src"])
    pred = to_ids(sel).select("s1", "tid")
    rep = {"overall": macro_f05(pred, truth, vs["s1"])}
    # segments by number of true matches
    ntrue = truth.group_by("s1").len("nt")
    seg = vs.select("s1").join(ntrue, on="s1", how="left").with_columns(pl.col("nt").fill_null(0))
    rep["by_ntrue"] = {}
    for k, lo, hi in (("0", 0, 0), ("1", 1, 1), ("2-3", 2, 3), ("4-6", 4, 6), ("7+", 7, 99)):
        ids = seg.filter(pl.col("nt").is_between(lo, hi))["s1"]
        if ids.len():
            rep["by_ntrue"][k] = {**macro_f05(pred, truth, ids), "share": ids.len() / vs.height}
    lr, cands = layer_recall(val)
    rep["layer_recall"] = lr.to_dicts()
    rep["candidates"] = cands
    cal = calibration(val)
    rep["calibration"] = cal.to_dicts()
    rep["brier"] = float(((val["p"] - val["y"]) ** 2).mean())
    # error taxonomy
    fp = sel.join(val.select("s1_idx", "src", "t_idx", "y", "p"), on=["s1_idx", "src", "t_idx"]).filter(pl.col("y") == 0)
    lab = labels().join(vs.select("s1_idx"), on="s1_idx")
    fn = lab.join(sel, on=["s1_idx", "src", "t_idx"], how="anti")
    fn_scored = fn.join(val.select("s1_idx", "src", "t_idx", "p"), on=["s1_idx", "src", "t_idx"], how="left")
    rep["errors"] = {"false_pos": fp.height, "false_neg": fn.height,
                     "fn_not_in_candidates": int(fn_scored["p"].is_null().sum()),
                     "fn_scored_but_rejected": int(fn_scored["p"].is_not_null().sum()),
                     "fp_on_singletons": int(fp.join(seg.filter(pl.col("nt") == 0).join(s1_ids.select("s1", "s1_idx"), on="s1"),
                                                     on="s1_idx", how="semi").height)}
    # samples for manual inspection
    rec = {s: load("train", s, cols=["idx", "business_name", "business_address"]) for s in ("1", "2", "3")}
    def show(df: pl.DataFrame, n: int = 12) -> list[str]:
        d = df.sample(min(n, df.height), seed=0)
        d = d.join(rec["1"].rename({"idx": "s1_idx", "business_name": "n1", "business_address": "a1"}), on="s1_idx")
        tg = pl.concat([rec[s].with_columns(pl.lit(int(s), pl.UInt8).alias("src")) for s in ("2", "3")])
        d = d.join(tg.rename({"idx": "t_idx", "business_name": "n2", "business_address": "a2"}), on=["t_idx", "src"])
        return [f"{r['n1']} | {r['a1']}  <>  {r['n2']} | {r['a2']}  (p={r.get('p')})" for r in d.iter_rows(named=True)]
    rep["fp_examples"] = show(fp)
    rep["fn_rejected_examples"] = show(fn_scored.filter(pl.col("p").is_not_null()))
    rep["fn_blocking_examples"] = show(fn_scored.filter(pl.col("p").is_null()))
    (rd / "analysis.json").write_text(json.dumps(rep, indent=1, default=str))
    return rep


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", default="v1")
    a = ap.parse_args()
    pl.Config.set_tbl_width_chars(200)
    r = analyse(a.run)
    print(json.dumps({k: v for k, v in r.items() if not k.endswith("examples")}, indent=1, default=str))
    for k in ("fp_examples", "fn_rejected_examples", "fn_blocking_examples"):
        print(f"\n== {k}")
        for e in r[k]:
            print("  ", e)


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/auxfit.py
"""Fit data-derived auxiliary models once, from train positives only (plus label-free S1 names).

Outputs in work/aux/:
  filler.json       token -> P(inserted by noise)                     (er.noise, R1a)
  name_model.pkl    pseudo-word name-replacement detector             (er.noise, R1b)
  core_alias.json   name token aliases   (kansalting -> consulting)   (er.aliases, N4)
  addr_alias.json   address token aliases (calcutta -> kolkata)       (er.aliases, N4)
and helpers that apply them to normalised record tables.
"""
from __future__ import annotations

import json
import logging
from pathlib import Path

import numpy as np
import polars as pl

from . import config as C
from .aliases import mine_token_aliases
from .noise import NameReplacementModel, fit_from_training, name_replaced_score, split_filler
from .prep import load_gt

log = logging.getLogger(__name__)
AUX = C.WORK / "aux"


def fit_aux(n_pairs: int = 300_000) -> None:
    AUX.mkdir(parents=True, exist_ok=True)
    if (AUX / "addr_alias.json").exists():
        return
    filler, model = fit_from_training(C.WORK, n_pairs=n_pairs, seed=C.SEED)
    model.save(AUX / "name_model.pkl")
    s1_core = pl.scan_parquet(C.WORK / "train_s1.parquet").select("core").collect()["core"]
    vocab = dict(s1_core.explode().drop_nulls().value_counts().iter_rows())
    # filler must be *common* in clean S1 names; rare S1 tokens (rayal, skai, siti) are identity-bearing
    fill_set, foreign = split_filler(filler, {t for t, c in vocab.items() if c >= 100})
    (AUX / "filler.json").write_text(json.dumps({"filler": sorted(fill_set), "foreign": sorted(foreign),
                                                 "rates": filler}))
    gt = load_gt()
    gt = gt.sample(min(n_pairs, gt.height), seed=C.SEED + 1)
    cols = ["entity_id", "core", "atoks"]
    s1 = (pl.scan_parquet(C.WORK / "train_s1.parquet").select(cols)
            .join(gt.lazy().select(pl.col("s1").alias("entity_id")).unique(), on="entity_id").collect())
    tg = pl.concat([pl.scan_parquet(C.WORK / f"train_s{k}.parquet").select(cols)
                    .join(gt.lazy().select(pl.col("tid").alias("entity_id")), on="entity_id").collect()
                    for k in "23"])
    d = (gt.join(s1.rename({"entity_id": "s1"}), on="s1")
           .join(tg.rename({"entity_id": "tid"}), on="tid", suffix="_2"))
    core_alias = mine_token_aliases(d["core"], d["core_2"], min_count=5, min_purity=0.6, s1_vocab_freq=vocab)
    a_vocab = dict(pl.scan_parquet(C.WORK / "train_s1.parquet").select("atoks").collect()["atoks"]
                   .explode().drop_nulls().value_counts().iter_rows())
    addr_alias = mine_token_aliases(d["atoks"], d["atoks_2"], min_count=8, min_purity=0.6, s1_vocab_freq=a_vocab)
    (AUX / "core_alias.json").write_text(json.dumps(core_alias))
    (AUX / "addr_alias.json").write_text(json.dumps(addr_alias))
    log.info("aux: filler=%d foreign=%d core_alias=%d addr_alias=%d name_model pos=%s",
             len(fill_set), len(foreign), len(core_alias), len(addr_alias), model.meta.get("n_pos"))


def ocr_fix(tok: pl.Expr) -> pl.Expr:
    """Undo the generator's OCR-style digit swaps in word tokens (g1obal, c0astal, 5uper, roya1).

    Applied only to tokens with >= 3 letters whose digits are all in {0, 1, 5}, so real
    alphanumerics (b2b, 24x7, 3m, d2o, 41st) are left untouched."""
    cond = (tok.str.count_matches(r"[a-z]") >= 3) & tok.str.contains(r"[015]") & ~tok.str.contains(r"[2-46-9]")
    return pl.when(cond).then(tok.str.replace_many(["0", "1", "5"], ["o", "l", "s"])).otherwise(tok)


class Aux:
    """Loaded auxiliary models + the table-level transforms used by stage B."""

    def __init__(self, path: Path = AUX):
        f = json.loads((path / "filler.json").read_text())
        self.filler = f["filler"]
        self.core_alias = json.loads((path / "core_alias.json").read_text())
        self.addr_alias = json.loads((path / "addr_alias.json").read_text())
        self.model = NameReplacementModel.load(path / "name_model.pkl")

    def enrich(self, tab: pl.DataFrame) -> pl.DataFrame:
        """Add core_a (alias-mapped, filler-free core), atoks_a (alias-mapped address) and nrep."""
        ca, aa = self.core_alias, self.addr_alias
        tab = tab.with_columns(
            pl.col("core").list.eval(pl.element().replace(ca)).list.eval(ocr_fix(pl.element())).list.eval(
                pl.element().filter(~pl.element().is_in(self.filler))).alias("core_a"),
            pl.col("atoks").list.eval(pl.element().replace(aa)).alias("atoks_a"),
        )
        nrep = name_replaced_score(self.model, tab["core"], tab["legal"])
        return tab.with_columns(pl.Series("nrep", nrep.astype(np.float32)))


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/blocking.py
"""Candidate generation: IDF-weighted multi-key blocking, run per country and per target source.

Each record emits hashed blocking keys from several families. Name families: core
tokens, phonetic skeletons, token pairs, name x locality, prefix4 x locality. Address
families: house-number x street, house-number x locality, street x locality. A pair's
score is the sum of IDF weights of the keys it shares. Keys that are too frequent in
the target source are dropped (document-frequency cap). The name and address families
give independent retrieval paths, so name swaps are still recovered through the address.
"""
from __future__ import annotations

import logging
import math
import time

import polars as pl

log = logging.getLogger(__name__)

NAME_FAMS = ("n", "s", "np", "sp", "nl", "sl", "nx", "fn", "fs")
ADDR_FAMS = ("a1", "a2", "a4", "aa", "h0", "h1")
PURE_NAME = ("n", "s", "np", "sp", "fn", "fs")  # name-only evidence (usable when the address is missing)
SINGLE = ("n", "s", "h0")                 # single-token keys get a tighter frequency cap
NUM_FAMS = ("a1", "a2", "a4", "h0", "h1")  # house-number evidence (own candidate budget)
FULL_NAME_CAP = 2000
# coarse phonetic classes for the whole-name skeleton key (Soundex-style): sound-alike consonants
# that transliteration swaps (s/z/j, b/v/f/p, t/d, m/n) collapse to one symbol
_SOUNDEX_FROM = ["b", "f", "p", "v", "c", "g", "j", "k", "q", "s", "x", "z", "d", "t", "m", "n"]
_SOUNDEX_TO = ["1", "1", "1", "1", "2", "2", "2", "2", "2", "2", "2", "2", "3", "3", "5", "5"]                      # whole-name key: exact names with a missing address
# alphanumeric house ids ("C-346", "A/98", "19-B", "B0702") that the word tokenizer splits apart
_HID_JOIN1 = r"\b([a-z])\s?[-/]?\s?(\d{1,5})\b"
_HID_JOIN2 = r"\b(\d{1,5})\s?[-/]\s?([a-z])\b"
_HID = r"\b[a-z]\d{1,5}\b|\b\d{1,5}[a-z]\b"
FAM_ID = {f: i for i, f in enumerate(NAME_FAMS + ADDR_FAMS)}
BIT = {i: 1 << i for i in FAM_ID.values()}


def _tok(df: pl.DataFrame, col: str, limit: int, min_len: int = 2, prefix: int = 0) -> pl.DataFrame:
    """(idx, h): u64 hashes of the first ``limit`` distinct tokens of list column ``col``.

    Tokens are hashed immediately so compound keys are built from integers, never from
    concatenated strings (keeps key construction for ~5M-row tables within a few GB)."""
    e = (df.select("idx", pl.col(col).list.unique(maintain_order=True).list.head(limit).alias("t"))
           .explode("t").drop_nulls("t").filter(pl.col("t").str.len_chars() >= max(min_len, prefix)))
    if prefix:
        e = e.with_columns(pl.col("t").str.slice(0, prefix)).unique()
    return e.select("idx", pl.col("t").hash(seed=len(col) * 31 + prefix).alias("h"))


def _single(a: pl.DataFrame, fam: str) -> pl.DataFrame:
    return a.select("idx", pl.col("h").hash(seed=FAM_ID[fam]).alias("key"),
                    pl.lit(FAM_ID[fam], pl.UInt8).alias("fam"))


def _pairs(a: pl.DataFrame, b: pl.DataFrame, fam: str, ordered: bool = False) -> pl.DataFrame:
    j = a.join(b, on="idx", suffix="_b")
    if ordered:
        j = j.filter(pl.col("h") < pl.col("h_b"))
    return j.select("idx", pl.struct("h", "h_b").hash(seed=100 + FAM_ID[fam]).alias("key"),
                    pl.lit(FAM_ID[fam], pl.UInt8).alias("fam"))


def iter_key_families(df: pl.DataFrame):
    """Yield (family, keys) one family at a time: (idx, key u64, fam u8) frames."""
    core = _tok(df, "core", 4)
    skel = _tok(df, "skel", 4, 2)
    skel3 = _tok(df, "skel", 4, 3)
    loc = _tok(df, "loc", 2)
    street = _tok(df, "street", 3, 3)
    nums = _tok(df, "nums", 3, 1)
    pre = _tok(df, "core", 4, prefix=4)
    # street and locality tokens pooled: classification differs when a record has no number
    addr = (df.select("idx", pl.concat_list(pl.col("street").list.head(3), pl.col("loc").list.head(3))
                      .list.unique(maintain_order=True).list.head(5).alias("aa"))
              .pipe(_tok, "aa", 5, 3))
    if "business_address" in df.columns:
        hid = (df.select("idx", pl.col("business_address").fill_null("").str.to_lowercase()
                         .str.replace_all(_HID_JOIN1, "$1$2").str.replace_all(_HID_JOIN2, "$1$2")
                         .str.extract_all(_HID).list.eval(pl.element().str.replace(r"^([a-z]+)0+(\d)", "$1$2"))
                         .list.unique().list.head(3).alias("t"))
                 .explode("t").drop_nulls("t")
                 .select("idx", pl.col("t").hash(seed=977).alias("h")))
        yield "h0", _single(hid, "h0")
        yield "h1", _pairs(hid, loc, "h1")
    full = (df.select("idx", pl.col("core").list.sort().list.join(" ").alias("t"))
              .filter(pl.col("t").str.len_chars() >= 3).select("idx", pl.col("t").hash(seed=991).alias("h")))
    yield "fn", _single(full, "fn")
    # whole-name phonetic skeleton: transliterations / vowel typos ("siti midiya" ~ "city media" -> "md st")
    fskel = (df.select("idx", pl.col("skel").list.eval(
                    pl.element().filter(pl.element().str.len_chars() >= 2)
                    .str.replace_many(_SOUNDEX_FROM, _SOUNDEX_TO))
                              .list.unique().list.sort().list.join(" ").alias("t"))
               .filter(pl.col("t").str.len_chars() >= 4).select("idx", pl.col("t").hash(seed=997).alias("h")))
    yield "fs", _single(fskel, "fs")
    yield "n", _single(core, "n")
    yield "s", _single(skel3, "s")
    yield "np", _pairs(core, core, "np", ordered=True)
    yield "sp", _pairs(skel, skel, "sp", ordered=True)
    yield "nl", _pairs(core, loc, "nl")
    yield "sl", _pairs(skel3, loc, "sl")
    yield "nx", _pairs(pre, loc, "nx")
    yield "a1", _pairs(nums, street, "a1")
    yield "a2", _pairs(nums, loc, "a2")
    yield "a4", _pairs(nums, nums, "a4", ordered=True)
    yield "aa", _pairs(addr, addr, "aa", ordered=True)


def make_keys(df: pl.DataFrame) -> pl.DataFrame:
    """df needs idx, core, skel, loc, street, nums. Returns (idx, key u64, fam u8)."""
    return pl.concat([k for _, k in iter_key_families(df)])


def _weighted_target_keys(tgt: pl.DataFrame, s1_keys: pl.DataFrame, n_t: int, cap_single: int,
                          cap_pair: int) -> pl.DataFrame:
    """Target keys with IDF weights, built one family at a time (bounded memory). Document
    frequencies are computed over *all* targets before pruning; keys absent from S1 are then
    dropped because they can never produce a candidate pair."""
    out = []
    for fam, part in iter_key_families(tgt):
        cap = cap_single if fam in SINGLE else (FULL_NAME_CAP if fam in ("fn", "fs") else cap_pair)
        dfk = (part.group_by("key").len("df").filter(pl.col("df") <= cap)
                   .join(s1_keys.filter(pl.col("fam") == FAM_ID[fam]).select("key"), on="key", how="semi"))
        dfk = dfk.with_columns((math.log(n_t + 1) - pl.col("df").cast(pl.Float32).log()).alias("w"),
                               (1.0 / pl.col("df").cast(pl.Float32)).alias("inv_df"))
        out.append(part.join(dfk.select("key", "w", "inv_df"), on="key"))
        del part, dfk
    return pl.concat(out)


def block(s1: pl.DataFrame, tgt: pl.DataFrame, cap_single: int = 300, cap_pair: int = 1500,
          top_k: int = 50, top_name: int = 20, top_pure: int = 15, top_addr: int = 25, top_num: int = 15,
          chunk: int = 50_000, join_budget: int = 20_000_000, on_chunk=None) -> pl.DataFrame | None:
    """Candidate pairs (s1_idx, t_idx, bs, bs_name, bs_addr, nkeys, nfam, brank, ...).

    Keeps the union of the top_k by total score and the top_name / top_addr by the name-only
    and address-only scores, so name-only matches (null target address) and name-swap
    matches (address only) never compete with pairs supported by both.

    Memory: if ``on_chunk(g, chunk_id)`` is given, each S1 chunk's candidates are handed to it
    (e.g. written to disk) instead of being accumulated, and None is returned. A chunk always
    holds *all* candidates of its S1 records, so per-S1 window features are exact inside it."""
    s1k = make_keys(s1)
    tk = _weighted_target_keys(tgt, s1k.select("key", "fam").unique(), max(tgt.height, 1),
                               cap_single, cap_pair)
    # cost-aware chunking: an S1's join cost is the sum of the target frequencies of its keys, so
    # each chunk is capped at ``join_budget`` joined rows (and ``chunk`` S1) whatever the names are
    kdf = tk.group_by("key", "fam").len("df")
    s1k = s1k.join(kdf.select("key", "fam"), on=["key", "fam"], how="semi")  # keys that can match
    cost = (s1k.join(kdf, on=["key", "fam"]).group_by("idx").agg(pl.col("df").sum().alias("c"))
               .sort("idx").with_row_index("pos"))
    cost = cost.with_columns(pl.max_horizontal(pl.col("c").cum_sum() // join_budget,
                                               pl.col("pos") // chunk).alias("ch"))
    # make chunk ids monotone and dense
    cost = cost.with_columns(pl.col("ch").cum_max().rank("dense").cast(pl.UInt32).alias("ch"))
    s1k = s1k.join(cost.select("idx", "ch"), on="idx")
    del kdf, cost
    a_ids, p_ids = [FAM_ID[f] for f in ADDR_FAMS], [FAM_ID[f] for f in PURE_NAME]
    n_ids = [FAM_ID[f] for f in NUM_FAMS]
    out, n_pairs = [], 0
    for (ch,), sub in s1k.group_by("ch"):
        t0 = time.time()
        j = (sub.drop("ch").rename({"idx": "s1_idx"}).join(tk.rename({"idx": "t_idx"}), on=["key", "fam"])
                .with_columns(pl.col("fam").is_in(a_ids).cast(pl.Float32).alias("a"),
                              pl.col("fam").is_in(p_ids).cast(pl.Float32).alias("p"),
                              pl.col("fam").is_in(n_ids).cast(pl.Float32).alias("q"),
                              pl.col("fam").replace_strict(BIT, return_dtype=pl.UInt16).alias("bit")))
        g = j.group_by("s1_idx", "t_idx").agg(
            pl.col("w").sum().alias("bs"),
            (pl.col("w") * (1 - pl.col("a"))).sum().alias("bs_name"),
            (pl.col("w") * pl.col("a")).sum().alias("bs_addr"),
            (pl.col("w") * pl.col("p")).sum().alias("bs_pure"),
            (pl.col("w") * pl.col("q")).sum().alias("bs_num"),
            pl.len().alias("nkeys"),
            # supervised meta-blocking edge weights (GSM, VLDB'22): ARCS and per-family evidence
            pl.col("inv_df").sum().alias("arcs"),
            pl.col("a").sum().alias("nk_addr"),
            pl.col("bit").bitwise_or().alias("fam_mask"),
        ).with_columns(pl.col("fam_mask").bitwise_count_ones().alias("nfam"),
                       (pl.col("nkeys") - pl.col("nk_addr")).alias("nk_name"))
        j_rows = j.height
        del j
        g = (g.with_columns(
                pl.col("bs").rank("ordinal", descending=True).over("s1_idx").alias("brank"),
                pl.col("bs_name").rank("ordinal", descending=True).over("s1_idx").alias("brank_n"),
                pl.col("bs_addr").rank("ordinal", descending=True).over("s1_idx").alias("brank_a"),
                pl.col("bs_pure").rank("ordinal", descending=True).over("s1_idx").alias("brank_p"),
                pl.col("bs_num").rank("ordinal", descending=True).over("s1_idx").alias("brank_q"))
              .filter((pl.col("brank") <= top_k) | ((pl.col("brank_n") <= top_name) & (pl.col("bs_name") > 0))
                      | ((pl.col("brank_a") <= top_addr) & (pl.col("bs_addr") > 0))
                      | ((pl.col("brank_p") <= top_pure) & (pl.col("bs_pure") > 0))
                      | ((pl.col("brank_q") <= top_num) & (pl.col("bs_num") > 0))))
        n_pairs += g.height
        log.debug("chunk %s: join=%d pairs=%d %.1fs", ch, j_rows, g.height, time.time() - t0)
        if on_chunk is None:
            out.append(g)
        else:
            on_chunk(g, int(ch))
    log.info("block s1=%d tgt=%d pairs=%d", s1.height, tgt.height, n_pairs)
    return (pl.concat(out) if out else pl.DataFrame()) if on_chunk is None else None


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/compare.py
"""Acceptance gate: paired comparison of two runs on the SAME validation S1 entities.

python -m er.compare --a work/runs/v0 --b work_v3/runs/v3
For each run: apply its tuned decision to its validation scores, compute per-S1 F0.5 (exact
challenge rules), then a paired bootstrap over S1 (B resamples) of the difference b - a,
overall and per country, plus precision / recall / singleton-F deltas. The candidate (b) is
accepted only if the 95% CI of dF0.5 is entirely > 0 overall AND in every country.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import polars as pl

from .decide import DecisionParams, select, select_exact
from .prep import load_gt

BETA2 = 0.25


def _decision(rd: Path) -> dict:
    for name in ("decision_refined.json", "decision.json"):
        if (rd / name).exists():
            return json.loads((rd / name).read_text())["best"]
    return json.loads((rd / "summary.json").read_text())["best"]


def per_s1(rd: Path, work: Path) -> pl.DataFrame:
    """Per-S1 (s1, country, f, tp, np, nt) for one run on its validation S1 set."""
    val = pl.read_parquet(rd / "val_scores.parquet")
    b = _decision(rd)
    fn = select_exact if b.get("kind") == "exact" else select
    sel = fn(val, DecisionParams(margin=b["margin"], m0=b["m0"], empty_bias=b["empty_bias"], coh=b.get("coh", 0.0)))
    roles = pl.read_parquet(work / "train_roles.parquet").filter(pl.col("role") == "val")
    s1 = pl.read_parquet(work / "train_s1.parquet", columns=["idx", "entity_id"]).rename(
        {"idx": "s1_idx", "entity_id": "s1"})
    tg = pl.concat([pl.read_parquet(work / f"train_s{s}.parquet", columns=["idx", "entity_id"])
                      .rename({"idx": "t_idx", "entity_id": "tid"}).with_columns(pl.lit(int(s), pl.UInt8).alias("src"))
                    for s in ("2", "3")])
    base = roles.select(pl.col("idx").alias("s1_idx"), "country").join(s1, on="s1_idx")
    pred = sel.join(tg, on=["t_idx", "src"]).join(s1, on="s1_idx").select("s1", "tid").unique()
    truth = load_gt().join(base.select("s1"), on="s1").unique()
    tp = truth.join(pred, on=["s1", "tid"]).group_by("s1").len("tp")
    df = (base.join(truth.group_by("s1").len("nt"), on="s1", how="left")
              .join(pred.group_by("s1").len("np"), on="s1", how="left")
              .join(tp, on="s1", how="left").fill_null(0))
    p = pl.col("tp") / pl.col("np")
    r = pl.col("tp") / pl.col("nt")
    return df.with_columns(
        pl.when((pl.col("nt") == 0) & (pl.col("np") == 0)).then(1.0)
          .when((pl.col("nt") == 0) | (pl.col("np") == 0) | (pl.col("tp") == 0)).then(0.0)
          .otherwise((1 + BETA2) * p * r / (BETA2 * p + r)).alias("f"))


def compare(a: Path, b: Path, work_a: Path, work_b: Path, n_boot: int = 2000, seed: int = 0) -> dict:
    A, B = per_s1(a, work_a), per_s1(b, work_b)
    m = A.select("s1", "country", pl.col("f").alias("fa"), pl.col("tp").alias("tpa"), pl.col("np").alias("npa"),
                 "nt").join(B.select("s1", pl.col("f").alias("fb"), pl.col("tp").alias("tpb"),
                                     pl.col("np").alias("npb")), on="s1")
    rng = np.random.default_rng(seed)
    out = {"n_common_s1": m.height}
    for name, g in [("ALL", m), *[(c, d) for (c,), d in m.group_by("country")]]:
        d = (g["fb"] - g["fa"]).to_numpy()
        idx = rng.integers(0, len(d), size=(n_boot, len(d)))
        boots = d[idx].mean(axis=1)
        sing = g.filter(pl.col("nt") == 0)
        out[name] = {
            "f05_a": round(float(g["fa"].mean()), 5), "f05_b": round(float(g["fb"].mean()), 5),
            "delta": round(float(d.mean()), 5),
            "ci95": [round(float(np.percentile(boots, 2.5)), 5), round(float(np.percentile(boots, 97.5)), 5)],
            "p_b_better": round(float((boots > 0).mean()), 4),
            "precision_a": round(float((g["tpa"] / g["npa"]).drop_nans().mean()), 5),
            "precision_b": round(float((g["tpb"] / g["npb"]).drop_nans().mean()), 5),
            "recall_a": round(float((g.filter(pl.col("nt") > 0)["tpa"] / g.filter(pl.col("nt") > 0)["nt"]).mean()), 5),
            "recall_b": round(float((g.filter(pl.col("nt") > 0)["tpb"] / g.filter(pl.col("nt") > 0)["nt"]).mean()), 5),
            "singleton_f_a": round(float(sing["fa"].mean()), 5) if sing.height else None,
            "singleton_f_b": round(float(sing["fb"].mean()), 5) if sing.height else None,
        }
    out["ACCEPT"] = all(out[k]["ci95"][0] > 0 for k in out if isinstance(out[k], dict))
    return out


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--a", required=True)
    ap.add_argument("--b", required=True)
    ap.add_argument("--work-a", required=True)
    ap.add_argument("--work-b", required=True)
    x = ap.parse_args()
    print(json.dumps(compare(Path(x.a), Path(x.b), Path(x.work_a), Path(x.work_b)), indent=1))


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/config.py
"""Paths and global settings. Override the data/work roots with ER_DATA / ER_WORK."""
import os
from pathlib import Path

ROOT = Path(__file__).resolve().parents[4]  # repo root
DATA = Path(os.environ.get("ER_DATA", ROOT / "student_resource" / "dataset"))
WORK = Path(os.environ.get("ER_WORK", ROOT / "work"))
OUT = Path(os.environ.get("ER_OUT", ROOT / "output"))
WORK.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

WORKERS = int(os.environ.get("ER_WORKERS", max(1, (os.cpu_count() or 4) - 2)))
SEED = 42

# Fast-path switches (v0): block only fit/val S1 on train, and drop competition (tx_*) features,
# which would be biased when not every S1 is blocked. Full-strength runs set both to 0.
TRAIN_ROLES_ONLY = os.environ.get("ER_TRAIN_ROLES_ONLY", "1") == "1"
NO_TX = os.environ.get("ER_NO_TX", "1") == "1"
CAP_SINGLE = int(os.environ.get("ER_CAP_SINGLE", 300))
CAP_PAIR = int(os.environ.get("ER_CAP_PAIR", 600))
JOIN_BUDGET = int(os.environ.get("ER_JOIN_BUDGET", 20_000_000))   # max joined rows per blocking chunk
# blocking candidate budgets per S1 per source: total / name / address / pure-name / house-number
BUDGETS = dict(top_k=int(os.environ.get("ER_TOP_K", 50)), top_name=int(os.environ.get("ER_TOP_NAME", 20)),
               top_addr=int(os.environ.get("ER_TOP_ADDR", 25)), top_pure=int(os.environ.get("ER_TOP_PURE", 15)),
               top_num=int(os.environ.get("ER_TOP_NUM", 15)))


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/cross.py
"""Cross-encoder data exchange (GPU route): export raw-text pairs, import transformer scores.

A multilingual transformer (mdeberta-v3-base, MIT, 278M) is fine-tuned on raw "name | address"
pairs on a Kaggle GPU (scripts/kaggle/ce_kernel.py) and scores the uncertain band of a run.
Leakage-safe protocol:
  * transformer training : FIT S1 of fold 0 only (labels from train ground truth)
  * stacker training     : FIT S1 of fold 1 (stage-2 OOF p + transformer score), er.cross stack
  * gate                 : validation S1, never seen by either model

python -m er.cross export-train --run v4fr --out <dir>          (ER_WORK=<work dir>)
python -m er.cross export-band  --run v6 --out <dir> [--lo 0.005 --hi 0.995]
"""
from __future__ import annotations

import argparse
import logging

import polars as pl

from . import config as C

log = logging.getLogger(__name__)


def _texts(split: str) -> dict[str, pl.DataFrame]:
    out = {}
    for s in ("1", "2", "3"):
        d = pl.read_parquet(C.WORK / f"{split}_s{s}.parquet", columns=["idx", "business_name", "business_address"])
        out[s] = d.select(pl.col("idx").cast(pl.UInt32),
                          (pl.col("business_name").fill_null("") + " | " + pl.col("business_address").fill_null("")).alias("t"))
    return out


def _attach(pairs: pl.DataFrame, split: str) -> pl.DataFrame:
    tx = _texts(split)
    p = pairs.with_columns(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32))
    a = p.join(tx["1"].rename({"idx": "s1_idx", "t": "a"}), on="s1_idx")
    parts = [a.filter(pl.col("src") == int(s)).join(tx[s].rename({"idx": "t_idx", "t": "b"}), on="t_idx") for s in ("2", "3")]
    return pl.concat(parts)


def export_train(run: str, out: str, neg_easy_frac: float = 0.05, seed: int = 0) -> dict:
    from .pipeline import train_roles
    fold0 = train_roles().filter((pl.col("role") == "fit") & (pl.col("fold") == 0)).select(pl.col("idx").cast(pl.UInt32).alias("s1_idx"))
    f = pl.read_parquet(C.WORK / "runs" / run / "fit_oof.parquet").with_columns(pl.col("s1_idx").cast(pl.UInt32))
    f = f.join(fold0, on="s1_idx", how="semi")
    h = pl.col("t_idx").hash(seed) % 1000
    # every uncertain / hard pair, a quarter of the easy positives (p >= 0.99), 5% of the easy negatives
    keep = (((pl.col("y") == 1) & ((pl.col("oof") < 0.99) | (h < 250)))
            | ((pl.col("y") == 0) & ((pl.col("oof") >= 0.01) | (h < int(neg_easy_frac * 1000)))))
    ctry = pl.read_parquet(C.WORK / "train_s1.parquet", columns=["idx", "country"]).select(pl.col("idx").cast(pl.UInt32).alias("s1_idx"), "country")
    d = _attach(f.filter(keep).select("s1_idx", "src", "t_idx", pl.col("y").cast(pl.Int8).alias("label"), "oof"), "train")
    d = d.join(ctry, on="s1_idx")      # lets a kernel train on one country only (transfer / LOCO test)
    d = d.sample(fraction=1.0, shuffle=True, seed=seed)
    path = f"{out}/ce_train.parquet"
    d.write_parquet(path)
    info = {"rows": d.height, "pos": int(d["label"].sum()), "path": path}
    log.info("export-train %s", info)
    return info


def _fit1_calibrated(rd) -> pl.DataFrame:
    """FIT fold-1 pairs with the run's per-source isotonic calibration applied to the raw OOF score,
    so they are on the same scale as val/test scores (which are calibrated at prediction time)."""
    import pickle

    import numpy as np
    from .pipeline import train_roles
    fold1 = train_roles().filter((pl.col("role") == "fit") & (pl.col("fold") == 1)).select(pl.col("idx").cast(pl.UInt32).alias("s1_idx"))
    f = (pl.read_parquet(rd / "fit_oof.parquet").with_columns(pl.col("s1_idx").cast(pl.UInt32))
           .join(fold1, on="s1_idx", how="semi"))
    cal = pickle.load(open(rd / "calibrators.pkl", "rb"))
    raw, src = f["oof"].to_numpy(), f["src"].to_numpy()
    p = raw.copy()
    for k, c in cal.items():
        m = src == k
        if m.any():
            p[m] = c.predict(raw[m])
    return f.with_columns(pl.Series("p", p.astype(np.float32))).select("s1_idx", "src", "t_idx", "y", "p")


def export_band(run: str, out: str, lo: float = 0.005, hi: float = 0.995) -> dict:
    """Pairs whose stage-2 probability is uncertain: fit fold-1 (stacker), val (gate), test (submission)."""
    rd = C.WORK / "runs" / run
    band = (pl.col("p") >= lo) & (pl.col("p") <= hi)
    fit1 = _fit1_calibrated(rd)
    val = pl.read_parquet(rd / "val_scores.parquet").select("s1_idx", "src", "t_idx", "p")
    test = pl.read_parquet(rd / "test_scores.parquet").select("s1_idx", "src", "t_idx", "p")
    info = {}
    for name, d, split in (("fit1", fit1, "train"), ("val", val, "train"), ("test", test, "test")):
        d = _attach(d.filter(band).select("s1_idx", "src", "t_idx", "p"), split)
        d.write_parquet(f"{out}/ce_band_{name}.parquet")
        info[name] = d.height
    log.info("export-band %s", info)
    return info


def export_biencoder(out: str, max_pairs: int = 1_200_000, seed: int = 0) -> dict:
    """Positive (S1 text, target text) pairs of FIT S1 (never val) for a contrastive bi-encoder
    (in-batch negatives). Used for dense candidate retrieval (blocking recall)."""
    from .pipeline import labels, train_roles
    fit = train_roles().filter(pl.col("role") == "fit").select(pl.col("idx").cast(pl.UInt32).alias("s1_idx"), "country")
    lab = labels().select(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32)).join(fit, on="s1_idx")
    if lab.height > max_pairs:
        lab = lab.sample(max_pairs, seed=seed)
    d = _attach(lab, "train").sample(fraction=1.0, shuffle=True, seed=seed)
    d.select("s1_idx", "src", "t_idx", "country", "a", "b").write_parquet(f"{out}/bi_train.parquet")
    info = {"pairs": d.height, "by_country": dict(d.group_by("country").len().rows())}
    log.info("export-biencoder %s", info)
    return info


def export_bi_eval(out: str, country: str = "India", decomp_run: str = "v5") -> dict:
    """Retrieval evaluation set: val S1 texts of one country, all its train target texts, and its true val
    pairs flagged by whether key-based blocking found them (runs/<decomp_run>/val_recall_decomp.parquet)."""
    from .pipeline import train_roles
    val = train_roles().filter((pl.col("role") == "val") & (pl.col("country") == country)).select(pl.col("idx").cast(pl.UInt32))
    tx = _texts("train")
    tx["1"].join(val, on="idx").rename({"t": "text"}).write_parquet(f"{out}/bi_eval_s1.parquet")
    parts = []
    for s_ in ("2", "3"):
        c = pl.read_parquet(C.WORK / f"train_s{s_}.parquet", columns=["idx", "country"]).filter(pl.col("country") == country)
        parts.append(tx[s_].join(c.select(pl.col("idx").cast(pl.UInt32)), on="idx")
                     .select(pl.lit(int(s_)).cast(pl.UInt8).alias("src"), "idx", pl.col("t").alias("text")))
    tg = pl.concat(parts)
    tg.write_parquet(f"{out}/bi_eval_tg.parquet")
    tr = (pl.read_parquet(C.WORK / "runs" / decomp_run / "val_recall_decomp.parquet")
            .filter(pl.col("country") == country).select("s1_idx", "src", "t_idx", "inA"))
    tr.write_parquet(f"{out}/bi_eval_truth.parquet")
    info = {"s1": val.height, "targets": tg.height, "true_pairs": tr.height, "blocking_missed": int((~tr["inA"]).sum())}
    log.info("export-bi-eval %s", info)
    return info


def export_pseudo(scores: str, split: str, out: str, hi: float = 0.97, lo: float = 0.02, neg_per_pos: float = 1.0,
                  country: str | None = None, seed: int = 0) -> dict:
    """Label-free transformer adaptation data for a target country: confident positives (p >= hi and the best
    S1 for their target) and confident negatives (p <= lo, sampled neg_per_pos per positive), with raw text."""
    d = pl.read_parquet(scores).select("s1_idx", "src", "t_idx", "p").with_columns(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32))
    if country:
        c = pl.read_parquet(C.WORK / f"{split}_s1.parquet", columns=["idx", "country"]).filter(pl.col("country") == country)
        d = d.join(c.select(pl.col("idx").cast(pl.UInt32).alias("s1_idx")), on="s1_idx", how="semi")
    d = d.with_columns((pl.col("p") >= pl.col("p").max().over("src", "t_idx")).alias("best"))
    pos = d.filter((pl.col("p") >= hi) & pl.col("best")).with_columns(pl.lit(1, pl.Int8).alias("label"))
    neg = d.filter(pl.col("p") <= lo)
    neg = neg.sample(min(neg.height, int(pos.height * neg_per_pos)), seed=seed).with_columns(pl.lit(0, pl.Int8).alias("label"))
    x = _attach(pl.concat([pos, neg]).select("s1_idx", "src", "t_idx", "label"), split).sample(fraction=1.0, shuffle=True, seed=seed)
    x.write_parquet(f"{out}/ce_pseudo.parquet")
    info = {"pos": pos.height, "neg": neg.height}
    log.info("export-pseudo %s", info)
    return info


def export_country_adapt(run: str, country: str, out: str, n_pseudo: int = 150_000, hi: float = 0.97, lo: float = 0.02,
                         blo: float = 0.005, bhi: float = 0.995, seed: int = 0) -> dict:
    """Everything the GPU needs to adapt the cross-encoder to an unseen test country (validated on India-as-unseen:
    +0.0094 [0.0087, 0.0101] F0.5): label-free pseudo pairs of the country (confident run decisions), the country's
    uncertain test band, and the run's labelled FIT fold-1 band (stacker training). Also keeps the country's full
    test scores and the fold-1 scores (p, y) locally for stacking and the final decision."""
    rd = C.WORK / "runs" / run
    cid = (pl.read_parquet(C.WORK / "test_s1.parquet", columns=["idx", "country"]).filter(pl.col("country") == country)
             .select(pl.col("idx").cast(pl.UInt32).alias("s1_idx")))
    t = (pl.scan_parquet(rd / "test_scores.parquet").select(*KEYS, "p")
           .with_columns(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32))
           .join(cid.lazy(), on="s1_idx", how="semi").collect())
    t.write_parquet(f"{out}/country_test_scores.parquet")
    t2 = t.with_columns((pl.col("p") >= pl.col("p").max().over("src", "t_idx")).alias("best"))
    pos = t2.filter((pl.col("p") >= hi) & pl.col("best"))
    pos = pos.sample(min(pos.height, n_pseudo), seed=seed).with_columns(pl.lit(1, pl.Int8).alias("label"))
    neg = t2.filter(pl.col("p") <= lo)
    neg = neg.sample(min(neg.height, n_pseudo), seed=seed).with_columns(pl.lit(0, pl.Int8).alias("label"))
    _attach(pl.concat([pos, neg]).select(*KEYS, "label"), "test").sample(fraction=1.0, shuffle=True, seed=seed).write_parquet(f"{out}/ce_pseudo.parquet")
    band = t.filter((pl.col("p") >= blo) & (pl.col("p") <= bhi))
    _attach(band.select(*KEYS, "p"), "test").write_parquet(f"{out}/ce_band_test.parquet")
    f1 = _fit1_calibrated(rd).with_columns(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32))
    f1.write_parquet(f"{out}/fit1_scores.parquet")
    _attach(f1.filter((pl.col("p") >= blo) & (pl.col("p") <= bhi)).select(*KEYS, "p"), "train").write_parquet(f"{out}/ce_band_fit1.parquet")
    info = {"country_pairs": t.height, "pseudo_pos": pos.height, "pseudo_neg": neg.height, "test_band": band.height,
            "fit1_pairs": f1.height, "fit1_band": int(((f1["p"] >= blo) & (f1["p"] <= bhi)).sum())}
    log.info("export-country-adapt %s", info)
    return info


KEYS = ["s1_idx", "src", "t_idx"]
STACK_FEATS = ["lp", "lce", "src", "ce_rank_s1", "ce_gap_t", "p_rank_s1", "dce"]


def _stack_feats(d: pl.DataFrame) -> pl.DataFrame:
    lg = lambda c: (pl.col(c).clip(1e-6, 1 - 1e-6) / (1 - pl.col(c).clip(1e-6, 1 - 1e-6))).log()
    return d.with_columns(lg("p").alias("lp"), lg("ce").alias("lce"),
                          pl.col("ce").rank("ordinal", descending=True).over("s1_idx", "src").cast(pl.Float32).alias("ce_rank_s1"),
                          pl.col("p").rank("ordinal", descending=True).over("s1_idx", "src").cast(pl.Float32).alias("p_rank_s1"),
                          # competition on the transformer score: lead over the best other S1 for the same target
                          # winner: lead over the runner-up; loser: (negative) deficit to the winner; sole candidate: ce
                          pl.when(pl.len().over("src", "t_idx") == 1).then(pl.col("ce"))
                            .when(pl.col("ce") >= pl.col("ce").max().over("src", "t_idx"))
                            .then(pl.col("ce") - pl.col("ce").top_k(2).min().over("src", "t_idx"))
                            .otherwise(pl.col("ce") - pl.col("ce").max().over("src", "t_idx")).alias("ce_gap_t"),
                          (pl.col("ce") - pl.col("p")).alias("dce"))


def stack(run: str, ce_dir: str, out_run: str, tag: str = "ce") -> dict:
    """Train a small LightGBM stacker on FIT fold-1 band pairs (stage-2 calibrated OOF p + transformer
    score), apply it to the val/test band; pairs outside the band keep their stage-2 probability.
    Writes runs/<out_run>/{val_scores.parquet, val_scores_<tag>.parquet, test_scores_<tag>.parquet} and
    links the run's models/calibrators/summary so tune / predict / compare work unchanged."""
    import os

    import lightgbm as lgb
    from .pipeline import labels
    rd, od = C.WORK / "runs" / run, C.WORK / "runs" / out_run
    od.mkdir(parents=True, exist_ok=True)
    for f in ("summary.json", "calibrators.pkl", "stage2_f0.lgb", "stage2_f1.lgb"):
        if not (od / f).exists():
            os.link(rd / f, od / f)
    cast = lambda d: d.with_columns(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32))
    ce = {n: cast(pl.read_parquet(f"{ce_dir}/ce_scores_{n}.parquet")) for n in ("fit1", "val", "test")}
    lab = cast(labels()).with_columns(pl.lit(1, pl.Int8).alias("yy"))
    tr = _stack_feats(cast(_fit1_calibrated(rd)).join(ce["fit1"], on=KEYS)).join(lab, on=KEYS, how="left")
    y = tr["yy"].fill_null(0).to_numpy()
    prm = dict(objective="binary", learning_rate=0.05, num_leaves=31, min_data_in_leaf=200, feature_fraction=0.9,
               bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, seed=C.SEED, num_threads=C.WORKERS)
    m = lgb.train(prm, lgb.Dataset(tr.select(STACK_FEATS).to_numpy(), y), 400)
    m.save_model(str(od / "stacker.lgb"))
    info = {"fit1_band_rows": tr.height, "fit1_pos": int(y.sum())}
    for name, fname in (("val", "val_scores.parquet"), ("test", "test_scores.parquet")):
        base = cast(pl.read_parquet(rd / fname).select(*KEYS, "p"))
        bd = _stack_feats(base.join(ce[name], on=KEYS))
        bd = bd.with_columns(pl.Series("p2", m.predict(bd.select(STACK_FEATS).to_numpy()).astype("float32")))
        out = (base.join(bd.select(*KEYS, "p2"), on=KEYS, how="left")
                   .with_columns(pl.coalesce("p2", "p").alias("p")).drop("p2"))
        out.write_parquet(od / f"{name}_scores_{tag}.parquet")
        if name == "val":
            out.write_parquet(od / "val_scores.parquet")      # er.compare reads val_scores.parquet
        info[f"{name}_band"] = bd.height
    log.info("stack %s", info)
    return info


def loco_prepare(run: str, out: str, max_rows: int = 3_000_000, lo: float = 0.005, hi: float = 0.995) -> dict:
    """Transfer test (India plays the unseen country): stage-2 LightGBM trained on US fit fold-0 only ->
    scores US fit fold-1 (stacker training) and India val (evaluation). Writes the uncertain band of both
    with raw text for the US-only transformer (kernel er-ce-loco): ce_band_fit1 / ce_band_val."""
    import json

    import lightgbm as lgb
    import numpy as np
    from .pipeline import S2_PARAMS, _scan_b, labels, train_roles
    feats = json.loads((C.WORK / "runs" / run / "summary.json").read_text())["features"]
    roles = train_roles().select(pl.col("idx").cast(pl.UInt32).alias("s1_idx"), "role", "fold", "country")
    lab = labels().select(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32)).with_columns(pl.lit(1, pl.Int8).alias("y"))

    def load(flt):
        return (_scan_b("train").with_columns(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32))
                .join(roles.lazy().filter(flt), on="s1_idx").join(lab.lazy(), on=KEYS, how="left")
                .with_columns(pl.col("y").fill_null(0))
                .select(*KEYS, "y", *[pl.col(c).cast(pl.Float32) for c in feats if c not in KEYS]).collect(engine="streaming"))
    tr = load((pl.col("country") == "US") & (pl.col("role") == "fit") & (pl.col("fold") == 0))
    if tr.height > max_rows:
        tr = tr.sample(max_rows, seed=C.SEED)
    prm = {**S2_PARAMS, "num_threads": C.WORKERS}
    m = lgb.train(prm, lgb.Dataset(tr.select(feats).to_numpy(), tr["y"].to_numpy()), 800)
    del tr
    info = {}
    for name, flt, split in (("fit1", (pl.col("country") == "US") & (pl.col("role") == "fit") & (pl.col("fold") == 1), "train"),
                             ("val", (pl.col("country") == "India") & (pl.col("role") == "val"), "train")):
        d = load(flt)
        d = d.select(*KEYS, "y").with_columns(pl.Series("p", m.predict(d.select(feats).to_numpy()).astype(np.float32)))
        d.write_parquet(f"{out}/loco_scores_{name}.parquet")
        b = _attach(d.filter((pl.col("p") >= lo) & (pl.col("p") <= hi)).select(*KEYS, "p"), split)
        b.write_parquet(f"{out}/ce_band_{name}.parquet")
        info[name] = {"rows": d.height, "band": b.height}
    log.info("loco-prepare %s", info)
    return info


def loco_eval(dir_: str, run: str, n_boot: int = 2000) -> dict:
    """India val macro F0.5: US-only stage 2 vs US-only stage 2 + US-only transformer (stacker trained on
    US fit fold 1). Decision: the run's tuned parameters. Paired bootstrap over India val S1."""
    import json

    import lightgbm as lgb
    import numpy as np
    from .decide import DecisionParams, select
    from .pipeline import id_maps, train_roles
    from .prep import load_gt
    cast = lambda d: d.with_columns(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32))
    sc = {n: cast(pl.read_parquet(f"{dir_}/loco_scores_{n}.parquet")) for n in ("fit1", "val")}
    ce = {n: cast(pl.read_parquet(f"{dir_}/ce_scores_{n}.parquet")) for n in ("fit1", "val")}
    tr = _stack_feats(sc["fit1"].join(ce["fit1"], on=KEYS))
    prm = dict(objective="binary", learning_rate=0.05, num_leaves=31, min_data_in_leaf=200, feature_fraction=0.9,
               bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, seed=C.SEED, num_threads=C.WORKERS)
    m = lgb.train(prm, lgb.Dataset(tr.select(STACK_FEATS).to_numpy(), tr["y"].to_numpy()), 400)
    v = sc["val"]
    bd = _stack_feats(v.join(ce["val"], on=KEYS))
    bd = bd.with_columns(pl.Series("p2", m.predict(bd.select(STACK_FEATS).to_numpy()).astype("float32")))
    v2 = v.join(bd.select(*KEYS, "p2"), on=KEYS, how="left").with_columns(pl.coalesce("p2", "p").alias("p")).drop("p2")
    b = json.loads((C.WORK / "runs" / run / "decision.json").read_text())["best"]
    prm_d = DecisionParams(margin=b["margin"], m0=b["m0"], empty_bias=b["empty_bias"], coh=b.get("coh", 0.0))
    s1_ids, tg_ids = id_maps("train")
    ids = (train_roles().filter((pl.col("role") == "val") & (pl.col("country") == "India"))
           .select(pl.col("idx").alias("s1_idx")).join(s1_ids.select("s1_idx", "s1"), on="s1_idx"))
    gt = load_gt()
    from .selftrain import _per_s1_f
    tg = tg_ids.with_columns(pl.col("t_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8))
    idc = ids.with_columns(pl.col("s1_idx").cast(pl.UInt32))
    fa = _per_s1_f(v.select(*KEYS, "p"), prm_d, idc, tg, gt)
    fb = _per_s1_f(v2.select(*KEYS, "p"), prm_d, idc, tg, gt)
    d = fb - fa
    rng = np.random.default_rng(0)
    boots = d[rng.integers(0, len(d), size=(n_boot, len(d)))].mean(1)
    res = {"india_unseen_f05_stage2_only": round(float(fa.mean()), 5), "with_transformer": round(float(fb.mean()), 5),
           "delta": round(float(d.mean()), 5), "ci95": [round(float(np.percentile(boots, 2.5)), 5), round(float(np.percentile(boots, 97.5)), 5)]}
    res["TRANSFERS"] = res["ci95"][0] > 0
    log.info("loco-eval %s", res)
    return res


def country_adapt_predict(run: str, adapt_dir: str, country: str, base_output: str, out_name: str) -> dict:
    """Finish the unseen-country adaptation: stacker (calibrated stage-2 p + adapted cross-encoder features) trained on
    the run's labelled FIT fold-1 band, applied to the country's uncertain test band; the run's own tuned decision on
    the country's full candidate set. Writes output/<out_name>: the country's rows re-decided (same candidate set),
    every other country's rows copied unchanged from output/<base_output>. Validator run at the end."""
    import json
    import subprocess
    import sys

    import lightgbm as lgb
    from .decide import DecisionParams, select, select_exact
    cast = lambda d: d.with_columns(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32))
    f1 = cast(pl.read_parquet(f"{adapt_dir}/fit1_scores.parquet"))
    ce_f1 = cast(pl.read_parquet(f"{adapt_dir}/ce_scores_fit1.parquet"))
    ce_te = cast(pl.read_parquet(f"{adapt_dir}/ce_scores_test.parquet"))
    tr = _stack_feats(f1.join(ce_f1, on=KEYS))
    m = lgb.train(dict(objective="binary", learning_rate=0.05, num_leaves=31, min_data_in_leaf=200, feature_fraction=0.9,
                       bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, seed=C.SEED, num_threads=C.WORKERS),
                  lgb.Dataset(tr.select(STACK_FEATS).to_numpy(), tr["y"].to_numpy()), 400)
    t = cast(pl.read_parquet(f"{adapt_dir}/country_test_scores.parquet"))
    bd = _stack_feats(t.join(ce_te, on=KEYS))
    bd = bd.with_columns(pl.Series("p2", m.predict(bd.select(STACK_FEATS).to_numpy()).astype("float32")))
    t2 = t.join(bd.select(*KEYS, "p2"), on=KEYS, how="left").with_columns(pl.coalesce("p2", "p").alias("p")).drop("p2")
    b = json.loads((C.WORK / "runs" / run / "decision.json").read_text())["best"]
    fn = select_exact if b.get("kind") == "exact" else select
    sel = fn(t2, DecisionParams(margin=b["margin"], m0=b["m0"], empty_bias=b["empty_bias"], coh=b.get("coh", 0.0)))
    s1 = pl.read_parquet(C.WORK / "test_s1.parquet", columns=["idx", "entity_id", "country"]).select(
        pl.col("idx").cast(pl.UInt32).alias("s1_idx"), pl.col("entity_id").alias("source1_entity_id"), "country")
    tg = pl.concat([pl.read_parquet(C.WORK / f"test_s{k}.parquet", columns=["idx", "entity_id"]).select(
        pl.col("idx").cast(pl.UInt32).alias("t_idx"), pl.lit(k).cast(pl.UInt8).alias("src"), pl.col("entity_id").alias("tid")) for k in (2, 3)])
    ids = sel.join(s1, on="s1_idx").join(tg, on=["t_idx", "src"])
    ctry = s1.filter(pl.col("country") == country).select("source1_entity_id")
    mr = (ctry.join(ids.group_by("source1_entity_id").agg(pl.col("tid").sort().str.join(",").alias("matched_entity_ids")), on="source1_entity_id", how="left")
              .with_columns(pl.col("matched_entity_ids").fill_null("")))
    od = C.OUT / out_name
    od.mkdir(parents=True, exist_ok=True)
    for fname, col, new in (("matching_results.tsv", "matched_entity_ids", mr), ("candidate_pairs.tsv", "candidate_entity_ids", None)):
        base = pl.read_csv(C.OUT / base_output / fname, separator="\t", infer_schema=False)
        if new is not None:
            keep = base.join(ctry, on="source1_entity_id", how="anti")
            outp = base.select("source1_entity_id").join(pl.concat([keep.with_columns(pl.col(col).fill_null("")), new.rename({"matched_entity_ids": col})]), on="source1_entity_id", maintain_order="left")
        else:
            outp = base      # same candidate set: the run scored exactly these pairs
        outp.write_csv(od / fname, separator="\t", quote_style="never")
    r = subprocess.run([sys.executable, str(C.ROOT / "student_resource" / "utils" / "validate_submission.py"), "--matching", str(od / "matching_results.tsv"),
                        "--candidate", str(od / "candidate_pairs.tsv"), "--test-dir", str(C.DATA / "test")], capture_output=True, text=True, encoding="utf-8")
    out = (r.stdout + r.stderr).strip().splitlines()
    return {"band": bd.height, "selected_pairs": sel.height, "country_rows": ctry.height,
            "mean_matches": float(mr["matched_entity_ids"].str.split(",").list.eval(pl.element().filter(pl.element() != "")).list.len().mean()),
            "validator": out[-1] if out else r.returncode}


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("what", choices=["export-train", "export-band", "stack", "loco-prepare", "loco-eval", "export-biencoder", "export-bi-eval", "export-pseudo", "export-country-adapt", "country-predict"])
    ap.add_argument("--ce", default=None)
    ap.add_argument("--scores", default=None)
    ap.add_argument("--split", default="train")
    ap.add_argument("--country", default=None)
    ap.add_argument("--base-output", default=None)
    ap.add_argument("--out-run", default=None)
    ap.add_argument("--run", default=None)
    ap.add_argument("--out", default=None)
    ap.add_argument("--lo", type=float, default=0.005)
    ap.add_argument("--hi", type=float, default=0.995)
    a = ap.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s")
    if a.what == "export-train":
        print(export_train(a.run, a.out))
    elif a.what == "export-band":
        print(export_band(a.run, a.out, a.lo, a.hi))
    elif a.what == "stack":
        print(stack(a.run, a.ce, a.out_run))
    elif a.what == "country-predict":
        print(country_adapt_predict(a.run, a.out, a.country, a.base_output, a.out_run))
    elif a.what == "export-country-adapt":
        print(export_country_adapt(a.run, a.country, a.out))
    elif a.what == "export-pseudo":
        print(export_pseudo(a.scores, a.split, a.out, country=a.country))
    elif a.what == "export-bi-eval":
        print(export_bi_eval(a.out))
    elif a.what == "export-biencoder":
        print(export_biencoder(a.out))
    elif a.what == "loco-prepare":
        print(loco_prepare(a.run, a.out))
    else:
        print(loco_eval(a.out, a.run))


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/decide.py
"""Decision layer: pair probabilities -> per-S1 match sets.

N1  target exclusivity: every S2/S3 record belongs to at most one S1 (verified on train),
    so a target is only kept for the S1 that wants it most (optionally with a margin).
N2  expected-F0.5 set selection: for each S1, candidates are sorted by probability and the
    prefix size k (k = 0 means "no match") that maximises expected F0.5 is chosen.
    E[F(k)] ~ (1+b2) * sum_{i<=k} p_i / (b2 * (sum_i p_i + m0) + k)   (ratio of expectations)
    E[F(0)] = P(no match) = prod_i (1 - p_i) * (1 - q_miss)
    m0 / q_miss absorb true matches that blocking never retrieved; all knobs are tuned on
    validation against the exact macro-F0.5 metric.
N3  one-to-one coherent probabilities (optional, coh > 0): pairwise probabilities ignore that a
    target belongs to at most one S1. Under that constraint and independent pairwise evidence the
    posterior is P(target -> S1 i) = o_i / (1 + sum_j o_j), o = p / (1 - p) (bipartite record
    linkage, Sadinle 2017). coh scales the competitors' odds: 0 = raw p, 1 = full posterior.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import polars as pl

B2 = 0.25


@dataclass
class DecisionParams:
    margin: float = 0.0       # exclusivity: required lead over the runner-up S1 for a target
    floor: float = 0.0        # never select a candidate below this probability
    m0: float = 0.0           # expected matches outside the candidate set
    empty_bias: float = 1.0   # multiplier on P(no match) (<1 discourages empty predictions)
    exclusive: bool = True
    coh: float = 0.0          # one-to-one coherence strength (N3)


def coherent(d: pl.DataFrame, coh: float) -> pl.DataFrame:
    """Replace p by the one-to-one coherent posterior (N3); coh = 0 returns d unchanged."""
    if coh <= 0:
        return d
    pc = pl.col("p").clip(1e-6, 1 - 1e-6)
    d = d.with_columns((pc / (1 - pc)).alias("_o")).with_columns(pl.col("_o").sum().over("src", "t_idx").alias("_so"))
    return (d.with_columns((pl.col("_o") / (1 + pl.col("_o") + coh * (pl.col("_so") - pl.col("_o")))).alias("p"))
             .drop("_o", "_so"))


def select(df: pl.DataFrame, prm: DecisionParams, p: str = "p") -> pl.DataFrame:
    """df: s1_idx, src, t_idx, p. Returns selected (s1_idx, src, t_idx) pairs."""
    d = coherent(df.select("s1_idx", "src", "t_idx", pl.col(p).alias("p")), prm.coh)
    if prm.exclusive:
        d = d.with_columns(
            pl.col("p").rank("ordinal", descending=True).over("src", "t_idx").alias("_tr"),
            pl.col("p").top_k(2).min().over("src", "t_idx").alias("_p2"),
            pl.len().over("src", "t_idx").alias("_tn"))
        lead = pl.when(pl.col("_tn") > 1).then(pl.col("p") - pl.col("_p2")).otherwise(pl.col("p"))
        # candidates losing the target keep their probability mass for E[F] but cannot be chosen
        d = d.with_columns(((pl.col("_tr") == 1) & (lead >= prm.margin)).alias("_ok"))
    else:
        d = d.with_columns(pl.lit(True).alias("_ok"))
    d = d.with_columns(pl.when(pl.col("_ok")).then(pl.col("p")).otherwise(0.0).alias("_q"))
    d = d.sort(["s1_idx", "_q"], descending=[False, True]).with_columns(
        pl.col("_q").cum_sum().over("s1_idx").alias("_cs"),
        pl.int_range(1, pl.len() + 1).over("s1_idx").alias("_k"),
        pl.col("p").sum().over("s1_idx").alias("_tot"),
        (1 - pl.col("p")).log().sum().over("s1_idx").alias("_lp0"))
    d = d.with_columns(((1 + B2) * pl.col("_cs") / (B2 * (pl.col("_tot") + prm.m0) + pl.col("_k")))
                       .alias("_ef"))
    best = d.group_by("s1_idx").agg(
        pl.col("_ef").max().alias("_efmax"),
        pl.col("_k").get(pl.col("_ef").arg_max()).alias("_kbest"),
        (pl.col("_lp0").first().exp() * prm.empty_bias).alias("_ef0"))
    d = d.join(best, on="s1_idx").filter(
        (pl.col("_efmax") > pl.col("_ef0")) & (pl.col("_k") <= pl.col("_kbest"))
        & pl.col("_ok") & (pl.col("p") >= prm.floor))
    return d.select("s1_idx", "src", "t_idx")


def threshold_select(df: pl.DataFrame, thr: float, p: str = "p") -> pl.DataFrame:
    """Baseline: global threshold + exclusivity (argmax S1 per target)."""
    d = df.filter(pl.col(p) >= thr).with_columns(
        pl.col(p).rank("ordinal", descending=True).over("src", "t_idx").alias("_tr"))
    return d.filter(pl.col("_tr") == 1).select("s1_idx", "src", "t_idx")


# ---------------------------------------------------------------------------
# Exact Bayes-optimal expected-F selection (independent candidates)
# ---------------------------------------------------------------------------
def _pb_prefix(P: np.ndarray) -> list[np.ndarray]:
    """Poisson-binomial distributions of #true among the first k columns, k = 0..n.
    P: (G, n) probabilities. Returns list of (G, k+1) arrays."""
    G, n = P.shape
    out = [np.ones((G, 1))]
    for k in range(n):
        prev, p = out[-1], P[:, k:k + 1]
        nxt = np.zeros((G, k + 2))
        nxt[:, :-1] += prev * (1 - p)
        nxt[:, 1:] += prev * p
        out.append(nxt)
    return out


def expected_f_exact(P: np.ndarray, beta2: float = B2, m0: float = 0.0) -> np.ndarray:
    """E[F_beta] of predicting the top-k (columns already sorted descending), k = 0..n.

    Candidates are independent Bernoulli(p). ``m0`` is the expected number of true matches the
    candidate set missed (added to the recall denominator). k = 0 scores 1 only if there are
    no true matches: P(all false) * exp(-m0).  Returns (G, n+1)."""
    G, n = P.shape
    pre = _pb_prefix(P)
    suf = _pb_prefix(P[:, ::-1])          # suf[j] = distribution over the last j columns
    ef = np.zeros((G, n + 1))
    ef[:, 0] = pre[n][:, 0] * np.exp(-m0)
    for k in range(1, n + 1):
        A, B = pre[k], suf[n - k]          # A: #true in top-k, B: #true in the rest
        a = np.arange(k + 1)[:, None]
        b = np.arange(n - k + 1)[None, :]
        f = (1 + beta2) * a / (k + beta2 * (a + b + m0))          # (k+1, n-k+1)
        ef[:, k] = np.einsum("ga,ab,gb->g", A, f, B)
    return ef


def select_exact(df: pl.DataFrame, prm: DecisionParams, p: str = "p", max_n: int = 40) -> pl.DataFrame:
    """Like ``select`` but maximises the exact expected F0.5 per S1 (vectorised by set size)."""
    d = coherent(df.select("s1_idx", "src", "t_idx", pl.col(p).alias("p")), prm.coh)
    if prm.exclusive:
        d = d.with_columns(
            pl.col("p").rank("ordinal", descending=True).over("src", "t_idx").alias("_tr"),
            pl.col("p").top_k(2).min().over("src", "t_idx").alias("_p2"),
            pl.len().over("src", "t_idx").alias("_tn"))
        lead = pl.when(pl.col("_tn") > 1).then(pl.col("p") - pl.col("_p2")).otherwise(pl.col("p"))
        d = d.with_columns(((pl.col("_tr") == 1) & (lead >= prm.margin)).alias("_ok"))
    else:
        d = d.with_columns(pl.lit(True).alias("_ok"))
    # candidates that cannot be chosen keep their mass in the "rest" (they may still be true)
    d = d.with_columns(pl.when(pl.col("_ok")).then(pl.col("p")).otherwise(0.0).alias("_q"))
    d = d.sort(["s1_idx", "_ok", "p"], descending=[False, True, True]).with_columns(
        pl.int_range(1, pl.len() + 1).over("s1_idx").alias("_k"),
        pl.col("_ok").sum().over("s1_idx").alias("_nok"),
        pl.len().over("s1_idx").alias("_n"))
    d = d.filter(pl.col("_k") <= max_n).with_columns(pl.len().over("s1_idx").alias("_n"))
    picks = []
    for (n,), g in d.group_by("_n"):
        g = g.sort(["s1_idx", "_k"])
        ids = g["s1_idx"].to_numpy().reshape(-1, n)[:, 0]
        P = np.clip(g["p"].to_numpy().reshape(-1, n), 0.0, 1.0)
        nok = g["_nok"].to_numpy().reshape(-1, n)[:, 0]
        ef = expected_f_exact(P, m0=prm.m0)
        ef[:, 0] *= prm.empty_bias
        ks = np.arange(n + 1)[None, :]
        ef[ks > nok[:, None]] = -1.0                      # only choosable candidates can be picked
        kbest = ef.argmax(1)
        picks.append(pl.DataFrame({"s1_idx": ids, "_kbest": kbest}))
    best = pl.concat(picks) if picks else pl.DataFrame({"s1_idx": [], "_kbest": []})
    d = d.join(best, on="s1_idx").filter((pl.col("_k") <= pl.col("_kbest")) & pl.col("_ok")
                                         & (pl.col("p") >= prm.floor))
    return d.select("s1_idx", "src", "t_idx")


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/dense.py
"""Dense-retrieval candidate augmentation (GPU route, lever 3): export for the Kaggle kernel er-dense.

Streams the raw source TSVs (row order = our idx, exactly as er.prep assigns it) to compact parquet for US/India,
plus the run's existing candidate keys (so the GPU only scores NEW pairs), roles and ground truth.
python -m er.dense export --run v4fr --out <dir>      (ER_WORK=<work dir of the run>)
"""
from __future__ import annotations

import argparse

import polars as pl

from . import config as C


def export(run: str, out: str, countries=("US", "India")) -> dict:
    info = {}
    for split in ("train", "test"):
        for s in ("1", "2", "3"):
            path = C.DATA / split / f"{split}_source{s}.tsv"
            (pl.scan_csv(path, separator="\t", quote_char=None, infer_schema=False, encoding="utf8-lossy")
               .with_row_index("idx").with_columns(pl.col("idx").cast(pl.UInt32), pl.col("country").fill_null(""))
               .filter(pl.col("country").is_in(list(countries)))
               .select("idx", "entity_id", "country", (pl.col("business_name").fill_null("") + " | " + pl.col("business_address").fill_null("")).alias("text"))
               .sink_parquet(f"{out}/{split}_s{s}.parquet"))
            info[f"{split}_s{s}"] = pl.scan_parquet(f"{out}/{split}_s{s}.parquet").select(pl.len()).collect().item()
    rd = C.WORK / "runs" / run
    k = lambda f: (pl.scan_parquet(rd / f).select(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32)))
    k("fit_oof.parquet").sink_parquet(f"{out}/cand_fit.parquet")
    k("val_scores.parquet").sink_parquet(f"{out}/cand_val.parquet")
    k("test_scores.parquet").sink_parquet(f"{out}/cand_test.parquet")
    pl.read_parquet(C.WORK / "train_roles.parquet").write_parquet(f"{out}/train_roles.parquet")
    pl.scan_csv(C.DATA / "train" / "train_ground_truth.tsv", separator="\t", quote_char=None, infer_schema=False).sink_parquet(f"{out}/train_gt.parquet")
    return info


DENSE_FEATS = ["cos", "rank", "gap1", "ce", "lce", "src"]


def merge(base_run: str, dense_dir: str, out_run: str, tag: str = "ce") -> dict:
    """Score the new dense pairs with a LightGBM trained on FIT fold-1 dense pairs (labels from the ground truth)
    and add them to the base run's stacked scores -> runs/<out_run>/{val_scores, val_scores_<tag>, test_scores_<tag>}.
    Model files are hard-linked so tune / predict / compare work unchanged."""
    import os

    import lightgbm as lgb
    KEYS = ["s1_idx", "src", "t_idx"]
    rd, od = C.WORK / "runs" / base_run, C.WORK / "runs" / out_run
    od.mkdir(parents=True, exist_ok=True)
    for f in ("summary.json", "calibrators.pkl", "stage2_f0.lgb", "stage2_f1.lgb"):
        if not (od / f).exists():
            os.link(rd / f, od / f)
    lce = (pl.col("ce").clip(1e-6, 1 - 1e-6) / (1 - pl.col("ce").clip(1e-6, 1 - 1e-6))).log().alias("lce")
    ld = lambda n: pl.read_parquet(f"{dense_dir}/dense_{n}.parquet").with_columns(lce, pl.col("src").cast(pl.Float32).alias("srcf"))
    fe = [c if c != "src" else "srcf" for c in DENSE_FEATS]
    tr = ld("fit1")
    m = lgb.train(dict(objective="binary", learning_rate=0.05, num_leaves=31, min_data_in_leaf=100, feature_fraction=0.9,
                       bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, seed=C.SEED, num_threads=C.WORKERS),
                  lgb.Dataset(tr.select(fe).to_numpy(), tr["y"].to_numpy()), 300)
    m.save_model(str(od / "dense_stacker.lgb"))
    info = {"fit1_pairs": tr.height, "fit1_pos": int(tr["y"].sum())}
    for name in ("val", "test"):
        d = ld(name)
        d = d.select(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32),
                     pl.Series("p", m.predict(d.select(fe).to_numpy()).astype("float32")))
        base = pl.read_parquet(rd / f"{name}_scores_{tag}.parquet").select(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32), "p")
        u = pl.concat([base, d.join(base, on=KEYS, how="anti")])
        u.write_parquet(od / f"{name}_scores_{tag}.parquet")
        if name == "val":
            u.write_parquet(od / "val_scores.parquet")
        info[f"{name}_added"] = d.height
        info[f"{name}_added_p>0.5"] = int((d["p"] > 0.5).sum())
    return info


def add_test_pairs(run: str, dense_file: str, out_run: str, tag: str = "ce") -> dict:
    """Add extra test-only dense pairs (e.g. France, er-fr-dense) to a merged run: scored with the run's own dense
    stacker (trained on labelled FIT fold-1 US/India dense pairs), same decision. Validation files are shared unchanged."""
    import os
    import shutil

    import lightgbm as lgb
    KEYS = ["s1_idx", "src", "t_idx"]
    rd, od = C.WORK / "runs" / run, C.WORK / "runs" / out_run
    od.mkdir(parents=True, exist_ok=True)
    for f in ("summary.json", "calibrators.pkl", "stage2_f0.lgb", "stage2_f1.lgb", "dense_stacker.lgb",
              "val_scores.parquet", f"val_scores_{tag}.parquet"):
        if not (od / f).exists():
            os.link(rd / f, od / f)
    for f in ("decision.json", f"decision_{tag}.json"):
        shutil.copyfile(rd / f, od / f)
    m = lgb.Booster(model_file=str(rd / "dense_stacker.lgb"))
    lce = (pl.col("ce").clip(1e-6, 1 - 1e-6) / (1 - pl.col("ce").clip(1e-6, 1 - 1e-6))).log().alias("lce")
    d = pl.read_parquet(dense_file).with_columns(lce, pl.col("src").cast(pl.Float32).alias("srcf"))
    fe = [c if c != "src" else "srcf" for c in DENSE_FEATS]
    d = d.select(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32),
                 pl.Series("p", m.predict(d.select(fe).to_numpy()).astype("float32")))
    base = pl.read_parquet(rd / f"test_scores_{tag}.parquet").select(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32), "p")
    new = d.join(base, on=KEYS, how="anti")
    pl.concat([base, new]).write_parquet(od / f"test_scores_{tag}.parquet")
    return {"added": new.height, "added_p>0.5": int((new["p"] > 0.5).sum())}


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("what", choices=["export", "merge", "add-test"])
    ap.add_argument("--run", required=True)
    ap.add_argument("--out", default=None)
    ap.add_argument("--dense", default=None)
    ap.add_argument("--out-run", default=None)
    a = ap.parse_args()
    if a.what == "export":
        print(export(a.run, a.out))
    elif a.what == "merge":
        print(merge(a.run, a.dense, a.out_run))
    else:
        print(add_test_pairs(a.run, a.dense, a.out_run))


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/diagnostics.py
"""Layer-level diagnostics: blocking recall/volume, per-family contribution, miss inspection.

python -m er.diagnostics blocking --n 40000
"""
from __future__ import annotations

import argparse
import time

import polars as pl

from .blocking import block
from .pipeline import load
from .prep import load_gt

SHOW = ["entity_id", "business_name", "business_address"]


def blocking_report(n: int = 40000, seed: int = 1, show_miss: int = 6, **block_kw) -> pl.DataFrame:
    gt = load_gt()
    s1_all = load("train", "1").sample(n, seed=seed)
    rows = []
    for country in s1_all["country"].unique().sort().to_list():
        a = s1_all.filter(pl.col("country") == country)
        ids1 = a.select(pl.col("idx").alias("s1_idx"), pl.col("entity_id").alias("s1"))
        cands, tgts = [], []
        for src in ("2", "3"):
            t = load("train", src, country)
            t0 = time.time()
            r = block(a, t, **block_kw)
            dt = time.time() - t0
            cands.append(r.join(ids1, on="s1_idx").join(
                t.select(pl.col("idx").alias("t_idx"), pl.col("entity_id").alias("tid")), on="t_idx"))
            tgts.append(t.select(SHOW))
            print(f"{country} S{src}: {dt:.0f}s, {r.height / a.height:.1f} cands/S1")
        c = pl.concat(cands)
        m = load_gt_subset(gt, a).join(c, on=["s1", "tid"], how="left")
        hit = pl.col("bs").is_not_null()
        rec = {"country": country, "n_s1": a.height, "cands_per_s1": c.height / a.height,
               "recall": m.select(hit.mean()).item(),
               "recall_top40": m.select((pl.col("brank") <= 40).fill_null(False).mean()).item(),
               "recall_top20": m.select((pl.col("brank") <= 20).fill_null(False).mean()).item(),
               "recall_top10": m.select((pl.col("brank") <= 10).fill_null(False).mean()).item(),
               "only_name_budget": m.select((hit & (pl.col("brank") > 40) & (pl.col("brank_n") <= 15)).mean()).item(),
               "only_addr_budget": m.select((hit & (pl.col("brank") > 40) & (pl.col("brank_a") <= 15)).mean()).item()}
        rows.append(rec)
        if show_miss:
            tg = pl.concat(tgts).rename({"entity_id": "tid", "business_name": "n2", "business_address": "a2"})
            miss = (m.filter(~hit).head(show_miss).join(tg, on="tid")
                     .join(a.select(pl.col("entity_id").alias("s1"), pl.col("business_name").alias("n1"),
                                    pl.col("business_address").alias("a1")), on="s1"))
            for q in miss.iter_rows(named=True):
                print(f"  MISS {q['n1']} | {q['a1']}  <>  {q['n2']} | {q['a2']}")
    out = pl.DataFrame(rows)
    print(out)
    return out


def miss_anatomy(country: str = "India", src: str = "2", n: int = 5000, seed: int = 3,
                 cap_single: int = 300, cap_pair: int = 1500) -> pl.DataFrame:
    """Why does blocking miss true pairs? Classify every missed pair:
    no_shared_key | only_capped_keys (all shared keys too frequent) | cut_by_budget (scored, ranked out).
    Also reports the rank the pair would have had without budgets and its per-family shared keys."""
    from .blocking import FAM_ID, SINGLE, block, make_keys
    inv = {v: k for k, v in FAM_ID.items()}
    gt = load_gt()
    a = load("train", "1", country).sample(n, seed=seed)
    t = load("train", src, country)
    ids_t = t.select(pl.col("idx").alias("t_idx"), pl.col("entity_id").alias("tid"))
    ids_a = a.select(pl.col("idx").alias("s1_idx"), pl.col("entity_id").alias("s1"))
    truth = gt.join(ids_a, on="s1").join(ids_t, on="tid")
    got = block(a, t, cap_single=cap_single, cap_pair=cap_pair).select("s1_idx", "t_idx", pl.lit(1).alias("hit"))
    miss = truth.join(got, on=["s1_idx", "t_idx"], how="left").filter(pl.col("hit").is_null())
    tk = make_keys(t)
    dfk = tk.group_by("key", "fam").len("df").with_columns(
        (pl.col("df") <= pl.when(pl.col("fam").is_in([FAM_ID[f] for f in SINGLE])).then(cap_single)
         .otherwise(cap_pair)).alias("ok"))
    ka = make_keys(a.join(miss.select(pl.col("s1_idx").alias("idx")).unique(), on="idx"))
    kt = tk.join(miss.select(pl.col("t_idx").alias("idx")).unique(), on="idx")
    sh = (miss.select("s1_idx", "t_idx")
          .join(ka.rename({"idx": "s1_idx"}), on="s1_idx")
          .join(kt.rename({"idx": "t_idx"}), on=["t_idx", "key", "fam"])
          .join(dfk, on=["key", "fam"]))
    per = sh.group_by("s1_idx", "t_idx").agg(
        pl.col("ok").sum().alias("n_ok"), pl.len().alias("n_shared"),
        pl.col("fam").filter(pl.col("ok")).alias("fams_ok"))
    m = miss.join(per, on=["s1_idx", "t_idx"], how="left").with_columns(
        pl.when(pl.col("n_shared").is_null()).then(pl.lit("no_shared_key"))
          .when(pl.col("n_ok") == 0).then(pl.lit("only_capped_keys"))
          .otherwise(pl.lit("cut_by_budget")).alias("why"))
    print(f"{country} S{src}: true={truth.height} missed={miss.height} "
          f"recall={1 - miss.height / max(truth.height, 1):.4f}")
    print(m.group_by("why").len().sort("len", descending=True))
    fam_counts = (m.filter(pl.col("why") == "cut_by_budget").explode("fams_ok")
                    .group_by("fams_ok").len().with_columns(pl.col("fams_ok").replace_strict(inv)))
    print("families present in budget-cut misses:", dict(fam_counts.iter_rows()))
    show = (m.join(a.select(pl.col("idx").alias("s1_idx"), pl.col("business_name").alias("n1"),
                             pl.col("business_address").alias("a1")), on="s1_idx")
              .join(t.select(pl.col("idx").alias("t_idx"), pl.col("business_name").alias("n2"),
                             pl.col("business_address").alias("a2")), on="t_idx"))
    for why in ("no_shared_key", "only_capped_keys"):
        for q in show.filter(pl.col("why") == why).head(4).iter_rows(named=True):
            print(f"  [{why}] {q['n1']} | {q['a1']}  <>  {q['n2']} | {q['a2']}")
    return m


def load_gt_subset(gt: pl.DataFrame, s1: pl.DataFrame) -> pl.DataFrame:
    return gt.join(s1.select(pl.col("entity_id").alias("s1")), on="s1")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("what", choices=["blocking"])
    ap.add_argument("--n", type=int, default=40000)
    args = ap.parse_args()
    pl.Config.set_tbl_cols(20)
    pl.Config.set_tbl_width_chars(200)
    blocking_report(args.n)


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/ensemble.py
"""Stage-2 ensemble on top of a trained run: LightGBM lambdarank + CatBoost + logistic stacker.

python -m er.run ensemble --run v1
Reuses run v1's stage-B features and fold split. Every base model is trained 2-fold out-of-fold
on fit S1. The stacker is fit on OOF predictions only (no leakage), calibrated per source, and
scored on the untouched val S1. Writes val_scores_ens.parquet / test_scores_ens.parquet; `tune`
and `predict` can use them with --scores ens.
"""
from __future__ import annotations

import gc
import json
import logging
import pickle

import numpy as np
import polars as pl

from . import config as C
from .pipeline import META, _files, _parts, _scan_b, feature_cols, labels, train_roles

log = logging.getLogger(__name__)

RANK_PARAMS = dict(objective="lambdarank", learning_rate=0.05, num_leaves=127, min_data_in_leaf=100,
                   feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1, lambda_l2=3.0,
                   lambdarank_truncation_level=10, eval_at=[5], num_threads=C.WORKERS, verbose=-1, seed=C.SEED)
CAT_PARAMS = dict(loss_function="Logloss", iterations=3000, learning_rate=0.08, depth=8, l2_leaf_reg=5,
                  border_count=128, od_type="Iter", od_wait=100, thread_count=C.WORKERS, random_seed=C.SEED,
                  verbose=250, allow_writing_files=False)


def _logit(p: np.ndarray) -> np.ndarray:
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def _group_softmax(keys: np.ndarray, s: np.ndarray) -> np.ndarray:
    """Softmax of ranking scores within each (S1, source) group -> comparable across groups."""
    df = pl.DataFrame({"g": keys, "s": s}).with_columns(
        (pl.col("s") - pl.col("s").max().over("g")).exp().alias("e"))
    return (df["e"] / df.select(pl.col("e").sum().over("g"))["e"]).to_numpy()


def _load_fit(feats):
    roles = train_roles().select(pl.col("idx").alias("s1_idx"), "role", "fold")
    lab = labels().with_columns(pl.lit(1, pl.UInt8).alias("y"))
    fit = (_scan_b("train").join(roles.lazy().filter(pl.col("role") == "fit"), on="s1_idx")
           .join(lab.lazy(), on=["s1_idx", "src", "t_idx"], how="left").with_columns(pl.col("y").fill_null(0))
           .select("s1_idx", "t_idx", "y", "fold", *[pl.col(c).cast(pl.Float32) for c in feats])
           .sort("s1_idx", "src").collect(engine="streaming"))
    return fit


def _base_predict(X, keys, models) -> dict[str, np.ndarray]:
    import lightgbm as lgb  # noqa: F401
    out = {}
    out["lgb"] = np.mean([m.predict(X, num_threads=C.WORKERS) for m in models["lgb"]], axis=0)
    raw = np.mean([m.predict(X, num_threads=C.WORKERS) for m in models["rank"]], axis=0)
    out["rank"], out["rank_raw"] = _group_softmax(keys, raw), raw
    out["cat"] = np.mean([m.predict_proba(X)[:, 1] for m in models["cat"]], axis=0)
    return out


def _stack_X(base: dict[str, np.ndarray]) -> np.ndarray:
    return np.column_stack([_logit(base["lgb"]), _logit(base["rank"]), base["rank_raw"], _logit(base["cat"])])


def run_ensemble(run_name: str = "v1") -> dict:
    import lightgbm as lgb
    from catboost import CatBoostClassifier
    from sklearn.isotonic import IsotonicRegression
    from sklearn.linear_model import LogisticRegression

    rd = C.WORK / "runs" / run_name
    summ = json.loads((rd / "summary.json").read_text())
    feats = summ["features"]
    fit = _load_fit(feats)
    y, fold = fit["y"].to_numpy(), fit["fold"].to_numpy()
    keys = (fit["s1_idx"].cast(pl.Int64) * 4 + fit["src"].cast(pl.Int64)).to_numpy()
    X = fit.select(feats).to_numpy()
    src = fit["src"].to_numpy()
    del fit
    gc.collect()
    models = {"lgb": [lgb.Booster(model_file=str(rd / f"stage2_f{f}.lgb")) for f in (0, 1)], "rank": [], "cat": []}
    oof = {k: np.zeros(len(y), dtype=np.float64) for k in ("lgb", "rank", "rank_raw", "cat")}
    for f in (0, 1):
        tr, va = fold != f, fold == f
        oof["lgb"][va] = models["lgb"][f].predict(X[va], num_threads=C.WORKERS)
        # lambdarank: rows are sorted by (s1_idx, src) so groups are contiguous
        _, gtr = np.unique(keys[tr], return_counts=True)
        _, gva = np.unique(keys[va], return_counts=True)
        dtr = lgb.Dataset(X[tr], y[tr], group=gtr, feature_name=feats)
        dva = lgb.Dataset(X[va], y[va], group=gva, reference=dtr)
        mr = lgb.train(RANK_PARAMS, dtr, 2000, valid_sets=[dva],
                       callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(250)])
        mr.save_model(str(rd / f"rank_f{f}.lgb"))
        models["rank"].append(mr)
        oof["rank_raw"][va] = mr.predict(X[va], num_threads=C.WORKERS)
        oof["rank"][va] = _group_softmax(keys[va], oof["rank_raw"][va])
        del dtr, dva
        mc = CatBoostClassifier(**CAT_PARAMS)
        mc.fit(X[tr], y[tr], eval_set=(X[va], y[va]))
        mc.save_model(str(rd / f"cat_f{f}.cbm"))
        models["cat"].append(mc)
        oof["cat"][va] = mc.predict_proba(X[va])[:, 1]
        gc.collect()
    S = _stack_X(oof)
    stacker = LogisticRegression(C=1.0, max_iter=1000).fit(S, y)
    ps = stacker.predict_proba(S)[:, 1]
    cal = {s: IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(ps[src == s], y[src == s])
           for s in (2, 3)}
    from sklearn.metrics import log_loss, roc_auc_score
    diag = {k: {"auc": float(roc_auc_score(y, v))} for k, v in {**oof, "stack": ps}.items()}
    diag["stacker_coef"] = stacker.coef_.tolist()
    log.info("ensemble OOF diagnostics %s", diag)
    del X
    gc.collect()
    with open(rd / "ensemble.pkl", "wb") as fh:
        pickle.dump({"stacker": stacker, "cal": cal, "feats": feats}, fh)
    for split, role, name in (("train", "val", "val_scores_ens.parquet"), ("test", None, "test_scores_ens.parquet")):
        _score_split(split, role, models, stacker, cal, feats).write_parquet(rd / name)
    (rd / "ensemble.json").write_text(json.dumps(diag, indent=1))
    return diag


def _score_split(split, role, models, stacker, cal, feats) -> pl.DataFrame:
    roles = train_roles().select(pl.col("idx").alias("s1_idx"), "role") if role else None
    lab = labels().with_columns(pl.lit(1, pl.UInt8).alias("y")) if split == "train" else None
    out = []
    for d_ in _parts(split, "B"):
        for f in _files(d_):
            d = pl.read_parquet(f)
            if roles is not None:
                d = d.join(roles.filter(pl.col("role") == role).select("s1_idx"), on="s1_idx", how="semi")
            if d.height == 0:
                continue
            d = d.sort("s1_idx", "src")
            X = d.select([pl.col(c).cast(pl.Float32) if c in d.columns else pl.lit(None, pl.Float32).alias(c)
                          for c in feats]).to_numpy()
            keys = (d["s1_idx"].cast(pl.Int64) * 4 + d["src"].cast(pl.Int64)).to_numpy()
            ps = stacker.predict_proba(_stack_X(_base_predict(X, keys, models)))[:, 1]
            src = d["src"].to_numpy()
            p = ps.copy()
            for s, c in cal.items():
                if (src == s).any():
                    p[src == s] = c.predict(ps[src == s])
            r = d.select("s1_idx", "src", "t_idx").with_columns(pl.Series("p", p.astype(np.float32)))
            out.append(r)
    res = pl.concat(out)
    if lab is not None:
        res = res.join(lab, on=["s1_idx", "src", "t_idx"], how="left").with_columns(pl.col("y").fill_null(0))
    return res


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/features.py
"""Pair feature computation (vectorised: rapidfuzz cpdist + Polars list/set ops)."""
from __future__ import annotations

import logging
import math

import numpy as np
import polars as pl
from rapidfuzz import fuzz, process
from rapidfuzz.distance import JaroWinkler, Levenshtein

log = logging.getLogger(__name__)

REC_COLS = ["name_norm", "core", "legal", "skel", "addr_norm", "atoks", "nums", "hno", "state",
            "loc", "street", "country"]
EXTRA_COLS = ["core_a", "atoks_a", "nrep"]   # optional, added by er.auxfit.Aux.enrich


def token_idf(tables: list[pl.DataFrame], col: str) -> pl.DataFrame:
    """Per-country smoothed IDF for tokens in list column ``col`` over the given tables."""
    parts = [t.select("country", pl.col(col).list.unique().alias("tok")) for t in tables]
    ex = pl.concat(parts).explode("tok").drop_nulls("tok")
    n = pl.concat([t.select("country") for t in tables]).group_by("country").len("n")
    df = ex.group_by("country", "tok").len("df").join(n, on="country")
    return df.select("country", "tok",
                     ((pl.col("n") + 1) / (pl.col("df") + 1)).log().cast(pl.Float32).alias("idf"))


def _cp(scorer, a, b, **kw) -> np.ndarray:
    return process.cpdist(a, b, scorer=scorer, workers=-1, dtype=np.float32, **kw)


def _set_feats(df: pl.DataFrame, col: str, name: str) -> list[pl.Expr]:
    a, b = pl.col(col + "_1"), pl.col(col + "_2")
    inter = a.list.set_intersection(b).list.len().cast(pl.Float32)
    la, lb = a.list.unique().list.len().cast(pl.Float32), b.list.unique().list.len().cast(pl.Float32)
    return [
        inter.alias(f"{name}_inter"),
        (inter / (la + lb - inter)).fill_nan(0).alias(f"{name}_jac"),
        (inter / la).fill_nan(0).alias(f"{name}_c12"),
        (inter / lb).fill_nan(0).alias(f"{name}_c21"),
        la.alias(f"{name}_n1"), lb.alias(f"{name}_n2"),
    ]


def _idf_overlap(df: pl.DataFrame, col: str, idf: pl.DataFrame, name: str) -> pl.DataFrame:
    """IDF-weighted overlap / union mass for list column ``col`` (per pair row ``pid``)."""
    def mass(expr_df: pl.DataFrame, alias: str) -> pl.DataFrame:
        return (expr_df.explode("tok").drop_nulls("tok")
                .join(idf, on=["country", "tok"], how="left")
                .group_by("pid").agg(pl.col("idf").fill_null(12.0).sum().alias(alias)))
    base = df.select("pid", "country", pl.col(col + "_1").alias("a"), pl.col(col + "_2").alias("b"))
    inter = mass(base.select("pid", "country", pl.col("a").list.set_intersection("b").alias("tok")), "i")
    uni = mass(base.select("pid", "country", pl.col("a").list.set_union("b").alias("tok")), "u")
    m1 = mass(base.select("pid", "country", pl.col("a").list.unique().alias("tok")), "m1")
    m2 = mass(base.select("pid", "country", pl.col("b").list.unique().alias("tok")), "m2")
    r = (df.select("pid").join(inter, on="pid", how="left").join(uni, on="pid", how="left")
           .join(m1, on="pid", how="left").join(m2, on="pid", how="left")
           .with_columns(pl.col("i", "u", "m1", "m2").fill_null(0.0)))  # never touch the u32 key
    return r.select(
        "pid",
        pl.col("i").alias(f"{name}_widf"),
        (pl.col("i") / pl.col("u")).fill_nan(0).alias(f"{name}_wjac"),
        (pl.col("i") / pl.col("m1")).fill_nan(0).alias(f"{name}_wc12"),
        (pl.col("i") / pl.col("m2")).fill_nan(0).alias(f"{name}_wc21"),
        pl.col("m1").alias(f"{name}_mass1"),
    )


def pair_features(pairs: pl.DataFrame, s1: pl.DataFrame, tg: pl.DataFrame,
                  idf_core: pl.DataFrame, idf_addr: pl.DataFrame,
                  name_freq1: pl.DataFrame, name_freq2: pl.DataFrame) -> pl.DataFrame:
    """pairs: s1_idx, t_idx, src + blocking cols. s1/tg: normalised record tables keyed by idx
    (tg has a ``src`` column). Returns pairs with feature columns appended."""
    extra = [c for c in EXTRA_COLS if c in s1.columns and c in tg.columns]
    cols = REC_COLS + extra
    df = (pairs.with_row_index("pid")
          .join(s1.select(pl.col("idx").alias("s1_idx"), *cols), on="s1_idx", how="left")
          .join(tg.select(pl.col("idx").alias("t_idx"), "src", *[c for c in cols if c != "country"]),
                on=["t_idx", "src"], how="left", suffix="_2")
          .rename({c: c + "_1" for c in cols if c != "country"}))
    s = lambda c: df[c].fill_null("").to_list()
    j = lambda c: df[c].list.join(" ").fill_null("").to_list()
    n1, n2 = s("name_norm_1"), s("name_norm_2")
    c1, c2 = j("core_1"), j("core_2")
    k1, k2 = j("skel_1"), j("skel_2")
    a1, a2 = s("addr_norm_1"), s("addr_norm_2")
    h1, h2 = s("hno_1"), s("hno_2")
    f = {
        "n_ratio": _cp(fuzz.ratio, n1, n2), "n_pratio": _cp(fuzz.partial_ratio, n1, n2),
        "n_tsort": _cp(fuzz.token_sort_ratio, n1, n2), "n_tset": _cp(fuzz.token_set_ratio, n1, n2),
        "n_wratio": _cp(fuzz.WRatio, n1, n2), "n_jw": _cp(JaroWinkler.normalized_similarity, n1, n2),
        "c_ratio": _cp(fuzz.ratio, c1, c2), "c_pratio": _cp(fuzz.partial_ratio, c1, c2),
        "c_tsort": _cp(fuzz.token_sort_ratio, c1, c2), "c_tset": _cp(fuzz.token_set_ratio, c1, c2),
        "c_jw": _cp(JaroWinkler.normalized_similarity, c1, c2),
        "c_lev": _cp(Levenshtein.normalized_similarity, c1, c2),
        "k_ratio": _cp(fuzz.ratio, k1, k2), "k_tset": _cp(fuzz.token_set_ratio, k1, k2),
        "a_ratio": _cp(fuzz.ratio, a1, a2), "a_pratio": _cp(fuzz.partial_ratio, a1, a2),
        "a_tset": _cp(fuzz.token_set_ratio, a1, a2), "a_tsort": _cp(fuzz.token_sort_ratio, a1, a2),
        "h_lev": _cp(Levenshtein.distance, h1, h2),
    }
    if "core_a" in extra:  # alias-mapped, filler-free name (er.auxfit)
        ca1, ca2 = j("core_a_1"), j("core_a_2")
        f.update({"ca_ratio": _cp(fuzz.ratio, ca1, ca2), "ca_tset": _cp(fuzz.token_set_ratio, ca1, ca2),
                  "ca_tsort": _cp(fuzz.token_sort_ratio, ca1, ca2)})
    if "atoks_a" in extra:
        aa1, aa2 = j("atoks_a_1"), j("atoks_a_2")
        f.update({"aa_tset": _cp(fuzz.token_set_ratio, aa1, aa2)})
    df = df.with_columns([pl.Series(k, v) for k, v in f.items()])
    for c, nm in (("core_a", "ca"), ("atoks_a", "aa")):
        if c in extra:
            df = df.with_columns(*_set_feats(df, c, nm))
    first1 = df["core_1"].list.first().fill_null("").to_list()
    first2 = df["core_2"].list.first().fill_null("").to_list()
    df = df.with_columns(pl.Series("first_jw", _cp(JaroWinkler.normalized_similarity, first1, first2)))

    df = df.with_columns(
        *_set_feats(df, "core", "core"), *_set_feats(df, "skel", "skel"),
        *_set_feats(df, "atoks", "at"), *_set_feats(df, "nums", "num"),
        *_set_feats(df, "loc", "loc"), *_set_feats(df, "street", "str"),
        (pl.col("hno_1") != "").cast(pl.Int8).alias("h_has1"),
        (pl.col("hno_2") != "").cast(pl.Int8).alias("h_has2"),
        ((pl.col("hno_1") == pl.col("hno_2")) & (pl.col("hno_1") != "")).cast(pl.Int8).alias("h_eq"),
        (pl.col("hno_1").cast(pl.Float64, strict=False) - pl.col("hno_2").cast(pl.Float64, strict=False))
            .abs().log1p().cast(pl.Float32).alias("h_logdiff"),
        pl.when((pl.col("state_1") == "") | (pl.col("state_2") == "")).then(-1)
          .otherwise((pl.col("state_1") == pl.col("state_2")).cast(pl.Int8)).alias("state_eq"),
        (pl.col("addr_norm_2") == "").cast(pl.Int8).alias("a_missing2"),
        pl.when((pl.col("legal_1").list.len() == 0) | (pl.col("legal_2").list.len() == 0)).then(-1)
          .otherwise(pl.col("legal_1").list.set_intersection("legal_2").list.len().cast(pl.Int32))
          .alias("legal_inter"),
        pl.col("legal_1").list.len().alias("legal_n1"), pl.col("legal_2").list.len().alias("legal_n2"),
        pl.col("name_norm_1").str.len_chars().alias("n_len1"),
        pl.col("name_norm_2").str.len_chars().alias("n_len2"),
    )
    df = (df.join(_idf_overlap(df, "core", idf_core, "core"), on="pid", how="left")
            .join(_idf_overlap(df, "atoks", idf_addr, "at"), on="pid", how="left")
            .join(name_freq1, on="name_norm_1", how="left")
            .join(name_freq2, on="name_norm_2", how="left"))
    drop = [c for c in df.columns if c.endswith(("_1", "_2")) and c.split("_")[0] in
            {"name", "core", "legal", "skel", "addr", "atoks", "nums", "hno", "state", "loc", "street"}]
    return df.drop(drop + ["pid"])


def context_features(df: pl.DataFrame, score: str = "bs") -> pl.DataFrame:
    """Within-S1 and competition (within-target) features over a candidate set."""
    return df.with_columns(
        pl.len().over("s1_idx").alias("cx_n"),
        pl.len().over("s1_idx", "src").alias("cx_n_src"),
        (pl.col(score) / pl.col(score).max().over("s1_idx", "src")).alias("cx_rel"),
        pl.col(score).rank("ordinal", descending=True).over("s1_idx", "src").alias("cx_rank"),
        pl.col("n_tset").rank("ordinal", descending=True).over("s1_idx", "src").alias("cx_rank_ntset"),
        pl.col("a_tset").rank("ordinal", descending=True).over("s1_idx", "src").alias("cx_rank_atset"),
        (pl.col("n_tset") - pl.col("n_tset").max().over("s1_idx", "src")).alias("cx_gap_ntset"),
        (pl.col("a_tset") - pl.col("a_tset").max().over("s1_idx", "src")).alias("cx_gap_atset"),
        pl.len().over("src", "t_idx").alias("tx_n"),
        pl.col(score).rank("ordinal", descending=True).over("src", "t_idx").alias("tx_rank"),
        (pl.col(score) - pl.col(score).max().over("src", "t_idx")).alias("tx_gap"),
        (pl.col("n_tset") - pl.col("n_tset").max().over("src", "t_idx")).alias("tx_gap_ntset"),
        (pl.col("a_tset") - pl.col("a_tset").max().over("src", "t_idx")).alias("tx_gap_atset"),
    )


def coherence_features(d: pl.DataFrame, tabs: dict[str, pl.DataFrame], anchors: pl.DataFrame
                       ) -> pl.DataFrame:
    """Cluster coherence (collective ER, README N3).

    anchors: per (s1_idx, src) the top stage-1 candidate (a_idx, a_p1). For each pair we compare
    the candidate with the S1's anchor in the *other* source and with its *own-source* anchor
    (S2/S3 duplicates of one business resemble each other). Self-comparisons are masked (-1).
    """
    strs = pl.concat([t.select(pl.col("idx").alias("x_idx"), pl.lit(int(s), pl.UInt8).alias("x_src"),
                               pl.col("core").list.join(" ").alias("xc"), pl.col("addr_norm").fill_null("").alias("xa"))
                      for s, t in tabs.items() if s in ("2", "3")])
    out = d.select("s1_idx", "src", "t_idx").with_row_index("rid")
    me = out.join(strs, left_on=["t_idx", "src"], right_on=["x_idx", "x_src"], how="left")
    feats = {}
    for tag, src_expr in (("same", pl.col("src")), ("oth", (5 - pl.col("src").cast(pl.Int16)).cast(pl.UInt8))):
        a = (out.with_columns(src_expr.alias("a_src"))
                .join(anchors, left_on=["s1_idx", "a_src"], right_on=["s1_idx", "src"], how="left")
                .join(strs, left_on=["a_idx", "a_src"], right_on=["x_idx", "x_src"], how="left").sort("rid"))
        c = _cp(fuzz.token_set_ratio, me.sort("rid")["xc"].fill_null("").to_list(), a["xc"].fill_null("").to_list())
        ad = _cp(fuzz.token_set_ratio, me.sort("rid")["xa"].fill_null("").to_list(), a["xa"].fill_null("").to_list())
        miss = a["a_idx"].is_null().to_numpy()
        if tag == "same":
            miss |= (a["a_idx"] == a["t_idx"]).fill_null(False).to_numpy()
        c[miss], ad[miss] = -1, -1
        feats[f"coh_c_{tag}"], feats[f"coh_a_{tag}"] = c, ad
        feats[f"coh_p_{tag}"] = a["a_p1"].fill_null(-1).to_numpy().astype(np.float32)
    return d.with_columns([pl.Series(k, v) for k, v in feats.items()])


def name_freq(tbl: pl.DataFrame, suffix: str) -> pl.DataFrame:
    return (tbl.group_by("name_norm").len(f"nfreq{suffix}")
               .rename({"name_norm": f"name_norm{suffix}"}))


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/hybrid.py
"""Per-country submission assembly: each country's rows come from the version that is best for it.

US / India are chosen on labelled validation (er.compare). France has no labels anywhere, so it is chosen
on leaderboard evidence. The LB decomposition (test mix US 38.3% / India 46.8% / France 15.0%, US/India
test ~ validation, verified by sum-p calibration) implies France F0.5 ~0.930 for v3 vs ~0.902 for v5.
v5 added ~60k weaker-name France pairs (median name similarity 84 vs 100), consistent with
co-located French businesses (17.5% of France S1 share an address vs 4-6% in US/India) being merged.

Both files of a country come from the same version, so candidate_pairs.tsv stays the exact candidate
set that version's model scored for those S1 rows.

python -m er.hybrid --base v6 --override France=v3 --out v6h
"""
from __future__ import annotations

import argparse
import subprocess
import sys

import polars as pl

from . import config as C

FILES = (("matching_results.tsv", "matched_entity_ids"), ("candidate_pairs.tsv", "candidate_entity_ids"))


def assemble(base: str, override: dict[str, str], out: str) -> dict:
    s1 = (pl.read_parquet(C.WORK / "test_s1.parquet", columns=["entity_id", "country"])
            .rename({"entity_id": "source1_entity_id"}))
    od = C.OUT / out
    od.mkdir(parents=True, exist_ok=True)
    info = {}
    for fname, col in FILES:
        b = pl.read_csv(C.OUT / base / fname, separator="\t", infer_schema=False)
        order = b.select("source1_entity_id")
        parts = [b.join(s1, on="source1_entity_id").filter(~pl.col("country").is_in(list(override)))]
        for country, ver in override.items():
            o = pl.read_csv(C.OUT / ver / fname, separator="\t", infer_schema=False)
            parts.append(o.join(s1, on="source1_entity_id").filter(pl.col("country") == country))
        m = pl.concat([p.select("source1_entity_id", col) for p in parts])
        assert m.height == b.height == s1.height and m["source1_entity_id"].n_unique() == m.height
        m = order.join(m, on="source1_entity_id", maintain_order="left")     # keep the base row order
        m.write_csv(od / fname, separator="\t", quote_style="never")
        info[fname] = m.height
    r = subprocess.run([sys.executable, str(C.ROOT / "student_resource" / "utils" / "validate_submission.py"),
                        "--matching", str(od / "matching_results.tsv"), "--candidate", str(od / "candidate_pairs.tsv"),
                        "--test-dir", str(C.DATA / "test")], capture_output=True, text=True, encoding="utf-8")
    info["validator"] = (r.stdout + r.stderr).strip().splitlines()[-1] if (r.stdout + r.stderr).strip() else r.returncode
    return info


def from_selection(base: str, country: str, sel_path: str, out: str) -> dict:
    """output/<out> = output/<base> with <country>'s matching rows rebuilt from a selected-pairs parquet
    (s1_idx, src, t_idx in C.WORK's test idx space, same candidate set as <base>; candidate_pairs.tsv copied)."""
    import shutil
    sel = pl.read_parquet(sel_path).select(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32))
    s1 = (pl.scan_parquet(C.WORK / "test_s1.parquet").filter(pl.col("country") == country)
            .select(pl.col("idx").cast(pl.UInt32).alias("s1_idx"), pl.col("entity_id").alias("source1_entity_id")).collect())
    tg = pl.concat([pl.scan_parquet(C.WORK / f"test_s{k}.parquet").select(pl.col("idx").cast(pl.UInt32).alias("t_idx"), pl.lit(k).cast(pl.UInt8).alias("src"),
                    pl.col("entity_id").alias("tid")).collect().join(sel.select("src", "t_idx").unique(), on=["src", "t_idx"], how="semi") for k in (2, 3)])
    m = (sel.join(s1, on="s1_idx").join(tg, on=["src", "t_idx"]).group_by("source1_entity_id")
            .agg(pl.col("tid").sort().str.join(",").alias("new")))
    b = pl.read_csv(C.OUT / base / "matching_results.tsv", separator="\t", infer_schema=False)
    rows = s1.select("source1_entity_id").join(m, on="source1_entity_id", how="left").with_columns(pl.col("new").fill_null(""))
    b = (b.join(rows, on="source1_entity_id", how="left")
          .with_columns(pl.when(pl.col("source1_entity_id").is_in(s1["source1_entity_id"].implode())).then(pl.col("new"))
                        .otherwise(pl.col("matched_entity_ids")).alias("matched_entity_ids")).drop("new"))
    od = C.OUT / out
    od.mkdir(parents=True, exist_ok=True)
    b.write_csv(od / "matching_results.tsv", separator="\t", quote_style="never")
    shutil.copyfile(C.OUT / base / "candidate_pairs.tsv", od / "candidate_pairs.tsv")
    return {"country_rows": s1.height, "rows_with_matches": m.height, "pairs": sel.height}


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", required=True)
    ap.add_argument("--from-selection", default=None, help="COUNTRY=path/to/selected_pairs.parquet")
    ap.add_argument("--override", action="append", default=[], help="COUNTRY=VERSION, repeatable")
    ap.add_argument("--out", required=True)
    a = ap.parse_args()
    if a.from_selection:
        c, path = a.from_selection.split("=", 1)
        print(from_selection(a.base, c, path, a.out))
    else:
        print(assemble(a.base, dict(x.split("=", 1) for x in a.override), a.out))


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/metric.py
"""Exact re-implementation of the challenge metric (macro F0.5 over Source-1 entities)."""
from __future__ import annotations

import polars as pl

BETA2 = 0.25


def macro_f05(pred: pl.DataFrame, truth: pl.DataFrame, s1_ids: pl.Series) -> dict:
    """pred/truth: long pairs (s1, tid). s1_ids: every evaluated Source-1 id.

    Singleton rules: empty/empty = 1, anything else involving an empty side = 0.
    """
    base = pl.DataFrame({"s1": s1_ids}).unique()
    t = truth.join(base, on="s1").unique()
    p = pred.join(base, on="s1").unique()
    tp = t.join(p, on=["s1", "tid"]).group_by("s1").len("tp")
    nt = t.group_by("s1").len("nt")
    np_ = p.group_by("s1").len("np")
    df = (base.join(nt, on="s1", how="left").join(np_, on="s1", how="left")
              .join(tp, on="s1", how="left").fill_null(0))
    prec = pl.col("tp") / pl.col("np")
    rec = pl.col("tp") / pl.col("nt")
    f = (1 + BETA2) * prec * rec / (BETA2 * prec + rec)
    df = df.with_columns(
        pl.when((pl.col("nt") == 0) & (pl.col("np") == 0)).then(1.0)
          .when((pl.col("nt") == 0) | (pl.col("np") == 0) | (pl.col("tp") == 0)).then(0.0)
          .otherwise(f).alias("f"),
        pl.when(pl.col("np") > 0).then(prec).otherwise(None).alias("p"),
        pl.when(pl.col("nt") > 0).then(rec).otherwise(None).alias("r"),
    )
    return {"f05": df["f"].mean(), "prec": df["p"].mean(), "rec": df["r"].mean(),
            "n": df.height, "singleton_f": df.filter(pl.col("nt") == 0)["f"].mean()}


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/minidata.py
"""Build a small, realistic dataset (real records) for end-to-end smoke/regression tests.

python -m er.minidata OUT_DIR --n 3000
Train: n sampled S1 per country with all their true matches, plus 3x distractor targets.
Test : n sampled S1 per country (incl. France) plus a random target sample of the same country.
"""
from __future__ import annotations

import argparse
from pathlib import Path

import polars as pl

from . import config as C
from .prep import read_tsv


def build(out: Path, n: int = 3000, seed: int = 0) -> None:
    for split in ("train", "test"):
        (out / split).mkdir(parents=True, exist_ok=True)
        s1 = read_tsv(C.DATA / split / f"{split}_source1.tsv")
        s1 = pl.concat([g.sample(min(n, g.height), seed=seed) for _, g in s1.group_by("country", maintain_order=True)])
        s1.write_csv(out / split / f"{split}_source1.tsv", separator="\t", quote_style="never")
        if split == "train":
            gt = read_tsv(C.DATA / "train" / "train_ground_truth.tsv").join(
                s1.select(pl.col("entity_id").alias("source1_entity_id")), on="source1_entity_id")
            gt.write_csv(out / "train" / "train_ground_truth.tsv", separator="\t", quote_style="never")
            want = set(gt["matched_entity_ids"].fill_null("").str.split(",").explode().to_list()) - {""}
        for src in ("2", "3"):
            t = read_tsv(C.DATA / split / f"{split}_source{src}.tsv")
            keep = t.filter(pl.col("entity_id").is_in(list(want))) if split == "train" else t.head(0)
            dist = t.join(keep.select("entity_id"), on="entity_id", how="anti")
            dist = pl.concat([g.sample(min(3 * n, g.height), seed=seed) for _, g in dist.group_by("country", maintain_order=True)])
            pl.concat([keep, dist]).sample(fraction=1.0, shuffle=True, seed=seed).write_csv(
                out / split / f"{split}_source{src}.tsv", separator="\t", quote_style="never")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("out")
    ap.add_argument("--n", type=int, default=3000)
    a = ap.parse_args()
    build(Path(a.out), a.n)


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/noise.py
"""Noise-generator inversion (research note R1a + R1b).

The corruptions in this dataset are synthetic and come from a finite set of operators. Two
of them are learned here from training positive pairs, using only the provided data:

R1a  filler tokens: words the generator *adds* to a target name ("services", "enterprises",
     transliterated legal forms such as "praivet"/"limitet"/"pra li", ...). For each
     token, ``rate = P(token absent from the S1 name | token present in the target name)``
     over positive pairs. Callers down-weight high-rate tokens in IDF overlap and blocking.

R1b  name replacement: the generator sometimes replaces the whole name with a pronounceable
     pseudo-word assembled from a syllable inventory ("Ciraecto", "Zephevozeph",
     "Orbisyndelta"). A hashed char-n-gram logistic classifier separates these from ordinary
     target names (clean, typo'd, transliterated, domains). Its score tells the matcher that
     name dissimilarity is uninformative, so it should trust the address.
"""
from __future__ import annotations

import logging
import pickle
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import polars as pl
import scipy.sparse as sp
from rapidfuzz import fuzz, process
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import LogisticRegression

log = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# R1a — filler tokens
# ---------------------------------------------------------------------------
def mine_filler_tokens(s1_core: pl.Series, tgt_core: pl.Series, min_count: int = 30,
                       prior: float = 1.0) -> dict[str, float]:
    """Per-token "added by noise" rate from aligned positive pairs.

    rate(t) = (#pairs where t is in the target but not in S1 + prior*0.5)
              / (#pairs where t is in the target + prior)
    Only tokens seen in at least ``min_count`` target names are returned. Rare tokens are
    mostly typos, and IDF already handles those.
    """
    if s1_core.len() != tgt_core.len():
        raise ValueError("s1_core and tgt_core must be aligned (same length)")
    df = pl.DataFrame({"a": s1_core, "b": tgt_core}).with_columns(
        pl.col("a").fill_null(pl.lit([], dtype=pl.List(pl.Utf8))),
        pl.col("b").fill_null(pl.lit([], dtype=pl.List(pl.Utf8))))
    present = df.select(pl.col("b").list.unique().alias("t")).explode("t").drop_nulls()
    added = df.select(pl.col("b").list.set_difference("a").alias("t")).explode("t").drop_nulls()
    cnt = present.group_by("t").len("n").join(added.group_by("t").len("k"), on="t", how="left")
    cnt = cnt.fill_null(0).filter(pl.col("n") >= min_count)
    cnt = cnt.with_columns(((pl.col("k") + prior * 0.5) / (pl.col("n") + prior)).alias("rate"))
    return dict(zip(cnt["t"].to_list(), cnt["rate"].to_list()))


def split_filler(rates: dict[str, float], s1_vocab: set[str], threshold: float = 0.8
                 ) -> tuple[set[str], set[str]]:
    """Split high-rate tokens into (filler, foreign).

    filler : words that also occur in clean S1 names ("services", "center", "formerly"). They
             are true noise insertions, so down-weight them.
    foreign: tokens never seen in S1 names, mostly transliteration artifacts ("kanstrakshan",
             "praivet"). They carry identity, so map them via skeleton/alias matching rather
             than dropping them.
    """
    hi = {t for t, r in rates.items() if r >= threshold}
    return {t for t in hi if t in s1_vocab}, {t for t in hi if t not in s1_vocab}


# ---------------------------------------------------------------------------
# R1b — whole-name replacement detector
# ---------------------------------------------------------------------------
_EMPTY = pl.lit([], dtype=pl.List(pl.Utf8))
_HASH = dict(analyzer="char_wb", ngram_range=(2, 4), n_features=2 ** 20, alternate_sign=False,
             norm="l2", lowercase=False, dtype=np.float32)


@dataclass
class NameReplacementModel:
    vocab: set[str]
    clf: LogisticRegression | None = None
    meta: dict = field(default_factory=dict)

    def save(self, path: str | Path) -> None:
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @staticmethod
    def load(path: str | Path) -> "NameReplacementModel":
        with open(path, "rb") as f:
            return pickle.load(f)


def _dense(core: pl.Series, legal: pl.Series, vocab: pl.Series) -> tuple[list[str], np.ndarray]:
    """Text for the char model plus dense features [ntok, oov_frac, has_legal, has_digit, len/20]."""
    df = pl.DataFrame({"c": core, "l": legal}).with_columns(
        pl.col("c").fill_null(_EMPTY), pl.col("l").fill_null(_EMPTY)).with_row_index("i")
    oov = (df.select("i", "c").explode("c").drop_nulls("c")
             .with_columns((~pl.col("c").is_in(vocab.implode())).alias("o"))
             .group_by("i").agg(pl.col("o").mean().alias("oov")))
    df = df.join(oov, on="i", how="left").sort("i")
    text = df["c"].list.join(" ")
    X = np.column_stack([
        df["c"].list.len().cast(pl.Float32).to_numpy(),
        df["oov"].fill_null(1.0).cast(pl.Float32).to_numpy(),
        (df["l"].list.len() > 0).cast(pl.Float32).to_numpy(),
        text.str.contains(r"\d").cast(pl.Float32).to_numpy(),
        (text.str.len_chars().cast(pl.Float32) / 20.0).to_numpy(),
    ]).astype(np.float32)
    return text.to_list(), X


def _features(model: NameReplacementModel, core: pl.Series, legal: pl.Series):
    vocab = model.meta["vocab_series"]
    text, dense = _dense(core, legal, vocab)
    H = HashingVectorizer(**_HASH).transform([f" {t} " for t in text])
    return sp.hstack([H, sp.csr_matrix(dense)], format="csr"), dense


def label_replacements(pos_s1_core: pl.Series, pos_tgt_core: pl.Series, pos_tgt_raw: pl.Series,
                       low: float = 45.0, high: float = 80.0) -> pl.Series:
    """Weak labels for target names in positive pairs.

    replaced (1): name similarity to its S1 < ``low``, raw name is ASCII-only (so not a
        transliteration) and contains no domain/hashtag marks ('.', '#', '@'), and there is
        at least one core token.
    ordinary (0): similarity >= ``high``, or the raw name is in a non-Latin script
        (transliterations are dissimilar but not replacements).
    Everything in between is null (ambiguous). Returns a label Series aligned with the inputs.
    """
    sim = process.cpdist(pos_s1_core.list.join(" ").fill_null("").to_list(),
                         pos_tgt_core.list.join(" ").fill_null("").to_list(),
                         scorer=fuzz.token_set_ratio, workers=-1)
    raw = pos_tgt_raw.fill_null("")
    ascii_ = ~raw.str.contains(r"[^\x00-\x7F]")
    marks = raw.str.contains(r"[.#@]")
    ntok = pos_tgt_core.list.len().fill_null(0)
    sim_s = pl.Series(sim)
    y = (pl.when((sim_s < low) & ascii_ & ~marks & (ntok > 0)).then(1)
           .when((sim_s >= high) | ~ascii_).then(0).otherwise(None))
    return pl.select(y.alias("y")).to_series()


def fit_name_lm(s1_core_tokens: pl.Series, pos_s1_core: pl.Series | None = None,
                pos_tgt_core: pl.Series | None = None, pos_tgt_raw: pl.Series | None = None,
                pos_tgt_legal: pl.Series | None = None, seed: int = 42,
                max_neg: int = 200_000, clean_core: pl.Series | None = None,
                max_clean: int = 200_000) -> NameReplacementModel:
    """Fit the name-replacement detector.

    s1_core_tokens: S1 core token lists (or flat tokens), which define the clean vocabulary.
    pos_*: aligned positive pairs (S1 core, target core, target raw name, target legal),
    used to mine weak labels. Without them the model falls back to the OOV rate
    (vocabulary-only).
    clean_core: extra *known-clean* names added as negatives, e.g. S1 names of any split.
    S1 is the deduplicated reference source and is never replaced, so this label-free data
    teaches the detector what unseen-country names (France) look like. It also joins the
    vocabulary.
    """
    s = s1_core_tokens
    if clean_core is not None:
        s = pl.concat([s if s.dtype == pl.List(pl.Utf8) else s.map_elements(lambda t: [t], return_dtype=pl.List(pl.Utf8)),
                       clean_core.cast(pl.List(pl.Utf8))])
    toks = s.explode() if s.dtype == pl.List(pl.Utf8) else s
    vc = toks.drop_nulls().value_counts()
    vocab = vc.filter(pl.col("count") >= 2)[vc.columns[0]]
    model = NameReplacementModel(vocab=set(vocab.to_list()), meta={"vocab_series": vocab})
    if pos_s1_core is None:
        return model
    y = label_replacements(pos_s1_core, pos_tgt_core, pos_tgt_raw)
    legal = (pos_tgt_legal if pos_tgt_legal is not None
             else pl.Series([[]] * pos_tgt_core.len(), dtype=pl.List(pl.Utf8)))
    lab = pl.DataFrame({"core": pos_tgt_core, "legal": legal, "y": y}).drop_nulls("y")
    pos, neg = lab.filter(pl.col("y") == 1), lab.filter(pl.col("y") == 0)
    neg = neg.sample(min(max_neg, neg.height), seed=seed)
    if clean_core is not None:
        cc = clean_core.drop_nulls().sample(min(max_clean, clean_core.drop_nulls().len()), seed=seed)
        neg = pl.concat([neg, pl.DataFrame({"core": cc.cast(pl.List(pl.Utf8)),
                                            "legal": pl.Series([[]] * cc.len(), dtype=pl.List(pl.Utf8)),
                                            "y": pl.Series([0] * cc.len(), dtype=neg["y"].dtype)})])
    tr = pl.concat([pos, neg]).sample(fraction=1.0, shuffle=True, seed=seed)
    X, _ = _features(model, tr["core"], tr["legal"])
    clf = LogisticRegression(C=2.0, max_iter=300, class_weight="balanced", solver="liblinear")
    clf.fit(X, tr["y"].to_numpy())
    model.clf = clf
    model.meta.update(n_pos=pos.height, n_neg=neg.height)
    log.info("name-replacement model: pos=%d neg=%d", pos.height, neg.height)
    return model


def name_replaced_score(model: NameReplacementModel, core: pl.Series, legal: pl.Series,
                        batch: int = 500_000) -> np.ndarray:
    """P(name was replaced by a generated pseudo-word), in [0, 1]. Vectorised and batched."""
    out = []
    for i in range(0, core.len(), batch):
        c, l = core.slice(i, batch), legal.slice(i, batch)
        X, dense = _features(model, c, l)
        if model.clf is None:  # vocabulary-only fallback: single fully-OOV token
            out.append(((dense[:, 0] == 1) & (dense[:, 1] == 1)).astype(np.float32))
        else:
            out.append(model.clf.predict_proba(X)[:, 1].astype(np.float32))
    return np.concatenate(out) if out else np.zeros(0, np.float32)


# ---------------------------------------------------------------------------
# Convenience: fit both from the training split (sampled, RAM-bounded)
# ---------------------------------------------------------------------------
def fit_from_training(work_dir: str | Path, n_pairs: int = 300_000, seed: int = 42,
                      clean_splits: tuple[str, ...] = ("train", "test")):
    """Return (filler_rates, replacement_model) fitted on a sample of train positives.

    S1 names of ``clean_splits`` are added as label-free clean negatives and vocabulary.
    This includes test S1 (France), which is legitimate because no labels are used.
    """
    from .prep import load_gt
    work = Path(work_dir)
    gt = load_gt()
    gt = gt.sample(min(n_pairs, gt.height), seed=seed)
    cols = ["entity_id", "core", "legal", "business_name"]
    s1 = (pl.scan_parquet(work / "train_s1.parquet").select(cols)
            .join(gt.lazy().select(pl.col("s1").alias("entity_id")).unique(), on="entity_id").collect())
    tg = pl.concat([pl.scan_parquet(work / f"train_s{k}.parquet").select(cols)
                    .join(gt.lazy().select(pl.col("tid").alias("entity_id")), on="entity_id").collect()
                    for k in "23"])
    d = (gt.join(s1.rename({c: c + "_1" for c in cols}).rename({"entity_id_1": "s1"}), on="s1")
           .join(tg.rename({c: c + "_2" for c in cols}).rename({"entity_id_2": "tid"}), on="tid"))
    vocab_src = pl.scan_parquet(work / "train_s1.parquet").select("core").collect()["core"]
    clean = [pl.scan_parquet(work / f"{sp_}_s1.parquet").select("core").collect()["core"]
             for sp_ in clean_splits if (work / f"{sp_}_s1.parquet").exists()]
    clean_core = pl.concat(clean) if clean else None
    filler = mine_filler_tokens(d["core_1"], d["core_2"])
    vocab_base = vocab_src if clean_core is None else pl.Series([], dtype=pl.List(pl.Utf8))
    model = fit_name_lm(vocab_base, d["core_1"], d["core_2"], d["business_name_2"], d["legal_2"],
                        seed=seed, clean_core=clean_core)
    return filler, model


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/normalize.py
"""Record-level normalisation of business names and addresses.

Everything here uses only the provided data plus general linguistic knowledge
(legal-form abbreviations, street-type abbreviations, state abbreviations). No
entity-level external data is used.
"""
from __future__ import annotations

import os
import re

from .aliases import split_alias_markers
from .text import skeleton, to_ascii

# --- names ------------------------------------------------------------------
LEGAL = {
    "private": "pvt", "pvt": "pvt", "pv": "pvt", "limited": "ltd", "ltd": "ltd", "limted": "ltd",
    "llp": "llp", "llc": "llc", "inc": "inc", "incorporated": "inc", "corporation": "corp",
    "corp": "corp", "company": "co", "co": "co", "pc": "pc", "pa": "pa", "plc": "plc",
    "lp": "lp", "ltda": "ltd", "sarl": "sarl", "sas": "sas", "sasu": "sasu", "sa": "sa",
    "eurl": "eurl", "sci": "sci", "snc": "snc", "gmbh": "gmbh", "opc": "opc", "public": "public",
    "pllc": "pllc", "lc": "llc", "cie": "cie", "md": "md", "dds": "dds", "cpa": "cpa",
    # transliterated legal forms (Indic-script names romanised by er.text)
    "praivet": "pvt", "praivat": "pvt", "privet": "pvt", "prayvet": "pvt", "limited": "ltd",
    "limitad": "ltd", "limited.": "ltd", "elelpi": "llp", "elaelapi": "llp", "elelapi": "llp", "kampani": "co", "kampni": "co",
    "korporeshan": "corp", "karporeshan": "corp", "inkarporetad": "inc",
}
NAME_STOP = {"and", "the", "of", "et", "de", "du", "des", "la", "le", "les", "l", "d", "a", "an",
             "dba", "m/s", "ms", "india", "france", "usa", "us"}
_DOMAIN = re.compile(r"^(https?://)?(www\.)?|\.(co\.in|com|in|net|org|fr|biz|info|co|io|us)\b")
_ONE_LETTER_DOT = re.compile(r"\b([a-z])\.")
_NON_ALNUM = re.compile(r"[^a-z0-9]+")
# learned back-transliteration (romanised Indic token -> English word), applied only to names
# written in an Indic script. Loaded from ER_TRANSLIT (JSON) so spawned workers see it. See er.translit.
TRANSLIT: dict[str, str] = {}
if os.environ.get("ER_TRANSLIT") and os.path.exists(os.environ["ER_TRANSLIT"]):
    import json as _json
    TRANSLIT = _json.load(open(os.environ["ER_TRANSLIT"], encoding="utf-8"))
_INDIC = re.compile(r"[ऀ-෿]")


def normalize_name(raw: str | None):
    s = to_ascii(raw or "")
    parts = [p.strip() for p in s.split("|")]
    s = next((p for p in parts if p), "")
    # "<pseudo-word> d/b/a <real name>": the identity-bearing name is the alias (see er.aliases)
    main, alias = split_alias_markers(s)
    if alias:
        s = alias
    s = _DOMAIN.sub(" ", s)
    if " " not in s.strip() and len(s.strip()) >= 9 and s.strip().endswith("com"):
        s = s.strip()[:-3]  # glued domain without a dot: "bsservicescom" -> "bsservices"
    s = _ONE_LETTER_DOT.sub(r"\1", s)
    s = s.replace("&", " and ")
    toks = _NON_ALNUM.sub(" ", s).split()
    if TRANSLIT and raw and _INDIC.search(raw):
        toks = [TRANSLIT.get(t, t) for t in toks]
    seen, uniq = set(), []
    for t in toks:
        if t not in seen:
            seen.add(t)
            uniq.append(t)
    legal = sorted({LEGAL[t] for t in uniq if t in LEGAL})
    core = [t for t in uniq if t not in LEGAL and t not in NAME_STOP]
    if not core:  # name made only of legal/stop words: keep them rather than nothing
        core = [t for t in uniq if t not in NAME_STOP] or uniq
    return " ".join(uniq), core, legal


# --- addresses ---------------------------------------------------------------
ABBR = {
    # street types (US / India / France); 'saint' is merged with 'st' on purpose (noise maps St->Saint)
    "street": "st", "st": "st", "str": "st", "saint": "st", "road": "rd", "rd": "rd",
    "avenue": "ave", "ave": "ave", "av": "ave", "boulevard": "blvd", "blvd": "blvd", "bd": "blvd",
    "bld": "blvd", "drive": "dr", "dr": "dr", "lane": "ln", "ln": "ln", "court": "ct", "ct": "ct",
    "place": "pl", "pl": "pl", "highway": "hwy", "hwy": "hwy", "parkway": "pkwy", "pkwy": "pkwy",
    "circle": "cir", "cir": "cir", "trail": "trl", "trl": "trl", "terrace": "ter", "ter": "ter",
    "square": "sq", "sq": "sq", "way": "way", "plaza": "plz", "plz": "plz", "mount": "mt",
    "mt": "mt", "fort": "ft", "ft": "ft", "center": "ctr", "centre": "ctr", "ctr": "ctr",
    "north": "n", "n": "n", "south": "s", "s": "s", "east": "e", "e": "e", "west": "w", "w": "w",
    "northeast": "ne", "northwest": "nw", "southeast": "se", "southwest": "sw",
    "apartment": "unit", "apt": "unit", "unit": "unit", "suite": "unit", "ste": "unit",
    "floor": "fl", "fl": "fl", "flr": "fl", "building": "bldg", "bldg": "bldg",
    "rue": "rue", "r": "rue", "quai": "quai", "q": "quai", "chemin": "ch", "ch": "ch",
    "allee": "all", "all": "all", "impasse": "imp", "imp": "imp", "route": "rte", "rte": "rte",
    "near": "nr", "nr": "nr", "opposite": "opp", "opp": "opp", "behind": "bh", "beside": "bs",
    "district": "dist", "dist": "dist", "sector": "sec", "sec": "sec", "phase": "ph", "ph": "ph",
    "marg": "marg", "nagar": "ngr", "ngr": "ngr", "colony": "col", "col": "col",
    "first": "1st", "second": "2nd", "third": "3rd",
}
ADDR_MARKERS = {"no", "h", "hno", "door", "plot", "flat", "shop", "house", "box", "po", "pmb",
                "c", "o", "co", "city", "of", "county", "icty", "null", "none", "na", "the",
                "and", "de", "du", "des", "la", "le", "les", "l", "d", "bis", "ter", "half"}
_TYPES = set(ABBR.values())

US_STATES = {
    "al": "alabama", "ak": "alaska", "az": "arizona", "ar": "arkansas", "ca": "california",
    "co": "colorado", "ct": "connecticut", "de": "delaware", "fl": "florida", "ga": "georgia",
    "hi": "hawaii", "id": "idaho", "il": "illinois", "in": "indiana", "ia": "iowa", "ks": "kansas",
    "ky": "kentucky", "la": "louisiana", "me": "maine", "md": "maryland", "ma": "massachusetts",
    "mi": "michigan", "mn": "minnesota", "ms": "mississippi", "mo": "missouri", "mt": "montana",
    "ne": "nebraska", "nv": "nevada", "nh": "new hampshire", "nj": "new jersey",
    "nm": "new mexico", "ny": "new york", "nc": "north carolina", "nd": "north dakota",
    "oh": "ohio", "ok": "oklahoma", "or": "oregon", "pa": "pennsylvania", "ri": "rhode island",
    "sc": "south carolina", "sd": "south dakota", "tn": "tennessee", "tx": "texas", "ut": "utah",
    "vt": "vermont", "va": "virginia", "wa": "washington", "wv": "west virginia",
    "wi": "wisconsin", "wy": "wyoming", "dc": "district of columbia", "pr": "puerto rico",
}
IN_STATES = {
    "ap": "andhra pradesh", "ar": "arunachal pradesh", "as": "assam", "br": "bihar",
    "cg": "chhattisgarh", "ct": "chhattisgarh", "ga": "goa", "gj": "gujarat", "hr": "haryana",
    "hp": "himachal pradesh", "jh": "jharkhand", "ka": "karnataka", "kl": "kerala",
    "mp": "madhya pradesh", "mh": "maharashtra", "mn": "manipur", "ml": "meghalaya",
    "mz": "mizoram", "nl": "nagaland", "od": "odisha", "or": "odisha", "orissa": "odisha",
    "pb": "punjab", "rj": "rajasthan", "sk": "sikkim", "tn": "tamil nadu", "tg": "telangana",
    "ts": "telangana", "tr": "tripura", "up": "uttar pradesh", "uk": "uttarakhand",
    "ut": "uttarakhand", "uttaranchal": "uttarakhand", "wb": "west bengal", "dl": "delhi",
    "jk": "jammu and kashmir", "ch": "chandigarh", "py": "puducherry", "pondicherry": "puducherry",
    "la": "ladakh", "dn": "dadra and nagar haveli", "an": "andaman and nicobar islands",
}


def _state_map(abbr: dict[str, str]) -> dict[str, str]:
    m = {}
    for k, v in abbr.items():
        canon = "st_" + v.replace(" ", "")
        m[k] = canon
        m[v] = canon
        m[v.replace(" ", "")] = canon
    return m


STATE_MAPS = {"US": _state_map(US_STATES), "India": _state_map(IN_STATES)}
# learned aliases (e.g. native-script state names) are merged in at runtime
STATE_ALIASES: dict[str, dict[str, str]] = {}
# learned administrative parts (region / department ...) for countries without a state lexicon;
# dropped from tokens and locality (state stays unknown). Loaded from ER_ADMIN_PARTS (JSON file,
# {country: [normalised part, ...]}) so spawned worker processes see them too. See er.admin.
ADMIN_DROP: dict[str, set[str]] = {}
if os.environ.get("ER_ADMIN_PARTS") and os.path.exists(os.environ["ER_ADMIN_PARTS"]):
    import json as _json
    ADMIN_DROP = {c: set(v) for c, v in _json.load(open(os.environ["ER_ADMIN_PARTS"], encoding="utf-8")).items()}

_HALF = re.compile(r"(\d)\s*1/2\b")
_NUM = re.compile(r"\d+")
_ORDINAL = re.compile(r"^\d+(st|nd|rd|th)$")


def normalize_address(raw: str | None, country: str):
    """Return (tokens, nums, hno, state, loc_tokens, street_tokens)."""
    s = to_ascii(raw or "")
    s = _HALF.sub(r"\1 half", s)
    smap = STATE_MAPS.get(country, {})
    alias = STATE_ALIASES.get(country, {})
    drop = ADMIN_DROP.get(country, ())
    tokens: list[str] = []
    loc_parts: list[list[str]] = []
    street: list[str] = []
    nums: list[str] = []
    hno = ""
    state = ""
    for part in s.split(","):
        ptoks = _NON_ALNUM.sub(" ", part).split()
        if not ptoks:
            continue
        pj = " ".join(ptoks)
        if pj in drop:            # learned admin part (e.g. French region / department)
            continue
        st = smap.get(pj) or alias.get(pj)
        if st and not state:
            state = st
            tokens.append(st)
            continue
        has_digit = any(c.isdigit() for c in pj)
        ploc: list[str] = []
        for t in ptoks:
            if any(c.isdigit() for c in t):
                ordinal = bool(_ORDINAL.match(t))
                for n in _NUM.findall(t):
                    n = n.lstrip("0") or "0"
                    nums.append(n)
                    if not hno and not ordinal:
                        hno = n
                tokens.append(t)
                continue
            t = ABBR.get(t, t)
            if t in ADDR_MARKERS:
                continue
            tokens.append(t)
            if t in _TYPES or len(t) < 2:
                continue
            (street if has_digit else ploc).append(t)
        if ploc:
            loc_parts.append(ploc)
    # locality tokens ordered from the end of the address (city/district sit near the end)
    loc = [t for p in reversed(loc_parts) for t in p]
    if not hno and nums:
        hno = nums[0]
    return tokens, list(dict.fromkeys(nums)), hno, state, loc, street


def normalize_record(name, addr, country):
    name_norm, core, legal = normalize_name(name)
    toks, nums, hno, state, loc, street = normalize_address(addr, country)
    skel = [skeleton(t) for t in core if not t.isdigit()]
    return (name_norm, core, legal, skel, " ".join(toks), toks, nums, hno, state,
            list(dict.fromkeys(loc)), list(dict.fromkeys(street)))


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/output.py
"""Submission writers with hard invariants (checked before and after writing)."""
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import polars as pl


def id_lists(pairs: pl.DataFrame, s1_ids: pl.Series, col: str) -> pl.DataFrame:
    """pairs (s1, tid) -> one row per S1 in s1_ids, sorted unique comma-joined ids ('' if none)."""
    g = (pairs.unique(["s1", "tid"]).sort("s1", "tid")
              .group_by("s1", maintain_order=True).agg(pl.col("tid").str.join(",").alias(col)))
    out = (pl.DataFrame({"s1": s1_ids}).join(g, on="s1", how="left")
             .with_columns(pl.col(col).fill_null("")).sort("s1")
             .rename({"s1": "source1_entity_id"}))
    return out


def check_invariants(match: pl.DataFrame, cand: pl.DataFrame, s1_ids: pl.Series) -> None:
    """Fail loudly on anything the scorer would reject or that signals a pipeline bug."""
    n = s1_ids.n_unique()
    for name, df in (("matching", match), ("candidate", cand)):
        assert df.height == n, f"{name}: {df.height} rows != {n} test S1"
        assert df["source1_entity_id"].n_unique() == n, f"{name}: duplicate S1 rows"
    for name, pairs in (("matching", match), ("candidate", cand)):
        col = pairs.columns[1]
        ex = pairs.select(pl.col(col).str.split(",")).explode(col).filter(pl.col(col) != "")[col]
        assert ex.str.contains(r"^S[23]-").all(), f"{name}: non S2/S3 id present"
    mm = match.select("source1_entity_id", pl.col("matched_entity_ids").str.split(",")).explode("matched_entity_ids")
    mm = mm.filter(pl.col("matched_entity_ids") != "")
    cc = cand.select("source1_entity_id", pl.col("candidate_entity_ids").str.split(",")).explode("candidate_entity_ids")
    missing = mm.join(cc, left_on=["source1_entity_id", "matched_entity_ids"],
                      right_on=["source1_entity_id", "candidate_entity_ids"], how="anti")
    assert missing.height == 0, f"{missing.height} matches are not in candidate_pairs"
    per_s1 = mm.group_by("source1_entity_id").agg(pl.len().alias("n"),
                                                  pl.col("matched_entity_ids").n_unique().alias("u"))
    assert (per_s1["n"] == per_s1["u"]).all(), "duplicate id within a list"


def write_submission(matches: pl.DataFrame, cands: pl.DataFrame, s1_ids: pl.Series, out_dir: Path,
                     validator: Path | None = None, test_dir: Path | None = None) -> None:
    """matches/cands: long (s1, tid). Writes matching_results.tsv + candidate_pairs.tsv."""
    out_dir.mkdir(parents=True, exist_ok=True)
    cands = pl.concat([cands.select("s1", "tid"), matches.select("s1", "tid")]).unique()
    m = id_lists(matches, s1_ids, "matched_entity_ids")
    c = id_lists(cands, s1_ids, "candidate_entity_ids")
    check_invariants(m, c, s1_ids)
    m.write_csv(out_dir / "matching_results.tsv", separator="\t", quote_style="never")
    c.write_csv(out_dir / "candidate_pairs.tsv", separator="\t", quote_style="never")
    if validator is not None:
        r = subprocess.run([sys.executable, str(validator), "--matching", str(out_dir / "matching_results.tsv"),
                            "--candidate", str(out_dir / "candidate_pairs.tsv"), "--test-dir", str(test_dir)],
                           capture_output=True, text=True)
        print(r.stdout[-2000:])
        if r.returncode != 0:
            raise RuntimeError("official validator FAILED:\n" + r.stdout[-3000:])


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/percountry.py
"""Per-country decision experiment (evaluated: 0.000 gain vs global; not used).

v2's stage-2 scores re-decided with per-country tuned decision parameters.
python -m er.percountry
Reads work/runs/v0/test_scores.parquet (v2's calibrated stage-2 probabilities) and
work/runs/v0/decision_per_country.json (US / India parameters tuned on validation), applies the
expected-F0.5 selector per country (France: v2's validated global parameters) and writes
output/percountry/. Independent of v3 (separate work dir and outputs).
"""
from __future__ import annotations

import json
import logging

import polars as pl

from . import config as C
from .decide import DecisionParams, select
from .output import write_submission
from .pipeline import id_maps, load

log = logging.getLogger(__name__)


def run_percountry() -> None:
    rd = C.WORK / "runs" / "v0"
    glob = json.loads((rd / "decision_refined.json").read_text())["best"]
    per = json.loads((rd / "decision_per_country.json").read_text())
    test = pl.read_parquet(rd / "test_scores.parquet")
    ctry = load("test", "1", cols=["idx", "country"]).rename({"idx": "s1_idx"})
    test = test.join(ctry, on="s1_idx")
    sels = []
    for (c,), d in test.group_by("country"):
        prm = per.get(c, {}).get("params") if per.get(c, {}).get("gain", 0) > 0 else None
        prm = prm or {"m0": glob["m0"], "empty_bias": glob["empty_bias"], "margin": glob["margin"]}
        log.info("percountry %s: %s", c, prm)
        sels.append(select(d.drop("country"), DecisionParams(**prm)))
    sel = pl.concat(sels)
    s1_ids, tg_ids = id_maps("test")
    to_ids = lambda x: (x.join(s1_ids.select("s1_idx", "s1"), on="s1_idx")
                         .join(tg_ids, on=["t_idx", "src"]).select("s1", "tid"))
    write_submission(to_ids(sel), to_ids(test.select("s1_idx", "src", "t_idx")), s1_ids["s1"], C.OUT / "percountry",
                     validator=C.ROOT / "student_resource" / "utils" / "validate_submission.py",
                     test_dir=C.DATA / "test")
    log.info("percountry: %d matches, %d S1 with >=1 match", sel.height, sel["s1_idx"].n_unique())


if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s")
    run_percountry()


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/pipeline.py
"""Stage orchestration.

Stage A  (per country x target-source partition, ALL S1): blocking -> cheap string features
          -> context (within-S1) and competition (within-target) features. Competition
          features see every S1, exactly as at test time.
Stage 1  : 2-fold out-of-fold LightGBM on stage-A features -> prune to final candidate set.
Stage B  : rich pair features on the pruned set -> stage-2 model -> decision layer.
"""
from __future__ import annotations

import functools
import gc
import logging
import os

import numpy as np
import polars as pl
from rapidfuzz import fuzz, process

from . import config as C
from .blocking import block
from .prep import load_gt

log = logging.getLogger(__name__)

N_VAL = int(os.environ.get("ER_N_VAL", 150_000))   # held-out train S1 entities (validation, mirrors test)
N_FIT = int(os.environ.get("ER_N_FIT", 350_000))   # train S1 entities used to fit models


def load(split: str, src: str, country: str | None = None, cols: list[str] | None = None) -> pl.DataFrame:
    """Lazy, predicate-pushed load of a normalised table (optionally one country / some columns)."""
    q = pl.scan_parquet(C.WORK / f"{split}_s{src}.parquet")
    if country is not None:
        q = q.filter(pl.col("country") == country)
    if cols is not None:
        q = q.select(cols)
    return q.collect()


def train_roles() -> pl.DataFrame:
    """Deterministic S1 role assignment on train: fit / val (others unused for fitting)."""
    path = C.WORK / "train_roles.parquet"
    if path.exists():
        return pl.read_parquet(path)
    s1 = load("train", "1", cols=["entity_id", "idx", "country"]).sample(fraction=1.0, shuffle=True, seed=C.SEED)
    roles = pl.concat([
        s1.head(N_VAL).with_columns(pl.lit("val").alias("role")),
        s1.slice(N_VAL, N_FIT).with_columns(pl.lit("fit").alias("role")),
    ]).with_columns((pl.col("idx") % 2).cast(pl.UInt8).alias("fold"))
    roles.write_parquet(path)
    return roles


def _cheap_strings(pairs: pl.DataFrame, s1: pl.DataFrame, tg: pl.DataFrame, chunk: int = 4_000_000):
    """Core-name and address token-set ratios, computed in chunks to bound memory."""
    out_c, out_a = [], []
    for i in range(0, pairs.height, chunk):
        p = pairs.slice(i, chunk).select("s1_idx", "t_idx")
        a = p.join(s1, left_on="s1_idx", right_on="idx", how="left", maintain_order="left")
        b = p.join(tg, left_on="t_idx", right_on="idx", how="left", maintain_order="left")
        out_c.append(process.cpdist(a["core_s"].to_list(), b["core_s"].to_list(),
                                    scorer=fuzz.token_set_ratio, workers=-1, dtype=np.uint8))
        out_a.append(process.cpdist(a["addr_norm"].to_list(), b["addr_norm"].to_list(),
                                    scorer=fuzz.token_set_ratio, workers=-1, dtype=np.uint8))
    return np.concatenate(out_c), np.concatenate(out_a)


def cx_features(g: pl.DataFrame) -> pl.DataFrame:
    """Context features within each S1 (exact inside a blocking chunk)."""
    return g.with_columns(
        pl.len().over("s1_idx").cast(pl.UInt16).alias("cx_n"),
        (pl.col("bs") / pl.col("bs").max().over("s1_idx")).alias("cx_rel"),
        (pl.col("c_tset").cast(pl.Int16) - pl.col("c_tset").max().over("s1_idx")).alias("cx_gap_c"),
        (pl.col("a_tset").cast(pl.Int16) - pl.col("a_tset").max().over("s1_idx")).alias("cx_gap_a"),
        pl.col("c_tset").rank("ordinal", descending=True).over("s1_idx").cast(pl.UInt16).alias("cx_rank_c"),
    )


def target_aggs(files: list, score: str, extra: tuple[str, ...] = ()) -> pl.DataFrame:
    """Streaming per-target aggregates over all chunk files: count, max and runner-up of
    ``score`` and max of ``extra`` columns. Memory is proportional to #targets, not #pairs."""
    q = pl.scan_parquet(files).group_by("t_idx").agg(
        pl.len().cast(pl.UInt16).alias("tx_n"),
        pl.col(score).max().alias("_m1"),
        pl.col(score).top_k(2).min().alias("_m2"),
        *[pl.col(c).max().alias(f"_mx_{c}") for c in extra])
    return q.collect(engine="streaming")


def tx_features(g: pl.DataFrame, aggs: pl.DataFrame, score: str, extra: tuple[str, ...] = (),
                tag: str = "") -> pl.DataFrame:
    """Competition features of a pair against every S1 competing for the same target."""
    g = g.join(aggs, on="t_idx", how="left")
    best = pl.col(score) >= pl.col("_m1")
    other = pl.when(best).then(pl.when(pl.col("tx_n") <= 1).then(0.0).otherwise(pl.col("_m2"))
                               ).otherwise(pl.col("_m1"))
    g = g.with_columns(
        pl.when(best).then(1).when(pl.col(score) >= pl.col("_m2")).then(2).otherwise(3)
          .cast(pl.UInt8).alias(f"tx_rank{tag}"),
        (pl.col(score) - pl.col("_m1")).alias(f"tx_gap{tag}"),
        (pl.col(score) / pl.col("_m1")).alias(f"tx_rel{tag}"),
        (pl.col(score) - other).alias(f"tx_margin{tag}"),   # lead over the best *other* S1
        *[(pl.col(c).cast(pl.Int16) - pl.col(f"_mx_{c}").cast(pl.Int16)).alias(f"tx_gap_{c[0]}{tag}")
          for c in extra],
    )
    g = g.drop(["_m1", "_m2", *[f"_mx_{c}" for c in extra]])
    return g.rename({"tx_n": f"tx_n{tag}"}) if tag else g


def _str_view(df: pl.DataFrame) -> pl.DataFrame:
    return df.select("idx", pl.col("core").list.join(" ").alias("core_s"),
                     pl.col("addr_norm").fill_null(""), pl.col("hno").fill_null(""))


def _exact_flags(g: pl.DataFrame, a: pl.DataFrame, t: pl.DataFrame) -> pl.DataFrame:
    """House-number equality and missing-target-address flags (cheap, exact)."""
    return (g.join(a.select(pl.col("idx").alias("s1_idx"), pl.col("hno").alias("_h1")), on="s1_idx", how="left")
             .join(t.select(pl.col("idx").alias("t_idx"), pl.col("hno").alias("_h2"),
                            (pl.col("addr_norm").fill_null("") == "").cast(pl.UInt8).alias("a_missing2")),
                   on="t_idx", how="left")
             .with_columns(pl.when((pl.col("_h1") == "") | (pl.col("_h2") == "")).then(-1)
                             .otherwise((pl.col("_h1") == pl.col("_h2")).cast(pl.Int8))
                             .cast(pl.Int8).alias("h_eq"))
             .drop("_h1", "_h2"))


def _done(d) -> bool:
    return (d / "_SUCCESS").exists()


def _finish(d) -> None:
    (d / "_SUCCESS").write_text("ok")


def _files(d) -> list:
    return sorted(d.glob("part_*.parquet"))


def run_stage_a(split: str) -> None:
    """Memory-bounded: blocking chunks are featurised and spilled to disk immediately; the
    competition features come from streaming per-target aggregates joined back per chunk."""
    import shutil
    cols = ["idx", "country", "core", "skel", "loc", "street", "nums", "addr_norm", "hno", "business_address"]
    countries = load(split, "1", cols=["country"])["country"].unique().sort().to_list()
    for country in countries:
        a = None
        for src in ("2", "3"):
            out = C.WORK / f"{split}_A_{country}_{src}"
            if _done(out):
                continue
            shutil.rmtree(out, ignore_errors=True)
            tmp = out / "tmp"
            tmp.mkdir(parents=True)
            if a is None:
                a = load(split, "1", country, cols)
                if split == "train" and C.TRAIN_ROLES_ONLY:   # fast path: only fit/val S1
                    a = a.join(train_roles().select("idx"), on="idx", how="semi")
            t = load(split, src, country, cols)
            if a.height == 0 or t.height == 0:
                _finish(out)
                continue
            av, tv = _str_view(a), _str_view(t)

            def on_chunk(g: pl.DataFrame, ch: int, a=a, t=t, av=av, tv=tv, tmp=tmp, src=src) -> None:
                g = g.with_columns(pl.col("bs", "bs_name", "bs_addr", "bs_pure", "arcs").cast(pl.Float32),
                                   pl.col("nkeys", "nfam", "nk_name", "nk_addr", "brank", "brank_n",
                                          "brank_a", "brank_p").cast(pl.UInt16))
                c_tset, a_tset = _cheap_strings(g, av, tv)
                g = g.with_columns(pl.Series("c_tset", c_tset), pl.Series("a_tset", a_tset))
                g = cx_features(_exact_flags(g, a, t)).with_columns(pl.lit(int(src), pl.UInt8).alias("src"))
                g.write_parquet(tmp / f"part_{ch:05d}.parquet")

            block(a, t, cap_single=C.CAP_SINGLE, cap_pair=C.CAP_PAIR, join_budget=C.JOIN_BUDGET, on_chunk=on_chunk,
                  **C.BUDGETS)
            del t, tv
            gc.collect()
            files = _files(tmp)
            aggs = target_aggs(files, "bs", ("c_tset", "a_tset"))
            for f in files:
                tx_features(pl.read_parquet(f), aggs, "bs", ("c_tset", "a_tset")).write_parquet(out / f.name)
            shutil.rmtree(tmp)
            _finish(out)
            log.info("stage A %s %s S%s done (%d chunks)", split, country, src, len(files))
            gc.collect()


def id_maps(split: str):
    s1 = load(split, "1", cols=["idx", "entity_id", "country"]).rename({"idx": "s1_idx", "entity_id": "s1"})
    tg = pl.concat([load(split, s, cols=["idx", "entity_id"]).rename({"idx": "t_idx", "entity_id": "tid"})
                    .with_columns(pl.lit(int(s), pl.UInt8).alias("src")) for s in ("2", "3")])
    return s1, tg


def labels(split: str = "train") -> pl.DataFrame:
    """(s1_idx, src, t_idx) of true matches."""
    s1, tg = id_maps(split)
    return (load_gt().join(s1.select("s1", "s1_idx"), on="s1").join(tg, on="tid")
            .select("s1_idx", "src", "t_idx"))


# ---------------------------------------------------------------------------
# Stage 1: out-of-fold pruning ranker
# ---------------------------------------------------------------------------
S1_FEATS = ["bs", "bs_name", "bs_addr", "bs_pure", "arcs", "nk_name", "nk_addr", "fam_mask",
            "brank_p", "h_eq", "a_missing2",
            "nkeys", "nfam", "brank", "brank_n", "brank_a", "c_tset",
            "a_tset", "cx_n", "cx_rel", "cx_gap_c", "cx_gap_a", "cx_rank_c", "tx_n", "tx_rank",
            "tx_gap", "tx_rel", "tx_margin", "tx_gap_c", "tx_gap_a", "src"]
S1_PARAMS = dict(objective="binary", learning_rate=0.08, num_leaves=127, min_data_in_leaf=200,
                 feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                 num_threads=C.WORKERS, verbose=-1, seed=C.SEED)
K1 = 20          # max candidates per S1 per source kept after stage 1
EPS1 = float(os.environ.get("ER_EPS1", 0.002))   # min stage-1 probability to keep


def _parts(split: str, stage: str):
    """Stage outputs named {split}_{stage}_{country}_{src}: directories (A) or .parquet files (P, B)."""
    return sorted(p for p in C.WORK.glob(f"{split}_{stage}_*") if p.is_dir() or p.suffix == ".parquet")


def s1_feats() -> list[str]:
    return [f for f in S1_FEATS if not (C.NO_TX and f.startswith("tx_"))]


def train_stage1() -> None:
    import lightgbm as lgb
    paths = [C.WORK / f"stage1_f{f}.lgb" for f in (0, 1)]
    if all(p.exists() for p in paths):
        return
    roles = train_roles().filter(pl.col("role") == "fit").select(pl.col("idx").alias("s1_idx"), "fold")
    lab = labels().with_columns(pl.lit(1, pl.UInt8).alias("y"))
    neg_mod = int(os.environ.get("ER_S1_NEG_MOD", 4))   # keep 1/neg_mod negatives, weight neg_mod (unbiased)
    data = []
    for d_ in _parts("train", "A"):
        d = (pl.scan_parquet(_files(d_)).join(roles.lazy(), on="s1_idx")
             .join(lab.lazy(), on=["s1_idx", "src", "t_idx"], how="left").with_columns(pl.col("y").fill_null(0))
             .filter((pl.col("y") == 1) | (pl.col("t_idx").hash(C.SEED) % neg_mod == 0))   # 1/neg_mod of negatives
             .select(*s1_feats(), "y", "fold").collect(engine="streaming"))
        data.append(d.with_columns(pl.when(pl.col("y") == 1).then(1.0).otherwise(float(neg_mod)).alias("w")))
    data = pl.concat(data)
    log.info("stage1 train rows=%d pos=%d", data.height, data["y"].sum())
    for f in (0, 1):
        tr, va = data.filter(pl.col("fold") != f), data.filter(pl.col("fold") == f)
        fs = s1_feats()
        dtr = lgb.Dataset(tr.select(fs).to_numpy(), tr["y"].to_numpy(), weight=tr["w"].to_numpy(),
                          feature_name=fs)
        dva = lgb.Dataset(va.select(fs).to_numpy(), va["y"].to_numpy(), weight=va["w"].to_numpy(),
                          reference=dtr)
        m = lgb.train(S1_PARAMS, dtr, 600, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(100)])
        m.save_model(str(paths[f]))
        log.info("stage1 fold %d best_iter=%d", f, m.best_iteration)


def _stage1_predict(d: pl.DataFrame, models, fold: pl.Series | None) -> np.ndarray:
    X = d.select(s1_feats()).to_numpy()
    if fold is None and os.environ.get("ER_S1_SINGLE") == "1":   # test-time speed: one fold model
        return models[0].predict(X, num_threads=C.WORKERS)
    p = [m.predict(X, num_threads=C.WORKERS) for m in models]
    avg = (p[0] + p[1]) / 2
    if fold is None:
        return avg
    f = fold.to_numpy()
    # out-of-fold for fit rows: model trained on the other fold
    return np.where(f == 0, p[0], np.where(f == 1, p[1], avg))


def run_stage1(split: str) -> None:
    """Score every retrieved pair (two streaming passes), then prune to the final candidate set."""
    import shutil

    import lightgbm as lgb
    models = [lgb.Booster(model_file=str(C.WORK / f"stage1_f{f}.lgb")) for f in (0, 1)]
    roles = train_roles() if split == "train" else None
    fm = (roles.filter(pl.col("role") == "fit").select(pl.col("idx").alias("s1_idx"), "fold")
          if roles is not None else None)
    for d_ in _parts(split, "A"):
        out = C.WORK / (d_.name.replace("_A_", "_P_") + ".parquet")
        if out.exists() or not _done(d_):   # skip finished outputs and still-running upstream partitions
            continue
        tmp = C.WORK / d_.name.replace("_A_", "_S_")
        shutil.rmtree(tmp, ignore_errors=True)
        tmp.mkdir()
        for f in _files(d_):  # pass 1: score + rank within S1 (complete inside a chunk)
            d = pl.read_parquet(f)
            if fm is not None:
                d = d.join(fm, on="s1_idx", how="left")
                p1 = _stage1_predict(d, models, d["fold"].fill_null(9))
                d = d.drop("fold")
            else:
                p1 = _stage1_predict(d, models, None)
            d = d.with_columns(pl.Series("p1", p1.astype(np.float32)))
            d.with_columns(pl.col("p1").rank("ordinal", descending=True).over("s1_idx").cast(pl.UInt16)
                           .alias("r1")).write_parquet(tmp / f.name)
        files = _files(tmp)  # pass 2: p1 competition over ALL S1, then prune
        extra = (pl.scan_parquet(files).group_by("t_idx").agg(
                     pl.col("p1").sum().alias("tx_sum_p1"),
                     (pl.col("p1") >= 0.5).sum().cast(pl.UInt16).alias("tx_n_hi"))
                 .collect(engine="streaming"))
        aggs = target_aggs(files, "p1")
        keep = []
        for f in files:
            d = pl.scan_parquet(f).filter((pl.col("r1") <= K1) & (pl.col("p1") >= EPS1))
            if roles is not None:
                d = d.join(roles.lazy().select(pl.col("idx").alias("s1_idx")), on="s1_idx", how="semi")
            d = d.collect().join(extra, on="t_idx", how="left")
            keep.append(tx_features(d, aggs, "p1", tag="_p1"))
        tmp_out = out.with_suffix(".part")
        pl.concat(keep).write_parquet(tmp_out)
        tmp_out.replace(out)            # atomic: downstream never sees a half-written file
        shutil.rmtree(tmp)
        log.info("stage1 %s -> %d pairs", out.name, sum(k.height for k in keep))


# ---------------------------------------------------------------------------
# Stage B: rich pair features on the pruned candidate set
# ---------------------------------------------------------------------------
from .features import REC_COLS, name_freq, pair_features, token_idf  # noqa: E402

META = {"s1_idx", "t_idx", "country", "fold", "role", "y", "w"}


def _part_meta(path) -> tuple[str, str, str]:
    """{split}_{stage}_{country}_{src}[.parquet] -> (split, country, src)."""
    stem = path.name.removesuffix(".parquet")
    split, _, rest = stem.split("_", 2)
    country, src = rest.rsplit("_", 1)
    return split, country, src


def _anchors(split: str, country: str) -> pl.DataFrame:
    """Top stage-1 candidate per (S1, source): the anchor for cluster-coherence features."""
    ps = [C.WORK / f"{split}_P_{country}_{s}.parquet" for s in ("2", "3")]
    d = pl.concat([pl.read_parquet(p, columns=["s1_idx", "src", "t_idx", "p1"]) for p in ps if p.exists()])
    return (d.sort("p1", descending=True).group_by("s1_idx", "src", maintain_order=True).first()
             .rename({"t_idx": "a_idx", "p1": "a_p1"}))


def _idf_stream(split: str, country: str, col: str) -> pl.DataFrame:
    """Same result as features.token_idf over the three sources, computed out-of-core."""
    scans = [pl.scan_parquet(C.WORK / f"{split}_s{s}.parquet").filter(pl.col("country") == country)
             for s in ("1", "2", "3")]
    n = sum(q.select(pl.len()).collect().item() for q in scans)
    df = (pl.concat([q.select(pl.col(col).list.unique().alias("tok")) for q in scans])
            .explode("tok").drop_nulls("tok").group_by("tok").len("df").collect(engine="streaming"))
    return df.select(pl.lit(country).alias("country"), "tok",
                     ((n + 1) / (pl.col("df") + 1)).log().cast(pl.Float32).alias("idf"))


def _name_freq_stream(split: str, country: str, src: str, suffix: str) -> pl.DataFrame:
    return (pl.scan_parquet(C.WORK / f"{split}_s{src}.parquet").filter(pl.col("country") == country)
              .group_by("name_norm").len(f"nfreq{suffix}").rename({"name_norm": f"name_norm{suffix}"})
              .collect(engine="streaming"))


def _country_tables(split: str, country: str, aux):
    """Record tables restricted to records that occur in this country's candidate pairs, plus
    corpus statistics (IDF, name frequency) computed over the *full* tables, column-pruned."""
    ps = [C.WORK / f"{split}_P_{country}_{s}.parquet" for s in ("2", "3")]
    pairs = pl.concat([pl.read_parquet(p, columns=["s1_idx", "src", "t_idx"]) for p in ps if p.exists()])
    need = {"1": pairs.select(pl.col("s1_idx").alias("idx")).unique()}
    for s in ("2", "3"):
        need[s] = pairs.filter(pl.col("src") == int(s)).select(pl.col("t_idx").alias("idx")).unique()
    # corpus statistics via streaming aggregation (tables are never materialised in memory)
    idf_c, idf_a = _idf_stream(split, country, "core"), _idf_stream(split, country, "atoks")
    nf1 = _name_freq_stream(split, country, "1", "_1")
    nf2 = {s: _name_freq_stream(split, country, s, "_2") for s in ("2", "3")}
    gc.collect()
    tabs = {}
    for s in ("1", "2", "3"):
        t = (pl.scan_parquet(C.WORK / f"{split}_s{s}.parquet").filter(pl.col("country") == country)
               .select("idx", *REC_COLS).join(need[s].lazy(), on="idx", how="semi").collect())
        tabs[s] = aux.enrich(t) if aux is not None else t
    return tabs, idf_c, idf_a, nf1, nf2, _anchors(split, country)


def run_stage_b(split: str, chunk: int = int(os.environ.get("ER_B_CHUNK", 500_000))) -> None:
    import shutil

    from .auxfit import AUX, Aux
    from .features import coherence_features
    aux = Aux() if (AUX / "addr_alias.json").exists() else None
    if aux is None:
        log.warning("stage B without aux models (run `er.run aux` first for alias/noise features)")
    cache: dict[str, tuple] = {}
    for p in _parts(split, "P"):
        out = C.WORK / p.name.removesuffix(".parquet").replace("_P_", "_B_")
        if _done(out):
            continue
        _, country_, _ = _part_meta(p)
        only = os.environ.get("ER_COUNTRIES")
        if only and country_ not in only.split(","):
            continue
        # coherence anchors need BOTH sources' stage-1 output for this country
        if not all((C.WORK / f"{split}_P_{country_}_{s}.parquet").exists() for s in ("2", "3")):
            continue
        lock = C.WORK / f"{out.name}.lock"   # lets several stage-B workers share the partitions safely
        try:
            fd = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            os.write(fd, str(os.getpid()).encode())
            os.close(fd)
        except FileExistsError:
            continue
        shutil.rmtree(out, ignore_errors=True)
        out.mkdir()
        _, country, src = _part_meta(p)
        if country not in cache:
            cache.clear()
            gc.collect()
            cache[country] = _country_tables(split, country, aux)
        tabs, idf_c, idf_a, nf1, nf2, anchors = cache[country]
        tg = tabs[src].with_columns(pl.lit(int(src), pl.UInt8).alias("src"))
        d = pl.read_parquet(p)
        minp1 = float(os.environ.get("ER_B_MINP1", 0))   # optional extra stage-1 floor (time budget)
        if minp1 > 0:
            d = d.filter(pl.col("p1") >= minp1)
        for k, i in enumerate(range(0, d.height, chunk)):
            f = pair_features(d.slice(i, chunk), tabs["1"], tg, idf_c, idf_a, nf1, nf2[src])
            coherence_features(f, tabs, anchors).drop("country", strict=False).write_parquet(
                out / f"part_{k:05d}.parquet")
        _finish(out)
        lock.unlink(missing_ok=True)
        log.info("stage B %s: %d pairs", out.name, d.height)
        del d
        gc.collect()


# ---------------------------------------------------------------------------
# Stage 2: matcher, calibration, decision tuning
# ---------------------------------------------------------------------------
S2_PARAMS = dict(objective="binary", learning_rate=float(os.environ.get("ER_S2_LR", 0.05)),
                 num_leaves=int(os.environ.get("ER_S2_LEAVES", 255)), min_data_in_leaf=100,
                 feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1, lambda_l2=3.0,
                 max_bin=255, num_threads=C.WORKERS, verbose=-1, seed=C.SEED)
S2_ROUNDS = int(os.environ.get("ER_S2_ROUNDS", 3000))
S2_EARLY = int(os.environ.get("ER_S2_EARLY", 100))


SIB = os.environ.get("ER_SIB", "0") == "1"   # sibling-aware target features (France: 21% of S1 have siblings)


@functools.lru_cache(maxsize=2)
def sibling_aggs(split: str) -> pl.DataFrame:
    """Per target, over ALL its stage-A S1 candidates (every S1, the same view as test): how many share its house
    number / have a different one, the best name similarity among same-number candidates, how many look alike by
    name. Siblings (same brand + street, other house number) are 21% of French S1 vs 7% India, 0.3% US."""
    path = C.WORK / f"sib_{split}.parquet"
    if not path.exists():
        files = [f for d in _parts(split, "A") for f in _files(d)]
        agg = (pl.scan_parquet(files).select("src", "t_idx", "h_eq", "c_tset")
                 .group_by("src", "t_idx").agg(
                     (pl.col("h_eq") == 1).sum().cast(pl.Float32).alias("sib_heq_n"),
                     (pl.col("h_eq") == 0).sum().cast(pl.Float32).alias("sib_hne_n"),
                     pl.when(pl.col("h_eq") == 1).then(pl.col("c_tset")).otherwise(None).max().cast(pl.Float32).alias("sib_heq_cmax"),
                     (pl.col("c_tset") >= 80).sum().cast(pl.Float32).alias("sib_alike_n"),
                     pl.col("c_tset").max().cast(pl.Float32).alias("sib_cmax"))
                 .collect(engine="streaming"))
        tmp = path.with_suffix(".part")
        agg.write_parquet(tmp)
        tmp.replace(path)
    return pl.read_parquet(path)


def _add_sib(lf: pl.LazyFrame, split: str) -> pl.LazyFrame:
    s = sibling_aggs(split)
    return (lf.join(s.lazy(), on=["src", "t_idx"], how="left")
              .with_columns((pl.col("sib_heq_n").fill_null(0) - (pl.col("h_eq") == 1).cast(pl.Float32)).alias("sib_heq_oth"),
                            (pl.col("c_tset") - pl.col("sib_heq_cmax")).alias("sib_heq_cgap"),
                            (pl.col("c_tset") - pl.col("sib_cmax")).alias("sib_cgap")))


def _scan_b(split: str) -> pl.LazyFrame:
    files = [f for d in _parts(split, "B") for f in _files(d)]
    lf = pl.scan_parquet(files, missing_columns="insert", extra_columns="ignore")
    return _add_sib(lf, split) if SIB else lf


def feature_cols(schema) -> list[str]:
    return [c for c, t in schema.items() if c not in META and t.is_numeric()
            and not (C.NO_TX and c.startswith("tx_"))]


def _predict_b(split: str, models, feats, cal, roles: pl.DataFrame | None = None, role: str | None = None
               ) -> pl.DataFrame:
    """Chunk-wise scoring of stage-B parts -> (s1_idx, src, t_idx, p[, y])."""
    out = []
    for d_ in _parts(split, "B"):
        for f in _files(d_):
            d = pl.read_parquet(f)
            if SIB:
                d = _add_sib(d.lazy(), split).collect()
            if roles is not None:
                d = d.join(roles.filter(pl.col("role") == role).select("s1_idx"), on="s1_idx", how="semi")
            if d.height == 0:
                continue
            keep = ["s1_idx", "src", "t_idx"] + (["y"] if "y" in d.columns else [])
            out.append(d.select(keep).with_columns(pl.Series("p", predict2(d, models, feats, cal))))
    return pl.concat(out)


def run_train2(run_name: str = "v1") -> dict:
    """2-fold OOF LightGBM on fit S1, isotonic calibration on OOF, decision tuning on val."""
    import json
    import pickle

    import lightgbm as lgb
    from sklearn.isotonic import IsotonicRegression

    from .decide import DecisionParams, select, threshold_select
    from .metric import macro_f05

    run_dir = C.WORK / "runs" / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    roles = train_roles().select(pl.col("idx").alias("s1_idx"), "role", "fold")
    lab = labels().with_columns(pl.lit(1, pl.UInt8).alias("y"))
    scan = _scan_b("train")
    feats = feature_cols(scan.collect_schema())
    fit = (scan.join(roles.lazy().filter(pl.col("role") == "fit"), on="s1_idx")
               .join(lab.lazy(), on=["s1_idx", "src", "t_idx"], how="left")
               .with_columns(pl.col("y").fill_null(0))
               .select("s1_idx", "t_idx", "y", "fold", *[pl.col(c).cast(pl.Float32) for c in feats])
               .collect(engine="streaming"))
    log.info("stage2: fit=%d (pos %d) feats=%d", fit.height, fit["y"].sum(), len(feats))
    models, oof = [], np.zeros(fit.height, dtype=np.float32)
    fold = fit["fold"].to_numpy()
    y = fit["y"].to_numpy()
    X = fit.select(feats).to_numpy()
    meta = fit.select("s1_idx", "src", "t_idx", "y")
    del fit
    gc.collect()
    # optional self-training rows (e.g. confident France test pseudo-labels): appended to the TRAINING
    # folds only - never to OOF calibration or validation, so all reported metrics stay label-honest
    Xp = yp = None
    if os.environ.get("ER_PSEUDO"):
        ps = pl.read_parquet(os.environ["ER_PSEUDO"]).select("s1_idx", "src", "t_idx", pl.col("y_pseudo").cast(pl.Int32))
        pb = (_scan_b("test").join(ps.lazy(), on=["s1_idx", "src", "t_idx"])
              .select("y_pseudo", *[pl.col(c).cast(pl.Float32) for c in feats]).collect(engine="streaming"))
        Xp, yp = pb.select(feats).to_numpy(), pb["y_pseudo"].to_numpy()
        wpseudo = float(os.environ.get("ER_PSEUDO_W", 1.0))
        log.info("stage2: +%d pseudo-labelled rows (pos %d, weight %.2f)", len(yp), int(yp.sum()), wpseudo)
        del pb
    for f in (0, 1):
        tr, va = fold != f, fold == f
        if Xp is not None:
            dtr = lgb.Dataset(np.vstack([X[tr], Xp]), np.concatenate([y[tr], yp]), feature_name=feats,
                              weight=np.concatenate([np.ones(int(tr.sum())), np.full(len(yp), wpseudo)]),
                              free_raw_data=True)
        else:
            dtr = lgb.Dataset(X[tr], y[tr], feature_name=feats, free_raw_data=True)
        dva = lgb.Dataset(X[va], y[va], reference=dtr)
        m = lgb.train(S2_PARAMS, dtr, S2_ROUNDS, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(S2_EARLY, verbose=False), lgb.log_evaluation(250)])
        oof[va] = m.predict(X[va], num_threads=C.WORKERS)
        m.save_model(str(run_dir / f"stage2_f{f}.lgb"))
        models.append(m)
        log.info("stage2 fold %d best_iter=%d", f, m.best_iteration)
        del dtr, dva
    del X
    gc.collect()
    cal = {}  # calibration per source on out-of-fold predictions
    for s in (2, 3):
        msk = meta["src"].to_numpy() == s
        cal[s] = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(oof[msk], y[msk])
    with open(run_dir / "calibrators.pkl", "wb") as fh:
        pickle.dump(cal, fh)
    meta.with_columns(pl.Series("oof", oof)).write_parquet(run_dir / "fit_oof.parquet")
    val = (_predict_b("train", models, feats, cal, roles, "val")
           .join(lab, on=["s1_idx", "src", "t_idx"], how="left").with_columns(pl.col("y").fill_null(0)))
    val.write_parquet(run_dir / "val_scores.parquet")
    # decision tuning on the untouched validation S1 set (incl. S1 with no candidates)
    s1_ids, tg_ids = id_maps("train")
    val_s1 = roles.filter(pl.col("role") == "val").join(s1_ids, on="s1_idx")
    truth = load_gt().join(val_s1.select("s1"), on="s1")

    def score(sel: pl.DataFrame, s1s: pl.DataFrame = val_s1, tr: pl.DataFrame = truth) -> dict:
        pred = sel.join(s1s.select("s1_idx", "s1"), on="s1_idx").join(tg_ids, on=["t_idx", "src"])
        return macro_f05(pred.select("s1", "tid"), tr, s1s["s1"])

    results = [{"kind": "thr", "thr": thr, **score(threshold_select(val, thr))} for thr in (0.3, 0.4, 0.5, 0.6, 0.7)]
    best = None
    for m0 in (0.0, 0.2, 0.5):
        for eb in (0.5, 1.0, 1.5):
            for margin in (0.0, 0.1):
                prm = DecisionParams(margin=margin, m0=m0, empty_bias=eb)
                r = {"kind": "ef", "m0": m0, "empty_bias": eb, "margin": margin, **score(select(val, prm))}
                results.append(r)
                if best is None or r["f05"] > best["f05"]:
                    best = r
    res = pl.DataFrame(results, infer_schema_length=None).sort("f05", descending=True)
    print(res.head(12))
    res.write_csv(run_dir / "decision_grid.csv")
    imp = sorted(zip(feats, models[0].feature_importance("gain")), key=lambda x: -x[1])
    summary = {"run": run_name, "features": feats, "best": best, "val_by_country": {},
               "top_features": [(f, float(g)) for f, g in imp[:40]]}
    sel = select(val, DecisionParams(margin=best["margin"], m0=best["m0"], empty_bias=best["empty_bias"]))
    for (country,), g in val_s1.group_by("country"):
        summary["val_by_country"][country] = score(sel, g, truth.join(g.select("s1"), on="s1"))
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=1, default=str))
    log.info("stage2 best %s | by country %s", best, summary["val_by_country"])
    return summary


def predict2(df: pl.DataFrame, models, feats, cal) -> np.ndarray:
    X = df.select([pl.col(f).cast(pl.Float32) if f in df.columns else pl.lit(None, pl.Float32).alias(f)
                   for f in feats]).to_numpy()
    raw = np.mean([m.predict(X, num_threads=C.WORKERS) for m in models], axis=0)
    out = raw.copy()  # sources without a calibrator keep raw scores
    src = df["src"].to_numpy()
    for s, c in cal.items():
        msk = src == s
        if msk.any():
            out[msk] = c.predict(raw[msk])
    return out.astype(np.float32)


# ---------------------------------------------------------------------------
# Test inference -> submission
# ---------------------------------------------------------------------------
def run_predict(run_name: str = "v1", tag: str = "") -> None:
    import json
    import pickle

    import lightgbm as lgb

    from .decide import DecisionParams, select
    from .output import write_submission

    run_dir = C.WORK / "runs" / run_name
    summ = json.loads((run_dir / "summary.json").read_text())
    feats, best = summ["features"], summ["best"]
    sfx = f"_{tag}" if tag else ""
    dec_file = run_dir / os.environ.get("ER_DECISION", f"decision{sfx}.json")
    if dec_file.exists():   # tuned decision layer (er.run tune / refined grid) takes precedence
        best = json.loads(dec_file.read_text())["best"]
    models = [lgb.Booster(model_file=str(run_dir / f"stage2_f{f}.lgb")) for f in (0, 1)]
    with open(run_dir / "calibrators.pkl", "rb") as fh:
        cal = pickle.load(fh)
    if tag:  # scores precomputed by another scorer (e.g. the ensemble)
        d = pl.read_parquet(run_dir / f"test_scores{sfx}.parquet")
    else:
        d = _predict_b("test", models, feats, cal)
        d.write_parquet(run_dir / "test_scores.parquet")
    from .decide import select_exact
    fn = select_exact if best.get("kind") == "exact" else select
    sel = fn(d, DecisionParams(margin=best["margin"], m0=best["m0"], empty_bias=best["empty_bias"], coh=best.get("coh", 0.0)))
    log.info("predict: decision %s", best)
    s1_ids, tg_ids = id_maps("test")
    to_ids = lambda x: (x.join(s1_ids.select("s1_idx", "s1"), on="s1_idx")
                         .join(tg_ids, on=["t_idx", "src"]).select("s1", "tid"))
    out_dir = C.OUT / os.environ["ER_SUB_NAME"] if os.environ.get("ER_SUB_NAME") else C.OUT
    write_submission(to_ids(sel), to_ids(d.select("s1_idx", "src", "t_idx")), s1_ids["s1"], out_dir,
                     validator=C.ROOT / "student_resource" / "utils" / "validate_submission.py",
                     test_dir=C.DATA / "test")
    log.info("predict: %d matches over %d test S1 (%d with >=1 match)", sel.height, s1_ids.height,
             sel["s1_idx"].n_unique())


# ---------------------------------------------------------------------------
# Decision tuning from saved validation scores (no retraining)
# ---------------------------------------------------------------------------
COH_GRID = tuple(float(x) for x in os.environ.get("ER_COH_GRID", "0,1").split(","))


def run_tune(run_name: str = "v1", tag: str = "") -> dict:
    import json

    from .decide import DecisionParams, select, select_exact
    from .metric import macro_f05

    rd = C.WORK / "runs" / run_name
    sfx = f"_{tag}" if tag else ""
    val = pl.read_parquet(rd / f"val_scores{sfx}.parquet")
    roles = train_roles().select(pl.col("idx").alias("s1_idx"), "role", "country")
    s1_ids, tg_ids = id_maps("train")
    val_s1 = roles.filter(pl.col("role") == "val").join(s1_ids.select("s1_idx", "s1"), on="s1_idx")
    truth = load_gt().join(val_s1.select("s1"), on="s1")

    def score(sel, s1s=val_s1, tr=truth):
        pred = sel.join(s1s.select("s1_idx", "s1"), on="s1_idx").join(tg_ids, on=["t_idx", "src"])
        return macro_f05(pred.select("s1", "tid"), tr, s1s["s1"])

    rows = []
    for kind, fn in (("ratio", select), ("exact", select_exact)):
        for m0 in (0.2, 0.5, 0.75):
            for eb in (1.0, 1.5, 2.0, 2.5):
                for margin in (0.05, 0.1):
                    for coh in COH_GRID:     # one-to-one coherent posterior off / on (decide N3); ER_COH_GRID
                        prm = DecisionParams(margin=margin, m0=m0, empty_bias=eb, coh=coh)
                        rows.append({"kind": kind, "m0": m0, "empty_bias": eb, "margin": margin, "coh": coh,
                                     **score(fn(val, prm))})
    res = pl.DataFrame(rows).sort("f05", descending=True)
    res.write_csv(rd / f"tune_grid{sfx}.csv")
    best = res.row(0, named=True)
    fn = select_exact if best["kind"] == "exact" else select
    sel = fn(val, DecisionParams(margin=best["margin"], m0=best["m0"], empty_bias=best["empty_bias"], coh=best.get("coh", 0.0)))
    by_c = {c: score(sel, g, truth.join(g.select("s1"), on="s1"))
            for (c,), g in val_s1.group_by("country")}
    out = {"best": best, "by_country": by_c,
           "best_ratio": res.filter(pl.col("kind") == "ratio").row(0, named=True),
           "best_exact": res.filter(pl.col("kind") == "exact").row(0, named=True)}
    (rd / f"decision{sfx}.json").write_text(json.dumps(out, indent=1, default=str))
    log.info("tune: best %s | by country %s", best, by_c)
    return out


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/prep.py
"""Load raw TSVs and write normalised Parquet tables (one per split x source)."""
from __future__ import annotations

import logging
from multiprocessing import Pool

import polars as pl

from . import config as C
from .normalize import normalize_record

log = logging.getLogger(__name__)

SCHEMA = {
    "name_norm": pl.Utf8, "core": pl.List(pl.Utf8), "legal": pl.List(pl.Utf8),
    "skel": pl.List(pl.Utf8), "addr_norm": pl.Utf8, "atoks": pl.List(pl.Utf8),
    "nums": pl.List(pl.Utf8), "hno": pl.Utf8, "state": pl.Utf8,
    "loc": pl.List(pl.Utf8), "street": pl.List(pl.Utf8),
}


def read_tsv(path) -> pl.DataFrame:
    return pl.read_csv(path, separator="\t", quote_char=None, infer_schema=False,
                       encoding="utf8-lossy")


def _norm_chunk(df: pl.DataFrame) -> pl.DataFrame:
    out = [normalize_record(n, a, c) for n, a, c in
           zip(df["business_name"], df["business_address"], df["country"])]
    cols = list(zip(*out)) if out else [[] for _ in SCHEMA]
    norm = pl.DataFrame({k: list(v) for k, v in zip(SCHEMA, cols)}, schema=SCHEMA)
    return pl.concat([df, norm], how="horizontal")


def normalize_df(df: pl.DataFrame, workers: int = C.WORKERS, chunk: int = 25000) -> pl.DataFrame:
    """Normalise in worker processes; each worker builds its own DataFrame chunk."""
    chunks = [df.slice(i, chunk) for i in range(0, df.height, chunk)]
    with Pool(workers) as pool:
        return pl.concat(pool.map(_norm_chunk, chunks))


def prep_split(split: str) -> None:
    for src in ("1", "2", "3"):
        out = C.WORK / f"{split}_s{src}.parquet"
        if out.exists():
            continue
        df = read_tsv(C.DATA / split / f"{split}_source{src}.tsv")
        df = df.with_columns(pl.col("country").fill_null(""),
                             pl.int_range(pl.len(), dtype=pl.UInt32).alias("idx"))
        normalize_df(df).write_parquet(out)
        log.info("prep %s s%s rows=%d", split, src, df.height)


def load_gt() -> pl.DataFrame:
    """Ground truth as long pairs (s1, tid)."""
    gt = read_tsv(C.DATA / "train" / "train_ground_truth.tsv")
    return (gt.with_columns(pl.col("matched_entity_ids").fill_null("").str.split(","))
              .explode("matched_entity_ids")
              .filter(pl.col("matched_entity_ids") != "")
              .select(pl.col("source1_entity_id").alias("s1"),
                      pl.col("matched_entity_ids").alias("tid")))


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/progress.py
"""Live progress report for a running pipeline:  python -m er.progress"""
from __future__ import annotations

import os
import time

import polars as pl

from . import config as C


def _s1_counts() -> dict[tuple[str, str], int]:
    out = {}
    for split in ("train", "test"):
        p = C.WORK / f"{split}_s1.parquet"
        roles = C.WORK / "train_roles.parquet"
        if split == "train" and C.TRAIN_ROLES_ONLY and roles.exists():
            for c, n in pl.read_parquet(roles).group_by("country").len().iter_rows():
                out[(split, c)] = n
        elif p.exists():
            for c, n in pl.scan_parquet(p).group_by("country").len().collect().iter_rows():
                out[(split, c)] = n
    return out


def report() -> None:
    now = time.time()
    counts = _s1_counts()
    print(f"{'partition':<22}{'status':<10}{'S1 done':>18}{'chunks':>8}{'min':>7}{'S1/min':>9}{'ETA min':>9}")
    total_left = 0.0
    rates = []
    for (split, country), n in sorted(counts.items(), key=lambda x: (x[0][0] != "train", x[0][1])):
        for src in ("2", "3"):
            d = C.WORK / f"{split}_A_{country}_{src}"
            name = f"{split}_{country}_S{src}"
            if (d / "_SUCCESS").exists():
                fs = list(d.glob("part_*.parquet"))
                ts = [os.path.getmtime(f) for f in fs]
                print(f"{name:<22}{'done':<10}{n:>18,}{len(fs):>8}")
                continue
            tmp = d / "tmp"
            fs = sorted(tmp.glob("part_*.parquet")) if tmp.exists() else []
            if not fs:
                print(f"{name:<22}{'queued':<10}{'0 / ' + format(n, ','):>18}")
                total_left += n
                continue
            ok = []
            for f in fs:  # skip a chunk that is still being written
                try:
                    ok.append(pl.read_parquet(f, columns=["s1_idx"]))
                except Exception:
                    pass
            done = pl.concat(ok)["s1_idx"].n_unique() if ok else 0
            start = os.path.getctime(tmp)
            mins = (now - start) / 60
            rate = done / max(mins, 1e-6)
            rates.append(rate)
            eta = (n - done) / rate if rate else float("nan")
            total_left += n - done
            print(f"{name:<22}{'running':<10}{f'{done:,} / {n:,}':>18}{len(fs):>8}{mins:>7.1f}{rate:>9,.0f}{eta:>9.1f}")
    if rates:
        r = sum(rates) / len(rates)
        print(f"\nStage A remaining: ~{total_left:,.0f} S1-partition units at ~{r:,.0f}/min "
              f"-> ~{total_left / r / 60:.1f} h (then stage1, stage B, train2, predict)")
    for st in ("P", "B"):
        fs = list(C.WORK.glob(f"*_{st}_*"))
        if fs:
            print(f"stage {st}: {len(fs)} partitions present")
    logs = sorted(C.WORK.glob("run_*.log"), key=os.path.getmtime)
    if logs:
        log = logs[-1]
        tail = [l for l in log.read_text(errors="ignore").splitlines() if " stage " in l or "exit=" in l]
        print("\nlast stage events:", *tail[-4:], sep="\n  ")
    mem = C.WORK / "mem.log"
    if mem.exists():
        print("memory:", mem.read_text().splitlines()[-1])


if __name__ == "__main__":
    report()


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/quick.py
"""Stage-1-only submission: decision layer on the stage-1 OOF matcher score p1.

python -m er.quick --run v0s1
Uses the stage-1 candidate files (work/{split}_P_*.parquet), which already hold p1 for every
candidate. The decision layer is tuned on untouched validation S1 (p1 = mean of both fold
models there), then applied to test. Lightweight: no stage-B features needed.
"""
from __future__ import annotations

import argparse
import json
import logging

import polars as pl

from . import config as C
from .decide import DecisionParams, select, select_exact
from .output import write_submission
from .pipeline import _parts, id_maps, labels, run_tune, train_roles

log = logging.getLogger(__name__)


def run_quick(run: str = "v0s1") -> dict:
    rd = C.WORK / "runs" / run
    rd.mkdir(parents=True, exist_ok=True)
    val_ids = train_roles().filter(pl.col("role") == "val").select(pl.col("idx").alias("s1_idx"))
    lab = labels().with_columns(pl.lit(1, pl.UInt8).alias("y"))
    val = (pl.concat([pl.read_parquet(f, columns=["s1_idx", "src", "t_idx", "p1"]) for f in _parts("train", "P")])
             .join(val_ids, on="s1_idx", how="semi").rename({"p1": "p"})
             .join(lab, on=["s1_idx", "src", "t_idx"], how="left").with_columns(pl.col("y").fill_null(0)))
    val.write_parquet(rd / "val_scores.parquet")
    out = run_tune(run)                      # exact/ratio expected-F0.5 grid on validation
    best = out["best"]
    test = pl.concat([pl.read_parquet(f, columns=["s1_idx", "src", "t_idx", "p1"]) for f in _parts("test", "P")]
                     ).rename({"p1": "p"})
    fn = select_exact if best["kind"] == "exact" else select
    sel = fn(test, DecisionParams(margin=best["margin"], m0=best["m0"], empty_bias=best["empty_bias"]))
    s1_ids, tg_ids = id_maps("test")
    to_ids = lambda x: (x.join(s1_ids.select("s1_idx", "s1"), on="s1_idx")
                         .join(tg_ids, on=["t_idx", "src"]).select("s1", "tid"))
    write_submission(to_ids(sel), to_ids(test.select("s1_idx", "src", "t_idx")), s1_ids["s1"], C.OUT / run,
                     validator=C.ROOT / "student_resource" / "utils" / "validate_submission.py",
                     test_dir=C.DATA / "test")
    info = {"best": best, "by_country": out["by_country"], "test_pairs": test.height,
            "matches": sel.height, "s1_with_match": sel["s1_idx"].n_unique(), "test_s1": s1_ids.height}
    (rd / "quick.json").write_text(json.dumps(info, indent=1, default=str))
    log.info("quick submission: %s", info)
    return info


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", default="v0s1")
    a = ap.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s")
    run_quick(a.run)


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/research.py
"""Unseen-country research data (India plays France): streamed export of stage-B features for Kaggle CPU kernels.

python -m er.research export --run v4fr --out <dir>      (ER_WORK=<work dir>)
Writes fit_US/part_*.parquet (US FIT S1 with fold), val_India/part_*.parquet (India VAL S1), each row
(s1_idx, src, t_idx, y, fold, <stage-2 features>), plus val_India_s1.parquet (all India val S1 idx, incl. S1
without candidates) and features.json. One partition file at a time, so peak memory stays small.
"""
from __future__ import annotations

import argparse
import json
import os

import polars as pl

from . import config as C
from .pipeline import _files, _parts, labels, train_roles

KEYS = ["s1_idx", "src", "t_idx"]


def export(run: str, out: str) -> dict:
    feats = json.loads((C.WORK / "runs" / run / "summary.json").read_text())["features"]
    roles = train_roles().select(pl.col("idx").cast(pl.UInt32).alias("s1_idx"), "role", "fold", "country")
    lab = labels().select(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32)).with_columns(pl.lit(1, pl.Int8).alias("y"))
    sets = {"fit_US": roles.filter((pl.col("country") == "US") & (pl.col("role") == "fit")),
            "val_India": roles.filter((pl.col("country") == "India") & (pl.col("role") == "val"))}
    cols = [c for c in feats if c not in KEYS]
    n = {k: 0 for k in sets}
    for name, r in sets.items():
        os.makedirs(f"{out}/{name}", exist_ok=True)
        ctry = "US" if name == "fit_US" else "India"
        i = 0
        for d_ in _parts("train", "B"):
            if f"_B_{ctry}_" not in d_.name:
                continue
            for f in _files(d_):
                d = (pl.read_parquet(f, columns=[c for c in KEYS + cols if c in pl.read_parquet_schema(f)])
                       .with_columns(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32))
                       .join(r.select("s1_idx", "fold"), on="s1_idx").join(lab, on=KEYS, how="left")
                       .with_columns(pl.col("y").fill_null(0), *[pl.col(c).cast(pl.Float32) for c in cols if c in pl.read_parquet_schema(f)]))
                if d.height:
                    d.write_parquet(f"{out}/{name}/part_{i:05d}.parquet")
                    i += 1
                    n[name] += d.height
    sets["val_India"].select("s1_idx").write_parquet(f"{out}/val_India_s1.parquet")
    lab.join(sets["val_India"].select("s1_idx"), on="s1_idx").select(KEYS).write_parquet(f"{out}/val_India_truth.parquet")
    json.dump({"features": feats}, open(f"{out}/features.json", "w"))
    return n


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("what", choices=["export"])
    ap.add_argument("--run", required=True)
    ap.add_argument("--out", required=True)
    a = ap.parse_args()
    print(export(a.run, a.out))


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/run.py
"""CLI entry point.

python -m er.run prep      [--split train|test|all]   normalise raw TSVs            -> work/{split}_s*.parquet
python -m er.run aux                                  fit noise/alias models      -> work/aux/
python -m er.run stage_a   [--split ...]              blocking + cheap/competition  -> work/{split}_A_*.parquet
python -m er.run stage1                               OOF pruning ranker + prune    -> work/{split}_P_*.parquet
python -m er.run stage_b   [--split ...]              rich pair features            -> work/{split}_B_*.parquet
python -m er.run train2    [--run v1]                 matcher + calibration + decision tuning (val)
python -m er.run predict   [--run v1]                 test inference -> output/*.tsv (validated)
python -m er.run all                                  everything, resumable (finished stages are skipped)
"""
from __future__ import annotations

import argparse
import logging
import time

STAGES = ["prep", "aux", "stage_a", "stage1", "stage_b", "train2", "tune", "predict"]
EXTRA = ["ensemble", "analysis", "loco"]


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("stage", choices=STAGES + EXTRA + ["all"])
    ap.add_argument("--split", default="all")
    ap.add_argument("--run", default="v1")
    ap.add_argument("--scores", default="", help="score set for tune/predict: '' (stage-2 LGB) or 'ens'")
    args = ap.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s")
    splits = ["train", "test"] if args.split == "all" else [args.split]
    todo = STAGES if args.stage == "all" else [args.stage]
    from . import pipeline as P
    for stage in todo:
        t = time.time()
        if stage == "prep":
            from .prep import prep_split
            for s in splits:
                prep_split(s)
        elif stage == "aux":
            from .auxfit import fit_aux
            fit_aux()
        elif stage == "stage_a":
            for s in splits:
                P.run_stage_a(s)
        elif stage == "stage1":
            P.train_stage1()
            for s in splits:
                P.run_stage1(s)
        elif stage == "stage_b":
            for s in splits:
                P.run_stage_b(s)
        elif stage == "train2":
            P.run_train2(args.run)
        elif stage == "tune":
            P.run_tune(args.run, args.scores)
        elif stage == "predict":
            P.run_predict(args.run, args.scores)
        elif stage == "ensemble":
            from .ensemble import run_ensemble
            run_ensemble(args.run)
        elif stage == "loco":
            from .transfer import run_loco
            run_loco(args.run)
        elif stage == "analysis":
            from .analysis import analyse
            analyse(args.run)
        logging.info("stage %s done in %.1fs", stage, time.time() - t)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/selftrain.py
"""Self-training (pseudo-label) domain adaptation, validated by simulation before use on France.

python -m er.selftrain simulate --run v3            (ER_WORK must point at the run's work dir)
Protocol (India plays the unseen country):
  1. base  : LightGBM trained on US fit pairs only -> India val macro F0.5 (zero-shot proxy)
  2. pseudo: score India FIT pairs with the base model as if unlabelled; confident positives
             (p >= hi and best candidate for its target) and confident negatives (p <= lo)
  3. adapt : retrain on US fit + India pseudo-labelled pairs -> India val macro F0.5
Accept the recipe only if adapt > base with a paired-bootstrap CI above zero.
"""
from __future__ import annotations

import argparse
import json
import logging

import lightgbm as lgb
import numpy as np
import polars as pl

from . import config as C
from .decide import DecisionParams, select
from .metric import macro_f05
from .pipeline import _scan_b, id_maps, labels, train_roles
from .prep import load_gt

log = logging.getLogger(__name__)
P = dict(objective="binary", learning_rate=0.1, num_leaves=127, min_data_in_leaf=100, feature_fraction=0.7,
         bagging_fraction=0.8, bagging_freq=1, lambda_l2=3.0, verbose=-1, seed=C.SEED,
         num_threads=int(__import__("os").environ.get("ER_WORKERS", 8)))


def _load(feats):
    roles = train_roles().select(pl.col("idx").alias("s1_idx"), "role", "country")
    lab = labels().with_columns(pl.lit(1, pl.UInt8).alias("y"))
    return (_scan_b("train").join(roles.lazy(), on="s1_idx")
            .join(lab.lazy(), on=["s1_idx", "src", "t_idx"], how="left").with_columns(pl.col("y").fill_null(0))
            .select("s1_idx", "t_idx", "y", "role", "country", pl.col("src").alias("src_key"),
                    *[pl.col(c).cast(pl.Float32) for c in feats])
            .collect(engine="streaming"))


def _per_s1_f(v: pl.DataFrame, prm: DecisionParams, ids: pl.DataFrame, tg_ids, gt) -> np.ndarray:
    sel = select(v, prm).join(ids, on="s1_idx").join(tg_ids, on=["t_idx", "src"]).select("s1", "tid").unique()
    truth = gt.join(ids.select("s1"), on="s1").unique()
    tp = truth.join(sel, on=["s1", "tid"]).group_by("s1").len("tp")
    df = (ids.select("s1").join(truth.group_by("s1").len("nt"), on="s1", how="left")
              .join(sel.group_by("s1").len("np"), on="s1", how="left").join(tp, on="s1", how="left").fill_null(0))
    p, r = pl.col("tp") / pl.col("np"), pl.col("tp") / pl.col("nt")
    return df.select(pl.when((pl.col("nt") == 0) & (pl.col("np") == 0)).then(1.0)
                     .when((pl.col("nt") == 0) | (pl.col("np") == 0) | (pl.col("tp") == 0)).then(0.0)
                     .otherwise(1.25 * p * r / (0.25 * p + r))).to_series().to_numpy()


def simulate(run: str, hi: float = 0.97, lo: float = 0.02, w_pseudo: float = 1.0, n_boot: int = 1000) -> dict:
    rd = C.WORK / "runs" / run
    feats = json.loads((rd / "summary.json").read_text())["features"]
    dec = json.loads((rd / "decision.json").read_text())["best"]
    prm = DecisionParams(margin=dec["margin"], m0=dec["m0"], empty_bias=dec["empty_bias"])
    d = _load(feats)
    us_fit = d.filter((pl.col("country") == "US") & (pl.col("role") == "fit"))
    in_fit = d.filter((pl.col("country") == "India") & (pl.col("role") == "fit"))
    in_val = d.filter((pl.col("country") == "India") & (pl.col("role") == "val"))
    del d
    X = lambda df: df.select(feats).to_numpy()
    base = lgb.train(P, lgb.Dataset(X(us_fit), us_fit["y"].to_numpy()), 600)
    # pseudo-labels on India fit (labels NOT used), target-exclusive confident positives
    pf = in_fit.select("s1_idx", "t_idx", pl.col("src_key").alias("src")).with_columns(
        pl.Series("p", base.predict(X(in_fit))))
    pf = pf.with_columns((pl.col("p") >= pl.col("p").max().over("src", "t_idx")).alias("best"))
    pos = (pf["p"] >= hi) & pf["best"]
    neg = pf["p"] <= lo
    keep = (pos | neg).to_numpy()
    y_pseudo = pos.to_numpy()[keep].astype(int)
    true_y = in_fit["y"].to_numpy()[keep]
    pl_acc = {"n_pseudo": int(keep.sum()), "pos": int(y_pseudo.sum()),
              "pseudo_label_accuracy": float((y_pseudo == true_y).mean())}
    Xa = np.vstack([X(us_fit), X(in_fit)[keep]])
    ya = np.concatenate([us_fit["y"].to_numpy(), y_pseudo])
    wa = np.concatenate([np.ones(us_fit.height), np.full(int(keep.sum()), w_pseudo)])
    adapt = lgb.train(P, lgb.Dataset(Xa, ya, weight=wa), 600)
    s1_ids, tg_ids = id_maps("train")
    ids = (train_roles().filter((pl.col("role") == "val") & (pl.col("country") == "India"))
           .select(pl.col("idx").alias("s1_idx")).join(s1_ids.select("s1_idx", "s1"), on="s1_idx"))
    gt = load_gt()
    res = {"pseudo": pl_acc}
    fs = {}
    for name, m in (("base_us_only", base), ("adapted", adapt)):
        v = in_val.select("s1_idx", pl.col("src_key").alias("src"), "t_idx").with_columns(
            pl.Series("p", m.predict(X(in_val)).astype(np.float32)))
        fs[name] = _per_s1_f(v, prm, ids, tg_ids, gt)
        res[name] = round(float(fs[name].mean()), 5)
    dlt = fs["adapted"] - fs["base_us_only"]
    rng = np.random.default_rng(0)
    boots = dlt[rng.integers(0, len(dlt), size=(n_boot, len(dlt)))].mean(1)
    res["delta"] = round(float(dlt.mean()), 5)
    res["ci95"] = [round(float(np.percentile(boots, 2.5)), 5), round(float(np.percentile(boots, 97.5)), 5)]
    res["ACCEPT"] = res["ci95"][0] > 0
    (rd / "selftrain_sim.json").write_text(json.dumps(res, indent=1))
    log.info("self-training simulation: %s", res)
    return res



def make_pseudo(run: str, country: str, hi: float = 0.97, lo: float = 0.02, out: str | None = None) -> dict:
    """Confident pseudo-labels for one test country from a run's calibrated test scores:
    positives p >= hi that are the best-scoring S1 for their target; negatives p <= lo."""
    from .pipeline import load
    rd = C.WORK / "runs" / run
    t = pl.read_parquet(rd / "test_scores.parquet")
    c = load("test", "1", cols=["idx", "country"]).rename({"idx": "s1_idx"}).filter(pl.col("country") == country)
    t = t.join(c.select("s1_idx"), on="s1_idx", how="semi").with_columns(
        (pl.col("p") >= pl.col("p").max().over("src", "t_idx")).alias("best"))
    ps = t.filter(((pl.col("p") >= hi) & pl.col("best")) | (pl.col("p") <= lo)).select(
        "s1_idx", "src", "t_idx", (pl.col("p") >= hi).cast(pl.Int8).alias("y_pseudo"))
    path = out or str(rd / f"pseudo_{country}.parquet")
    ps.write_parquet(path)
    info = {"country": country, "rows": ps.height, "pos": int(ps["y_pseudo"].sum()), "path": path}
    log.info("pseudo-labels: %s", info)
    return info

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("what", choices=["simulate", "pseudo"])
    ap.add_argument("--run", default="v3")
    ap.add_argument("--country", default="France")
    a = ap.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s")
    print(json.dumps(simulate(a.run) if a.what == "simulate" else make_pseudo(a.run, a.country), indent=1))


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/text.py
"""Scalar text utilities: script folding, Indic transliteration, phonetic skeletons.

These operate on single strings and are only applied to the (minority of) values that
need them; bulk normalisation is vectorised in ``er.prep`` with Polars.
"""
from __future__ import annotations

import re
import unicodedata

from anyascii import anyascii

# ---------------------------------------------------------------------------
# Indic transliteration. All Brahmic scripts in Unicode share the Devanagari
# layout at a fixed block offset, so we fold every script onto Devanagari
# offsets and transliterate with a single table.
# ---------------------------------------------------------------------------
_INDIC_BLOCKS = (0x0900, 0x0980, 0x0A00, 0x0A80, 0x0B00, 0x0B80, 0x0C00, 0x0C80, 0x0D00)

_CONS = {
    0x15: "k", 0x16: "kh", 0x17: "g", 0x18: "gh", 0x19: "n", 0x1A: "ch", 0x1B: "chh",
    0x1C: "j", 0x1D: "jh", 0x1E: "n", 0x1F: "t", 0x20: "th", 0x21: "d", 0x22: "dh",
    0x23: "n", 0x24: "t", 0x25: "th", 0x26: "d", 0x27: "dh", 0x28: "n", 0x29: "n",
    0x2A: "p", 0x2B: "ph", 0x2C: "b", 0x2D: "bh", 0x2E: "m", 0x2F: "y", 0x30: "r",
    0x31: "r", 0x32: "l", 0x33: "l", 0x34: "zh", 0x35: "v", 0x36: "sh", 0x37: "sh",
    0x38: "s", 0x39: "h",
    0x58: "q", 0x59: "kh", 0x5A: "g", 0x5B: "z", 0x5C: "d", 0x5D: "dh", 0x5E: "f", 0x5F: "y",
}
_VOWELS = {
    0x05: "a", 0x06: "a", 0x07: "i", 0x08: "i", 0x09: "u", 0x0A: "u", 0x0B: "ri",
    0x0C: "l", 0x0D: "e", 0x0E: "e", 0x0F: "e", 0x10: "ai", 0x11: "o", 0x12: "o",
    0x13: "o", 0x14: "au", 0x60: "ri", 0x61: "l",
}
_MATRAS = {
    0x3E: "a", 0x3F: "i", 0x40: "i", 0x41: "u", 0x42: "u", 0x43: "ri", 0x44: "ri",
    0x45: "e", 0x46: "e", 0x47: "e", 0x48: "ai", 0x49: "o", 0x4A: "o", 0x4B: "o",
    0x4C: "au", 0x62: "l", 0x63: "l",
}
_VIRAMA, _NUKTA = 0x4D, 0x3C
_NASAL = {0x01: "n", 0x02: "n", 0x03: "h"}


def _indic_offset(ch: str) -> int | None:
    cp = ord(ch)
    for base in _INDIC_BLOCKS:
        if base <= cp < base + 0x80:
            return cp - base
    return None


def transliterate_indic(text: str) -> str:
    """Rule-based transliteration of any Brahmic script to lowercase Latin.

    Implements inherent-vowel insertion with word-final schwa deletion, which is
    what romanised Indian business names typically look like (राम -> ram).
    """
    out: list[str] = []
    pending_schwa = False  # a consonant was emitted and may take an inherent 'a'
    conjunct = False  # current consonant closes a conjunct cluster (keeps final 'a')
    after_virama = False
    for ch in text:
        off = _indic_offset(ch)
        if off is None:
            if ch in "‌‍":
                continue
            if pending_schwa and not ch.isalpha():
                if conjunct:
                    out.append("a")  # aditya, not adity
                pending_schwa = False  # word end: drop schwa
            out.append(ch)
            continue
        if off in _CONS:
            if pending_schwa:
                out.append("a")
            out.append(_CONS[off])
            pending_schwa = True
            conjunct = after_virama
            after_virama = False
        elif off in _MATRAS:
            out.append(_MATRAS[off])
            pending_schwa = False
        elif off == _VIRAMA:
            pending_schwa = False
            after_virama = True
        elif off == _NUKTA:
            continue
        elif off in _VOWELS:
            if pending_schwa:
                out.append("a")
            out.append(_VOWELS[off])
            pending_schwa = False
        elif off in _NASAL:
            if pending_schwa:
                out.append("a")
            out.append(_NASAL[off])
            pending_schwa = False
        elif 0x66 <= off <= 0x6F:
            if pending_schwa:
                out.append("a")
            out.append(str(off - 0x66))
            pending_schwa = False
        else:  # danda, avagraha, misc signs
            if pending_schwa:
                pending_schwa = False
            out.append(" ")
    if pending_schwa and conjunct:
        out.append("a")
    return "".join(out)


_INDIC_RE = re.compile("[ऀ-ൿ]")


def to_ascii(text: str) -> str:
    """Fold any string to lowercase ASCII (accents stripped, Indic transliterated)."""
    if text.isascii():
        return text.lower()
    if _INDIC_RE.search(text):
        text = transliterate_indic(text)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    if not text.isascii():
        text = anyascii(text)
    return text.lower()


# ---------------------------------------------------------------------------
# Phonetic consonant skeleton: robust to vowel noise, transliteration and
# aspiration differences (praivet ~ private -> "prvt").
# ---------------------------------------------------------------------------
_SKEL_SUBS = (
    ("ph", "f"), ("ck", "k"), ("x", "ks"), ("q", "k"), ("w", "v"), ("z", "j"), ("b", "v"), ("g", "j"),
)
_C_SOFT = re.compile(r"c(?=[eiy])")
_G_SOFT = re.compile(r"g(?=[ei])")
_VOWEL_H = re.compile(r"[aeiouyh]")
_REPEAT = re.compile(r"(.)\1+")


def skeleton(token: str) -> str:
    t = token
    for a, b in _SKEL_SUBS:
        t = t.replace(a, b)
    t = _C_SOFT.sub("s", t)
    t = t.replace("c", "k")
    head, tail = t[:1], _VOWEL_H.sub("", t[1:])
    if head in "aeiouyh":
        head = ""
    t = _REPEAT.sub(r"\1", head + tail)
    return t


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/transfer.py
"""Leave-one-country-out (LOCO) transfer test: proxy for the unseen test country (France).

python -m er.run loco --run v1
Trains the stage-2 matcher on fit S1 of one country and evaluates macro F0.5 on val S1 of the
other country (and in-domain for reference), for the full feature set and a shift-robust subset.
The decision layer uses the run's tuned parameters, so only the model's transfer is measured.
"""
from __future__ import annotations

import json
import logging

import numpy as np
import polars as pl

from . import config as C
from .decide import DecisionParams, select, select_exact
from .metric import macro_f05
from .pipeline import S2_PARAMS, _scan_b, id_maps, labels, train_roles
from .prep import load_gt

log = logging.getLogger(__name__)

# signals whose meaning depends on country-specific lexicons (absent/different for France)
COUNTRY_SPECIFIC = ("state_eq",)


def _data(feats, role):
    roles = train_roles().select(pl.col("idx").alias("s1_idx"), "role", "fold", "country")
    lab = labels().with_columns(pl.lit(1, pl.UInt8).alias("y"))
    return (_scan_b("train").join(roles.lazy().filter(pl.col("role") == role), on="s1_idx")
            .join(lab.lazy(), on=["s1_idx", "src", "t_idx"], how="left").with_columns(pl.col("y").fill_null(0))
            .select("s1_idx", "t_idx", "y", "country", pl.col("src").alias("src_key"),
                    *[pl.col(c).cast(pl.Float32) for c in feats])
            .collect(engine="streaming"))


def run_loco(run_name: str = "v1", max_fit: int = 3_000_000) -> dict:
    import lightgbm as lgb

    rd = C.WORK / "runs" / run_name
    feats = json.loads((rd / "summary.json").read_text())["features"]
    dec = json.loads((rd / "decision.json").read_text())["best"] if (rd / "decision.json").exists() else \
        json.loads((rd / "summary.json").read_text())["best"]
    fn = select_exact if dec.get("kind") == "exact" else select
    prm = DecisionParams(margin=dec["margin"], m0=dec["m0"], empty_bias=dec["empty_bias"])
    fit, val = _data(feats, "fit"), _data(feats, "val")
    s1_ids, tg_ids = id_maps("train")
    roles = train_roles().filter(pl.col("role") == "val").select(pl.col("idx").alias("s1_idx"), "country")
    gt = load_gt()
    res = {}
    for name, fs in (("all", feats), ("robust", [f for f in feats if f not in COUNTRY_SPECIFIC])):
        for tr_c in ("US", "India"):
            tr = fit.filter(pl.col("country") == tr_c)
            if tr.height > max_fit:
                tr = tr.sample(max_fit, seed=C.SEED)
            m = lgb.train({**S2_PARAMS, "num_threads": C.WORKERS}, lgb.Dataset(tr.select(fs).to_numpy(), tr["y"].to_numpy()),
                          800)
            for ev_c in ("US", "India"):
                v = val.filter(pl.col("country") == ev_c)
                v = v.select("s1_idx", pl.col("src_key").alias("src"), "t_idx").with_columns(
                    pl.Series("p", m.predict(v.select(fs).to_numpy(), num_threads=C.WORKERS).astype(np.float32)))
                ids = roles.filter(pl.col("country") == ev_c).join(s1_ids.select("s1_idx", "s1"), on="s1_idx")
                pred = fn(v, prm).join(ids, on="s1_idx").join(tg_ids, on=["t_idx", "src"]).select("s1", "tid")
                r = macro_f05(pred, gt.join(ids.select("s1"), on="s1"), ids["s1"])
                res[f"{name}:{tr_c}->{ev_c}"] = round(r["f05"], 5)
                log.info("loco %s %s->%s f05=%.5f", name, tr_c, ev_c, r["f05"])
    (rd / "loco.json").write_text(json.dumps(res, indent=1))
    return res


In [ ]:
%%writefile er_pkg/code/business_entity_resolution/src/er/translit.py
"""Learned back-transliteration dictionary for Indic-script business names.

Indian S2/S3 records often write an English business name phonetically in a Brahmic script
("प्रीमियर बिजनेस" = Premier Business). er.text romanises it ("priimiyar bijnes"), but the
romanisation is not the English spelling, so name keys and similarities miss the pair
(v5: 9.4% blocking miss on Indic-script targets vs 2.5% for Latin script).

The dictionary is learned from the provided training pairs only. For a true pair whose target
name is Indic-script and whose normalised core has the same number of tokens as the S1 core,
tokens are aligned by position. A romanised token maps to its most frequent English partner
when seen >= 2 times with >= 70% agreement. Only FIT S1 are used, so the validation set stays
clean. The dictionary is applied only to names that contain Indic script (er.normalize).

python -m er.translit learn          (ER_WORK=<work dir>) -> <work>/aux/translit.json
"""
from __future__ import annotations

import json
import logging
from collections import Counter, defaultdict

import polars as pl

from . import config as C

log = logging.getLogger(__name__)
INDIC = r"[ऀ-෿]"          # Devanagari .. Sinhala blocks (all Brahmic scripts used in India)


def learn(min_count: int = 2, min_share: float = 0.7) -> dict[str, str]:
    from .pipeline import labels, train_roles
    fit = train_roles().filter(pl.col("role") == "fit").select(pl.col("idx").cast(pl.UInt32).alias("s1_idx"))
    lab = labels().select(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8),
                          pl.col("t_idx").cast(pl.UInt32)).join(fit, on="s1_idx")
    s1 = pl.read_parquet(C.WORK / "train_s1.parquet", columns=["idx", "core"]).select(
        pl.col("idx").cast(pl.UInt32).alias("s1_idx"), pl.col("core").alias("c1"))
    tg = pl.concat([pl.read_parquet(C.WORK / f"train_s{s}.parquet", columns=["idx", "business_name", "core"]).select(
        pl.lit(s).cast(pl.UInt8).alias("src"), pl.col("idx").cast(pl.UInt32).alias("t_idx"), "business_name",
        pl.col("core").alias("c2")) for s in (2, 3)])
    d = (lab.join(tg, on=["src", "t_idx"]).filter(pl.col("business_name").fill_null("").str.contains(INDIC))
            .join(s1, on="s1_idx").filter(pl.col("c1").list.len() == pl.col("c2").list.len()))
    cnt: dict[str, Counter] = defaultdict(Counter)
    for a, b in zip(d["c2"].to_list(), d["c1"].to_list()):
        for x, y in zip(a, b):
            if x != y:
                cnt[x][y] += 1
    dic = {}
    for x, c in cnt.items():
        y, n = c.most_common(1)[0]
        tot = sum(c.values())
        if tot >= min_count and n / tot >= min_share:
            dic[x] = y
    log.info("translit dictionary: %d entries from %d aligned Indic pairs", len(dic), d.height)
    return dic


if __name__ == "__main__":
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("what", choices=["learn"])
    ap.add_argument("--out", default=None)
    a = ap.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s")
    out = a.out or str(C.WORK / "aux" / "translit.json")
    dic = learn()
    with open(out, "w", encoding="utf-8") as fh:
        json.dump(dic, fh, indent=0, sort_keys=True)
    print(json.dumps({"entries": len(dic), "out": out}))


In [ ]:
# ============================== SETUP ==============================
import gc
import io
import json
import pickle
import shutil
import subprocess
import sys
import time
import zipfile

import numpy as np
import polars as pl

PKG = Path(os.environ.get("NB_PKG", "er_pkg/code/business_entity_resolution/src")).resolve()
WORK = (NB / "work_v6") if BUILD_BASE_FROM_SCRATCH else V6_WORK
NCPU = os.cpu_count() or 4
os.environ.update(ER_WORK=str(WORK), ER_DATA=str(DATA_DIR), ER_OUT=str(NB / "er_out"),
                  ER_WORKERS=str(max(1, NCPU - 1)), POLARS_MAX_THREADS=str(NCPU), PYTHONUTF8="1",
                  TOKENIZERS_PARALLELISM="true")
if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))
for d in ("tx", "dense", "ce", "feat", "rr", "eval", "sub"):
    (NB / d).mkdir(parents=True, exist_ok=True)

from er import config as C          # noqa: E402
from er import pipeline as P        # noqa: E402
from er.cross import _fit1_calibrated, export_train   # noqa: E402
from er.decide import DecisionParams, coherent        # noqa: E402
from er.output import check_invariants, id_lists      # noqa: E402

KEYS = ["s1_idx", "src", "t_idx"]
T0 = time.time()


def log(*a):
    print(f"[{(time.time() - T0) / 60:7.1f} min]", *a, flush=True)


def cast(d):
    d = d.with_columns(pl.col("s1_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8), pl.col("t_idx").cast(pl.UInt32))
    return d.with_columns(pl.col("p").cast(pl.Float32)) if "p" in d.columns else d


def logit(c):
    x = pl.col(c).clip(1e-6, 1 - 1e-6)
    return (x / (1 - x)).log()


def done(*paths):
    return all(Path(p).exists() for p in paths)


def save(df, path):
    tmp = Path(str(path) + ".part")
    df.write_parquet(tmp)
    tmp.replace(path)


log("work dir:", WORK, "| output dir:", NB, "| python", sys.version.split()[0], "| polars", pl.__version__)

In [ ]:
# ============================== INPUT CHECKS ==============================
need = [DATA_DIR / "train" / "train_ground_truth.tsv", DATA_DIR / "test" / "test_source1.tsv"]
if not BUILD_BASE_FROM_SCRATCH:
    need += [WORK / f"{s}_s{k}.parquet" for s in ("train", "test") for k in (1, 2, 3)]
    need += [WORK / "train_roles.parquet", WORK / "runs/v6/fit_oof.parquet", WORK / "runs/v6/calibrators.pkl",
             WORK / "runs/v6/val_scores.parquet", WORK / "runs/v6/test_scores.parquet",
             WORK / "runs/v6ce/val_scores_ce.parquet", WORK / "runs/v6ce/test_scores_ce.parquet"]
missing = [str(p) for p in need if not p.exists()]
assert not missing, ("missing inputs:\n  " + "\n  ".join(missing) + "\n\nOn Kaggle: 'Add Input' the competition data and a "
                     "dataset with the final2 artifacts (s-tables with name_norm/addr_norm/hno, train_roles, runs v6 / v6ce), "
                     "then re-run the KAGGLE cell. Elsewhere: fix DATA_DIR / V6_WORK in the CONFIG cell.")
HAVE_V6CD = done(WORK / "runs/v6cd/val_scores_ce.parquet", WORK / "runs/v6cd/test_scores_ce.parquet")
log("restored v6cd available for the gate:", HAVE_V6CD, "| France 0.988 file:", FR_0988.exists(),
    "| validator:", VALIDATOR.exists())
if not FR_0988.exists():
    print("WARNING: FR_0988 not found -> France rows will come from this notebook's base v6ce scores (NOT the LB-0.988 rule).")

In [ ]:
# ============================== (OPTIONAL) BUILD THE v6 BASE FROM SCRATCH ==============================
# Only when the final2 artifacts are not available. This runs Zayaan's full stage-A/1/B/2 pipeline with the code's
# default switches; the exact v6 environment is not recorded here, so scores can differ slightly from the committed v6.
if BUILD_BASE_FROM_SCRATCH and not done(WORK / "runs/v6/test_scores.parquet"):
    from er.auxfit import fit_aux
    from er.prep import prep_split
    for s in ("train", "test"):
        prep_split(s)
    fit_aux()
    for s in ("train", "test"):
        P.run_stage_a(s)
    P.train_stage1()
    for s in ("train", "test"):
        P.run_stage1(s)
    for s in ("train", "test"):
        P.run_stage_b(s)
    P.run_train2("v6")
    import lightgbm as lgb
    rd = WORK / "runs/v6"
    feats = json.loads((rd / "summary.json").read_text())["features"]
    models = [lgb.Booster(model_file=str(rd / f"stage2_f{f}.lgb")) for f in (0, 1)]
    cal = pickle.load(open(rd / "calibrators.pkl", "rb"))
    save(P._predict_b("test", models, feats, cal), rd / "test_scores.parquet")
    for n in ("val", "test"):          # no v6ce CE stack in this mode: the baseline uses stage-2 p
        (WORK / "runs/v6ce").mkdir(parents=True, exist_ok=True)
        shutil.copyfile(rd / f"{n}_scores.parquet", WORK / f"runs/v6ce/{n}_scores_ce.parquet")
    log("v6 base built from scratch")

## 2. Parts, labels, text tables, base (stage-2) scores
Parts: `fit0` (FIT fold 0: trains bi-encoder + CE-u), `fit1` (FIT fold 1: trains the re-ranker), `val` (decision +
gate), `test`. `s1_idx` / `t_idx` are row indices within a split, so train and test are always kept apart.

In [ ]:
roles = (P.train_roles().select(pl.col("idx").cast(pl.UInt32).alias("s1_idx"), "country", "role", "fold")
         .filter(pl.col("country").is_in(COUNTRIES)))
PARTS = {
    "fit0": roles.filter((pl.col("role") == "fit") & (pl.col("fold") == 0)).select("s1_idx", "country"),
    "fit1": roles.filter((pl.col("role") == "fit") & (pl.col("fold") == 1)).select("s1_idx", "country"),
    "val": roles.filter(pl.col("role") == "val").select("s1_idx", "country"),
}
TEST_S1_ALL = pl.read_parquet(WORK / "test_s1.parquet", columns=["idx", "entity_id", "country"]).with_columns(
    pl.col("idx").cast(pl.UInt32).alias("s1_idx"))
PARTS["test"] = TEST_S1_ALL.filter(pl.col("country").is_in(COUNTRIES)).select("s1_idx", "country")
SPLIT = {"fit0": "train", "fit1": "train", "val": "train", "test": "test"}
LAB = cast(P.labels("train")).unique().with_columns(pl.lit(1, pl.Int8).alias("y"))
log({k: v.height for k, v in PARTS.items()}, "| true train links:", LAB.height)


def text_table(split, src):
    """idx, country, text ('name | address', the CE / bi-encoder input), n/a/h (normalised name, address, house no.),
    st (address without digits: the street part)."""
    p = NB / "tx" / f"{split}_s{src}.parquet"
    if not p.exists():
        d = (pl.scan_parquet(WORK / f"{split}_s{src}.parquet")
             .select(pl.col("idx").cast(pl.UInt32), "country", "business_name", "business_address", "name_norm", "addr_norm", "hno")
             .filter(pl.col("country").is_in(COUNTRIES)).collect())
        d = d.select("idx", "country",
                     (pl.col("business_name").fill_null("") + " | " + pl.col("business_address").fill_null("")).alias("text"),
                     pl.col("name_norm").fill_null("").alias("n"), pl.col("addr_norm").fill_null("").alias("a"),
                     pl.col("hno").fill_null("").alias("h"))
        d = d.with_columns(pl.col("a").str.replace_all(r"\d+", " ").str.replace_all(r"\s+", " ").str.strip_chars().alias("st"))
        save(d, p)
    return pl.read_parquet(p)


for split in ("train", "test"):
    for src in (1, 2, 3):
        text_table(split, src)

# base candidates of the v6 run with stage-2 calibrated p (fit1: out-of-fold) -> base_{fit1,val,test}.parquet
if not done(*(NB / f"base_{p}.parquet" for p in ("fit1", "val", "test"))):
    rd = WORK / "runs/v6"
    save(cast(_fit1_calibrated(rd)).select(*KEYS, "p").join(PARTS["fit1"].select("s1_idx"), on="s1_idx", how="semi"),
         NB / "base_fit1.parquet")
    for part in ("val", "test"):
        d = cast(pl.read_parquet(rd / f"{part}_scores.parquet").select(*KEYS, "p"))
        save(d.join(PARTS[part].select("s1_idx"), on="s1_idx", how="semi"), NB / f"base_{part}.parquet")
    keys_tr = pl.concat([cast(pl.read_parquet(rd / "fit_oof.parquet").select(KEYS)),
                         cast(pl.read_parquet(rd / "val_scores.parquet").select(KEYS))]).unique()
    save(keys_tr, NB / "basekeys_train.parquet")
    save(cast(pl.read_parquet(rd / "test_scores.parquet").select(KEYS)).unique(), NB / "basekeys_test.parquet")
BASE_P = {p: pl.read_parquet(NB / f"base_{p}.parquet") for p in ("fit1", "val", "test")}
log({p: d.height for p, d in BASE_P.items()}, "base pairs;", {p: int((d["p"] >= P_MIN).sum()) for p, d in BASE_P.items()}, "with p >= P_MIN")

## 3. Bi-encoder (multilingual-e5-small, in-batch negatives) — FIT fold-0 positives
Symmetric InfoNCE (scale 20, cosine), in-batch pairs of the **same S1** are masked (they are also true, not negatives).

In [ ]:
import torch                                   # noqa: E402
import torch.nn.functional as F                # noqa: E402
from transformers import (AutoModel, AutoModelForSequenceClassification, AutoTokenizer,   # noqa: E402
                          get_linear_schedule_with_warmup)

DEV = "cuda" if torch.cuda.is_available() else "cpu"
EMB_DT = torch.float16 if DEV == "cuda" else torch.float32
AMP = DEV == "cuda"
torch.manual_seed(SEED)
np.random.seed(SEED)
log("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")


def mean_pool(h, mask):
    m = mask.unsqueeze(-1).to(h.dtype)
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)


def attach_text(d, split):
    """d with s1_idx, src, t_idx -> + a (S1 text), b (target text)."""
    a = text_table(split, 1).select(pl.col("idx").alias("s1_idx"), pl.col("text").alias("a"))
    parts = []
    for s in (2, 3):
        t = text_table(split, s).select(pl.col("idx").alias("t_idx"), pl.col("text").alias("b"))
        parts.append(d.filter(pl.col("src") == s).join(t, on="t_idx"))
    return pl.concat(parts).join(a, on="s1_idx")


BI_DIR = Path(BI_MODEL_DIR) if BI_MODEL_DIR else NB / "bi_model"
if not (BI_DIR / "config.json").exists():
    fit_s1 = roles.filter((pl.col("role") == "fit") & pl.col("fold").is_in(list(BI_TRAIN_FOLDS))).select("s1_idx")
    pos = LAB.join(fit_s1, on="s1_idx", how="semi").select(KEYS)
    if pos.height > BI_MAX_PAIRS:
        pos = pos.sample(BI_MAX_PAIRS, seed=SEED)
    pos = attach_text(pos, "train").sample(fraction=1.0, shuffle=True, seed=SEED)
    A = ["query: " + x for x in pos["a"].to_list()]
    B = ["query: " + x for x in pos["b"].to_list()]
    S1 = torch.tensor(pos["s1_idx"].to_numpy().astype(np.int64))
    tok = AutoTokenizer.from_pretrained(BI_BASE)
    model = AutoModel.from_pretrained(BI_BASE).to(DEV)
    steps = BI_EPOCHS * (len(A) // BI_BS)
    opt = torch.optim.AdamW(model.parameters(), lr=BI_LR, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    log("bi-encoder training pairs:", len(A), "steps:", steps)

    def emb(texts):
        e = tok(texts, padding=True, truncation=True, max_length=BI_MAXLEN, return_tensors="pt").to(DEV)
        return F.normalize(mean_pool(model(**e).last_hidden_state, e["attention_mask"]).float(), dim=-1)

    model.train()
    rng = np.random.default_rng(SEED)
    step = 0
    for ep in range(BI_EPOCHS):
        perm = rng.permutation(len(A))
        for i in range(len(A) // BI_BS):
            ix = perm[i * BI_BS:(i + 1) * BI_BS]
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
                qa, qb = emb([A[j] for j in ix]), emb([B[j] for j in ix])
            sim = (qa @ qb.T) * BI_SCALE
            s1b = S1[ix].to(DEV)
            same = (s1b[:, None] == s1b[None, :]) & ~torch.eye(len(ix), dtype=torch.bool, device=DEV)
            sim = sim.masked_fill(same, -1e4)
            lab_ = torch.arange(len(ix), device=DEV)
            loss = (F.cross_entropy(sim, lab_) + F.cross_entropy(sim.T, lab_)) / 2
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sch.step()
            step += 1
            if step % 1000 == 0 or step == 1:
                log(f"bi ep {ep} step {step}/{steps} loss {loss.item():.4f}")
    BI_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(BI_DIR)
    tok.save_pretrained(BI_DIR)
    del model, opt, pos, A, B
    gc.collect()
    if DEV == "cuda":
        torch.cuda.empty_cache()
    log("bi-encoder saved to", BI_DIR)

## 4. Dense retrieval (all FIT + VAL S1, and test) + bi-encoder cosine for every base candidate
Per country and source: S1 → top-K targets by cosine. New pairs (not in the v6 candidate set) are kept above a cosine
cut that retains `DENSE_KEEP` of the new **val** true pairs (the v6cd rule). Base candidates get their cosine too
(a re-ranker feature v6cd never had).

In [ ]:
@torch.no_grad()
def encode(model, tok, texts, bs=1024):
    order = np.argsort([len(t) for t in texts], kind="stable")
    dim = model.config.hidden_size
    out = torch.empty((len(texts), dim), dtype=EMB_DT, device=DEV)
    for i in range(0, len(texts), bs):
        ix = order[i:i + bs]
        e = tok(["query: " + texts[j] for j in ix], padding=True, truncation=True, max_length=BI_MAXLEN,
                return_tensors="pt").to(DEV)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            h = mean_pool(model(**e).last_hidden_state, e["attention_mask"])
        out[torch.as_tensor(ix, device=DEV)] = F.normalize(h.float(), dim=-1).to(EMB_DT)
    return out


def pair_cos(Ea, Et, a_idx, t_idx, qa, qt, bs=500_000):
    """cosine of (qa[i], qt[i]) pairs; a_idx / t_idx are the sorted record idx arrays behind Ea / Et."""
    ia, it = np.searchsorted(a_idx, qa), np.searchsorted(t_idx, qt)
    out = np.empty(len(qa), dtype=np.float32)
    for i in range(0, len(qa), bs):
        x = torch.as_tensor(ia[i:i + bs], device=DEV)
        y = torch.as_tensor(it[i:i + bs], device=DEV)
        out[i:i + bs] = (Ea[x].float() * Et[y].float()).sum(1).cpu().numpy()
    return out


DENSE_OUT = {p: NB / "dense" / f"dense_{p}.parquet" for p in ("fit0", "fit1", "val", "test")}
if not done(*DENSE_OUT.values(), NB / "dense" / "basecos_train.parquet", NB / "dense" / "basecos_test.parquet"):
    btok = AutoTokenizer.from_pretrained(BI_DIR)
    bmodel = AutoModel.from_pretrained(BI_DIR).to(DEV).eval()
    if DEV == "cuda":
        bmodel = bmodel.half()
    new_all, cos_all = {"train": [], "test": []}, {"train": [], "test": []}
    for split in ("train", "test"):
        s1_parts = ["fit0", "fit1", "val"] if split == "train" else ["test"]
        s1_set = pl.concat([PARTS[p] for p in s1_parts])
        keys = pl.read_parquet(NB / f"basekeys_{split}.parquet")
        bq = pl.concat([BASE_P[p].filter(pl.col("p") >= P_MIN).select(KEYS) for p in s1_parts if p in BASE_P])
        a_all = text_table(split, 1).join(s1_set.select("s1_idx"), left_on="idx", right_on="s1_idx", how="semi")
        for ctry in COUNTRIES:
            a = a_all.filter(pl.col("country") == ctry).sort("idx")
            if a.height == 0:
                continue
            Ea = encode(bmodel, btok, a["text"].to_list())
            for src in (2, 3):
                t = text_table(split, src).filter(pl.col("country") == ctry).sort("idx")
                if t.height == 0:
                    continue
                Et = encode(bmodel, btok, t["text"].to_list())
                k = min(DENSE_K, t.height)
                I, S = [], []
                for i in range(0, Ea.shape[0], 256):
                    v, ix = torch.topk(Ea[i:i + 256] @ Et.T, k, dim=1)
                    I.append(ix.cpu().numpy())
                    S.append(v.float().cpu().numpy())
                I, S = np.concatenate(I), np.concatenate(S)
                d = pl.DataFrame({"s1_idx": np.repeat(a["idx"].to_numpy(), k), "t_idx": t["idx"].to_numpy()[I.ravel()],
                                  "cos": S.ravel(), "rank": np.tile(np.arange(1, k + 1), a.height).astype(np.int16),
                                  "gap1": (S[:, :1] - S).ravel()}).with_columns(pl.lit(src, pl.UInt8).alias("src"))
                d = cast(d).join(keys, on=KEYS, how="anti")
                new_all[split].append(d)
                q = bq.filter(pl.col("src") == src).join(a.select(pl.col("idx").alias("s1_idx")), on="s1_idx", how="semi")
                q = q.join(t.select(pl.col("idx").alias("t_idx")), on="t_idx", how="semi")
                if q.height:
                    cos_all[split].append(q.with_columns(pl.Series("cos", pair_cos(
                        Ea, Et, a["idx"].to_numpy(), t["idx"].to_numpy(), q["s1_idx"].to_numpy(), q["t_idx"].to_numpy()))))
                log(split, ctry, f"S{src}", "S1", a.height, "targets", t.height, "new pairs", d.height, "base cos", q.height)
                del Et
                if DEV == "cuda":
                    torch.cuda.empty_cache()
            del Ea
    for split in ("train", "test"):
        save(pl.concat(cos_all[split]) if cos_all[split] else pl.DataFrame(schema={**{k: v for k, v in bq.schema.items()}, "cos": pl.Float32}),
             NB / "dense" / f"basecos_{split}.parquet")
    part_map = pl.concat([PARTS[p].select("s1_idx").with_columns(pl.lit(p).alias("part")) for p in ("fit0", "fit1", "val")])
    tr = (pl.concat(new_all["train"]).join(part_map, on="s1_idx").join(LAB, on=KEYS, how="left")
          .with_columns(pl.col("y").fill_null(0)))
    te = pl.concat(new_all["test"]).with_columns(pl.lit("test").alias("part"), pl.lit(0, pl.Int8).alias("y"))
    v = tr.filter(pl.col("part") == "val")
    tau = float(v.filter(pl.col("y") == 1)["cos"].quantile(1 - DENSE_KEEP)) if int(v["y"].sum()) else 1.0
    stats = {"tau": tau, "val_new_pairs": v.height, "val_new_true": int(v["y"].sum()),
             "val_new_true_kept": int(v.filter((pl.col("y") == 1) & (pl.col("cos") >= tau)).height)}
    for p in ("fit0", "fit1", "val", "test"):
        d = (tr if p != "test" else te).filter((pl.col("part") == p) & (pl.col("cos") >= tau)).drop("part")
        save(d, DENSE_OUT[p])
        stats[f"kept_{p}"] = d.height
        stats[f"kept_true_{p}"] = int(d["y"].sum())
    json.dump(stats, open(NB / "dense" / "dense_stats.json", "w"), indent=1)
    del bmodel, new_all, cos_all, tr, te
    gc.collect()
    if DEV == "cuda":
        torch.cuda.empty_cache()
log("dense:", json.load(open(NB / "dense" / "dense_stats.json")))

## 5. CE-u: cross-encoder trained on fold-0 base **and dense** pairs
The v6cd cross-encoder only ever saw base (key-blocked) candidates. Here the same recipe (xlm-roberta-base, BCE,
2 epochs, lr 2e-5, AMP) also sees fold-0 dense pairs: every positive ×3 and a hash-sample of the negatives.

In [ ]:
CE_DIR = Path(CE_MODEL_DIR) if CE_MODEL_DIR else NB / "ce_model"
if not (CE_DIR / "config.json").exists():
    if not done(NB / "ce" / "ce_train.parquet"):
        export_train("v6", str(NB / "ce"))
    base_tr = pl.read_parquet(NB / "ce" / "ce_train.parquet").select("a", "b", pl.col("label").cast(pl.Float32))
    dz = pl.read_parquet(DENSE_OUT["fit0"])
    dpos = dz.filter(pl.col("y") == 1)
    dneg = dz.filter((pl.col("y") == 0) & (pl.col("t_idx").hash(SEED) % 100 < CE_DENSE_NEG_PCT))
    dd = attach_text(pl.concat([dpos] * CE_DENSE_POS_REPEAT + [dneg]).select(*KEYS, "y"), "train")
    dd = dd.select("a", "b", pl.col("y").cast(pl.Float32).alias("label"))
    df = pl.concat([base_tr, dd]).sample(fraction=1.0, shuffle=True, seed=SEED)
    n_hold = max(1, int(0.01 * df.height))
    hold, df = df.head(n_hold), df.slice(n_hold)
    log("CE-u train rows:", df.height, "(base", base_tr.height, "+ dense", dd.height, ") pos rate", round(float(df["label"].mean()), 4))

    tok = AutoTokenizer.from_pretrained(CE_BASE)
    model = AutoModelForSequenceClassification.from_pretrained(CE_BASE, num_labels=1).to(DEV)
    A, B, Y = df["a"].to_list(), df["b"].to_list(), df["label"].to_numpy()
    steps = CE_EPOCHS * (len(A) // CE_BS)
    opt = torch.optim.AdamW(model.parameters(), lr=CE_LR, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    bce = torch.nn.BCEWithLogitsLoss()

    def ce_logits(m, a, b):
        e = tok(a, b, truncation=True, max_length=CE_MAXLEN, padding=True, return_tensors="pt").to(DEV)
        return m(**e).logits.float().squeeze(-1)

    def ce_hold():
        model.eval()
        ps = []
        with torch.no_grad():
            for i in range(0, hold.height, 256):
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
                    ps.append(torch.sigmoid(ce_logits(model, hold["a"][i:i + 256].to_list(), hold["b"][i:i + 256].to_list())).cpu().numpy())
        model.train()
        p, y = np.concatenate(ps), hold["label"].to_numpy()
        return {"logloss": float(-np.mean(y * np.log(np.clip(p, 1e-7, 1)) + (1 - y) * np.log(np.clip(1 - p, 1e-7, 1)))),
                "acc": float(((p > 0.5) == (y > 0.5)).mean())}

    model.train()
    rng = np.random.default_rng(SEED)
    step = 0
    for ep in range(CE_EPOCHS):
        perm = rng.permutation(len(A))
        for i in range(len(A) // CE_BS):
            ix = perm[i * CE_BS:(i + 1) * CE_BS]
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
                lo = ce_logits(model, [A[j] for j in ix], [B[j] for j in ix])
            loss = bce(lo, torch.as_tensor(Y[ix], device=DEV))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sch.step()
            step += 1
            if step % 2000 == 0 or step == 1:
                log(f"CE ep {ep} step {step}/{steps} loss {loss.item():.4f}")
            if step % 20000 == 0:
                log("CE hold", ce_hold())
    log("CE hold final", ce_hold())
    CE_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(CE_DIR)
    tok.save_pretrained(CE_DIR)
    del model, opt, df, A, B
    gc.collect()
    if DEV == "cuda":
        torch.cuda.empty_cache()
    log("CE-u saved to", CE_DIR)

## 6. Score every re-ranker row with CE-u
Rows per part = base candidates with stage-2 p ≥ `P_MIN` ∪ kept dense pairs. Scored in length-sorted batches;
1M-row chunks are checkpointed, so an interrupted run resumes.

In [ ]:
def rr_rows(part):
    b = BASE_P[part].filter(pl.col("p") >= P_MIN).select(KEYS)
    return pl.concat([b, pl.read_parquet(DENSE_OUT[part]).select(KEYS)]).unique()


CE_OUT = {p: NB / "ce" / f"ce_{p}.parquet" for p in ("fit1", "val", "test")}
if not done(*CE_OUT.values()):
    ctok = AutoTokenizer.from_pretrained(CE_DIR)
    cmodel = AutoModelForSequenceClassification.from_pretrained(CE_DIR).to(DEV).eval()
    if DEV == "cuda":
        cmodel = cmodel.half()
    for part in ("val", "fit1", "test"):
        if CE_OUT[part].exists():
            continue
        d = attach_text(rr_rows(part), SPLIT[part])
        d = d.with_columns((pl.col("a").str.len_chars() + pl.col("b").str.len_chars()).alias("_l")).sort("_l").drop("_l")
        CH = 1_000_000
        chunks = []
        for c0 in range(0, d.height, CH):
            cp = NB / "ce" / f"ce_{part}_chunk{c0 // CH:04d}.parquet"
            if not cp.exists():
                x = d.slice(c0, CH)
                A, B = x["a"].to_list(), x["b"].to_list()
                sc = []
                with torch.no_grad():
                    for i in range(0, len(A), 512):
                        e = ctok(A[i:i + 512], B[i:i + 512], truncation=True, max_length=CE_MAXLEN, padding=True,
                                 return_tensors="pt").to(DEV)
                        sc.append(torch.sigmoid(cmodel(**e).logits.float().squeeze(-1)).cpu().numpy())
                save(x.select(KEYS).with_columns(pl.Series("ce", np.concatenate(sc).astype(np.float32))), cp)
            chunks.append(cp)
            log("CE scored", part, min(c0 + CH, d.height), "/", d.height)
        save(pl.concat([pl.read_parquet(c) for c in chunks]), CE_OUT[part])
        for c in chunks:
            c.unlink()
    del cmodel
    gc.collect()
    if DEV == "cuda":
        torch.cuda.empty_cache()
log("CE scores:", {p: pl.scan_parquet(f).select(pl.len()).collect().item() for p, f in CE_OUT.items()})

## 7. Re-ranker features (base ∪ dense, per country)
* pair: stage-2 p / logit, CE-u, bi cosine, dense rank / gap, origin
* lexical: name token-set / ratio / partial, address token-set, **street without numbers**, house-number status,
  length ratio, missing address / number flags
* per-S1 context: counts, CE / p / cosine rank and gap within (S1, source), best CE in the other source, CE mass
* per-target competition: number of competing S1, CE / p / cosine lead over the best other S1
* **anchor**: the S1's most confident *other* candidate — does this target share its street / house number / name?

In [ ]:
from rapidfuzz import fuzz, process           # noqa: E402
from rapidfuzz.distance import Levenshtein    # noqa: E402

FEATS = ["is_dense", "p0", "lp", "ce", "lce", "cos", "d_rank", "d_gap1", "src",
         "lx_name", "lx_name_r", "lx_name_p", "lx_addr", "lx_street", "lx_hno", "lx_lenr", "lx_a1null", "lx_a2null",
         "cx_n", "cx_n_src", "cx_n_dense", "cx_ce_rank", "cx_lp_rank", "cx_cos_rank", "cx_ce_gap", "cx_cos_gap",
         "cx_ce_sum", "cx_ce_hi", "cx_ce_oth",
         "tx_n", "tx_ce_lead", "tx_lp_lead", "tx_cos_lead",
         "an_ce", "an_same_src", "an_addr", "an_street", "an_hno", "an_name"]


def _cp(scorer, a, b, dtype=np.float32):
    return process.cpdist(a, b, scorer=scorer, workers=-1, dtype=dtype)


def lead(c):
    """winner: lead over the runner-up S1 for the same target; loser: (negative) deficit to the winner;
    sole candidate: the value itself (same definition as er.cross ce_gap_t)."""
    return (pl.when(pl.len().over("src", "t_idx") == 1).then(pl.col(c))
              .when(pl.col(c) >= pl.col(c).max().over("src", "t_idx"))
              .then(pl.col(c) - pl.col(c).top_k(2).min().over("src", "t_idx"))
              .otherwise(pl.col(c) - pl.col(c).max().over("src", "t_idx")))


def build_features(part):
    out = NB / "feat" / f"feat_{part}.parquet"
    if out.exists():
        return pl.read_parquet(out)
    split = SPLIT[part]
    base = BASE_P[part].filter(pl.col("p") >= P_MIN).with_columns(pl.lit(0, pl.Int8).alias("is_dense"))
    dense = pl.read_parquet(DENSE_OUT[part]).select(*KEYS, "cos", pl.col("rank").cast(pl.Float32).alias("d_rank"),
                                                    pl.col("gap1").alias("d_gap1")).with_columns(pl.lit(1, pl.Int8).alias("is_dense"))
    dense = dense.join(base.select(KEYS), on=KEYS, how="anti")
    bcos = pl.read_parquet(NB / "dense" / f"basecos_{split}.parquet").select(*KEYS, "cos")
    base = base.join(bcos, on=KEYS, how="left")
    u = pl.concat([base.rename({"p": "p0"}), dense], how="diagonal_relaxed")
    u = u.join(pl.read_parquet(CE_OUT[part]), on=KEYS, how="left").join(PARTS[part], on="s1_idx", how="left")
    s1t = text_table(split, 1).select(pl.col("idx").alias("s1_idx"), pl.col("n").alias("n1"), pl.col("a").alias("a1"),
                                      pl.col("h").alias("h1"), pl.col("st").alias("st1"))
    tgt = pl.concat([text_table(split, s).select(pl.col("idx").alias("t_idx"), pl.lit(s, pl.UInt8).alias("src"),
                                                 "n", "a", "h", "st") for s in (2, 3)])
    res = []
    for ctry in COUNTRIES:
        x = u.filter(pl.col("country") == ctry)
        if x.height == 0:
            continue
        x = (x.join(s1t, on="s1_idx", how="left").join(tgt.rename({"n": "n2", "a": "a2", "h": "h2", "st": "st2"}),
                                                        on=["t_idx", "src"], how="left")
               .with_columns(pl.col("n1", "a1", "h1", "st1", "n2", "a2", "h2", "st2").fill_null("")))
        n1, n2, a1, a2 = x["n1"].to_list(), x["n2"].to_list(), x["a1"].to_list(), x["a2"].to_list()
        x = x.with_columns(
            pl.Series("lx_name", _cp(fuzz.token_set_ratio, n1, n2)), pl.Series("lx_name_r", _cp(fuzz.ratio, n1, n2)),
            pl.Series("lx_name_p", _cp(fuzz.partial_ratio, n1, n2)), pl.Series("lx_addr", _cp(fuzz.token_set_ratio, a1, a2)),
            pl.Series("lx_street", _cp(fuzz.token_set_ratio, x["st1"].to_list(), x["st2"].to_list())),
            pl.Series("_hed", _cp(Levenshtein.distance, x["h1"].to_list(), x["h2"].to_list(), np.int32)))
        del n1, n2, a1, a2
        x = x.with_columns(
            pl.when((pl.col("h1") == "") & (pl.col("h2") == "")).then(0).when((pl.col("h1") == "") | (pl.col("h2") == "")).then(1)
              .when(pl.col("_hed") == 0).then(2).when(pl.col("_hed") == 1).then(3).otherwise(4).cast(pl.Float32).alias("lx_hno"),
            (pl.min_horizontal(pl.col("n1").str.len_chars(), pl.col("n2").str.len_chars())
             / pl.max_horizontal(pl.col("n1").str.len_chars(), pl.col("n2").str.len_chars(), pl.lit(1))).cast(pl.Float32).alias("lx_lenr"),
            (pl.col("a1") == "").cast(pl.Float32).alias("lx_a1null"), (pl.col("a2") == "").cast(pl.Float32).alias("lx_a2null"),
            logit("p0").alias("lp"), logit("ce").alias("lce"))
        # per-S1 context and per-target competition (every S1 and target of this country is in x)
        x = x.with_columns(
            pl.len().over("s1_idx").cast(pl.Float32).alias("cx_n"),
            pl.len().over("s1_idx", "src").cast(pl.Float32).alias("cx_n_src"),
            pl.col("is_dense").sum().over("s1_idx").cast(pl.Float32).alias("cx_n_dense"),
            pl.col("ce").rank("ordinal", descending=True).over("s1_idx", "src").cast(pl.Float32).alias("cx_ce_rank"),
            pl.col("lp").rank("ordinal", descending=True).over("s1_idx", "src").cast(pl.Float32).alias("cx_lp_rank"),
            pl.col("cos").rank("ordinal", descending=True).over("s1_idx", "src").cast(pl.Float32).alias("cx_cos_rank"),
            (pl.col("ce") - pl.col("ce").max().over("s1_idx", "src")).alias("cx_ce_gap"),
            (pl.col("cos") - pl.col("cos").max().over("s1_idx", "src")).alias("cx_cos_gap"),
            pl.col("ce").sum().over("s1_idx").alias("cx_ce_sum"),
            (pl.col("ce") >= 0.5).sum().over("s1_idx").cast(pl.Float32).alias("cx_ce_hi"),
            pl.len().over("src", "t_idx").cast(pl.Float32).alias("tx_n"),
            lead("ce").alias("tx_ce_lead"), lead("lp").alias("tx_lp_lead"), lead("cos").alias("tx_cos_lead"))
        oth = x.group_by("s1_idx", "src").agg(pl.col("ce").max().alias("cx_ce_oth")).with_columns((5 - pl.col("src")).cast(pl.UInt8).alias("src"))
        x = x.join(oth, on=["s1_idx", "src"], how="left")
        # anchor: the S1's highest-CE candidate other than the pair itself (top-2 per S1)
        top = (x.select("s1_idx", "src", "t_idx", "ce", "n2", "a2", "h2", "st2").sort(["s1_idx", "ce"], descending=[False, True])
                .group_by("s1_idx", maintain_order=True).head(2).with_columns(pl.int_range(pl.len()).over("s1_idx").alias("_k")))
        ren = lambda k: top.filter(pl.col("_k") == k).drop("_k").rename({c: f"{c}_{k}" for c in top.columns if c not in ("s1_idx", "_k")})
        x = x.join(ren(0), on="s1_idx", how="left").join(ren(1), on="s1_idx", how="left")
        self0 = (pl.col("src") == pl.col("src_0")) & (pl.col("t_idx") == pl.col("t_idx_0"))
        pick = lambda c: pl.when(self0).then(pl.col(f"{c}_1")).otherwise(pl.col(f"{c}_0"))
        x = x.with_columns(pick("ce").alias("an_ce"), (pick("src") == pl.col("src")).cast(pl.Float32).alias("an_same_src"),
                           pick("n2").fill_null("").alias("_an_n"), pick("a2").fill_null("").alias("_an_a"),
                           pick("h2").fill_null("").alias("_an_h"), pick("st2").fill_null("").alias("_an_st"))
        has = x["an_ce"].is_not_null().to_numpy()
        f_an = {}
        for name, c_me, c_an in (("an_addr", "a2", "_an_a"), ("an_street", "st2", "_an_st"), ("an_name", "n2", "_an_n")):
            v = _cp(fuzz.token_set_ratio, x[c_me].to_list(), x[c_an].to_list())
            v[~has] = np.nan
            f_an[name] = v
        x = x.with_columns(*[pl.Series(k, v) for k, v in f_an.items()],
                           pl.when(pl.col("an_ce").is_null() | (pl.col("h2") == "") | (pl.col("_an_h") == "")).then(None)
                             .otherwise((pl.col("h2") == pl.col("_an_h")).cast(pl.Float32)).alias("an_hno"))
        res.append(x.select(*KEYS, "country", *[pl.col(c).cast(pl.Float32) for c in FEATS if c != "src"]))
        del x, top
        gc.collect()
    d = pl.concat(res)
    save(d, out)
    return d


for part in ("fit1", "val", "test"):
    build_features(part)
    log("features", part, pl.scan_parquet(NB / "feat" / f"feat_{part}.parquet").select(pl.len()).collect().item())

## 8. Re-ranker (LightGBM) — 2-fold cross-fit on FIT fold 1
Out-of-fold p on fit1 (used by the empty model), mean of the two fold models on val / test. Each fold model
early-stops on a 10% S1 slice of its own training fold (the OOF fold is never touched).

In [ ]:
import lightgbm as lgb    # noqa: E402

RR_PARAMS = dict(objective="binary", learning_rate=0.03 if not SMOKE else 0.1, num_leaves=127, min_data_in_leaf=100,
                 feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, max_bin=255,
                 verbose=-1, seed=SEED, num_threads=NCPU)
RR_OUT = {p: NB / "rr" / f"rr_{p}.parquet" for p in ("fit1", "val", "test")}


def x_of(d):
    return d.select([pl.col(c).cast(pl.Float32) for c in FEATS]).to_numpy()


if not done(*RR_OUT.values()):
    tr = build_features("fit1").join(LAB, on=KEYS, how="left").with_columns(pl.col("y").fill_null(0))
    h = (pl.col("s1_idx").cast(pl.UInt64) * 2654435761) % 1000
    tr = tr.with_columns((h < 500).cast(pl.Int8).alias("_fold"), (((h // 7) % 10) == 0).alias("_es"))
    log("re-ranker train rows:", tr.height, "positives:", int(tr["y"].sum()), "features:", len(FEATS))
    oof = np.zeros(tr.height, dtype=np.float32)
    models = []
    fold = tr["_fold"].to_numpy()
    es = tr["_es"].to_numpy()
    X, y = x_of(tr), tr["y"].to_numpy()
    for f in (0, 1):
        trn = (fold != f) & ~es
        vld = (fold != f) & es
        dtr = lgb.Dataset(X[trn], y[trn], feature_name=FEATS, free_raw_data=True)
        dva = lgb.Dataset(X[vld], y[vld], reference=dtr)
        m = lgb.train(RR_PARAMS, dtr, RR_ROUNDS, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(RR_EARLY, verbose=False), lgb.log_evaluation(500)])
        oof[fold == f] = m.predict(X[fold == f], num_iteration=m.best_iteration)
        m.save_model(str(NB / "rr" / f"rr_f{f}.lgb"))
        models.append(m)
        log(f"re-ranker fold {f}: best_iter {m.best_iteration}, OOF logloss",
            round(float(-np.mean(y[fold == f] * np.log(np.clip(oof[fold == f], 1e-7, 1))
                                  + (1 - y[fold == f]) * np.log(np.clip(1 - oof[fold == f], 1e-7, 1)))), 5))
    save(tr.select(KEYS).with_columns(pl.Series("p", oof)), RR_OUT["fit1"])
    imp = sorted(zip(FEATS, np.mean([m.feature_importance("gain") for m in models], 0)), key=lambda t: -t[1])
    print("top features:", [(k, round(v / 1e3)) for k, v in imp[:20]])
    del X, tr
    gc.collect()
    for part in ("val", "test"):
        d = build_features(part)
        Xp = x_of(d)
        p = np.mean([m.predict(Xp, num_iteration=m.best_iteration) for m in models], 0).astype(np.float32)
        save(d.select(KEYS).with_columns(pl.Series("p", p)), RR_OUT[part])
        del d, Xp
        gc.collect()
log("re-ranker scores ready")


def scores_rr(part):
    """B1 scores: re-ranker p on re-scored rows; base candidates below P_MIN keep their stage-2 p."""
    rr = pl.read_parquet(RR_OUT[part])
    low = BASE_P[part].filter(pl.col("p") < P_MIN).join(rr.select(KEYS), on=KEYS, how="anti")
    return cast(pl.concat([cast(rr), cast(low.select(*KEYS, "p"))]))

## 9. Per-S1 "no match" model (variant B2)
Trained on fit1 S1 aggregates of the **out-of-fold** re-ranker p; label = the S1 has no true match. Its output
replaces `∏(1-p)` as P(empty) in the expected-F0.5 decision.

In [ ]:
Q0_OUT = {p: NB / "rr" / f"q0_{p}.parquet" for p in ("val", "test")}


def s1_aggs(part, sc):
    f = build_features(part).select(*KEYS, "cos", "ce", "lx_a1null", "is_dense")
    d = sc.join(f, on=KEYS, how="left").sort(["s1_idx", "p"], descending=[False, True])
    g = d.group_by("s1_idx").agg(
        pl.len().cast(pl.Float32).alias("e_n"), pl.col("p").max().alias("e_pmax"),
        pl.col("p").top_k(2).min().alias("e_p2"), pl.col("p").sum().alias("e_psum"),
        (pl.col("p") >= 0.5).sum().cast(pl.Float32).alias("e_n50"), (pl.col("p") >= 0.2).sum().cast(pl.Float32).alias("e_n20"),
        (1 - pl.col("p").clip(0, 1 - 1e-6)).log().sum().alias("e_lp0"), pl.col("cos").max().alias("e_cosmax"),
        pl.col("ce").max().alias("e_cemax"), pl.col("lx_a1null").max().alias("e_a1null"),
        pl.col("is_dense").sum().cast(pl.Float32).alias("e_ndense"))
    return g


E_FEATS = ["e_n", "e_pmax", "e_p2", "e_psum", "e_n50", "e_n20", "e_lp0", "e_cosmax", "e_cemax", "e_a1null", "e_ndense"]
if not done(*Q0_OUT.values()):
    g = s1_aggs("fit1", scores_rr("fit1"))     # out-of-fold re-ranker p on fit1
    has = LAB.join(PARTS["fit1"].select("s1_idx"), on="s1_idx", how="semi").select("s1_idx").unique().with_columns(pl.lit(0, pl.Int8).alias("ye"))
    g = g.join(has, on="s1_idx", how="left").with_columns(pl.col("ye").fill_null(1))
    es = (g["s1_idx"].cast(pl.UInt64).to_numpy() * 2654435761) % 1000 < 100     # 10% of S1: early stopping
    Xe, ye = g.select([pl.col(c).cast(pl.Float32) for c in E_FEATS]).to_numpy(), g["ye"].to_numpy()
    em = lgb.train(dict(objective="binary", learning_rate=0.03, num_leaves=31, min_data_in_leaf=200, feature_fraction=0.9,
                        bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, seed=SEED, num_threads=NCPU),
                   lgb.Dataset(Xe[~es], ye[~es]), 2000 if not SMOKE else 100,
                   valid_sets=[lgb.Dataset(Xe[es], ye[es])], callbacks=[lgb.early_stopping(100, verbose=False)])
    em.save_model(str(NB / "rr" / "empty.lgb"))
    log("empty model: fit1 S1", g.height, "empty rate", round(float(ye.mean()), 4), "best_iter", em.best_iteration)
    for part in ("val", "test"):
        gp = s1_aggs(part, scores_rr(part))
        q = em.predict(gp.select([pl.col(c).cast(pl.Float32) for c in E_FEATS]).to_numpy(), num_iteration=em.best_iteration)
        save(gp.select("s1_idx").with_columns(pl.Series("q0", q.astype(np.float32))), Q0_OUT[part])

## 10. Baselines: v6cd rebuilt here (v6ce ∪ dense pairs scored by the 6-feature dense stacker) and, if provided,
the exact submitted v6cd

In [ ]:
B0_OUT = {p: NB / "rr" / f"b0_{p}.parquet" for p in ("val", "test")}
if not done(*B0_OUT.values()):
    lce_ = logit("ce").alias("lce")

    def dense_scored(part):
        return (pl.read_parquet(DENSE_OUT[part]).join(pl.read_parquet(CE_OUT[part]), on=KEYS, how="left")
                  .with_columns(lce_, pl.col("src").cast(pl.Float32).alias("srcf")))
    DF = ["cos", "rank", "gap1", "ce", "lce", "srcf"]
    tr = dense_scored("fit1")
    ds = lgb.train(dict(objective="binary", learning_rate=0.05, num_leaves=31, min_data_in_leaf=100, feature_fraction=0.9,
                        bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, seed=42, num_threads=NCPU),
                   lgb.Dataset(tr.select(DF).to_numpy(), tr["y"].to_numpy()), 300)
    for part in ("val", "test"):
        base = cast(pl.read_parquet(WORK / f"runs/v6ce/{part}_scores_ce.parquet").select(*KEYS, "p")).join(
            PARTS[part].select("s1_idx"), on="s1_idx", how="semi")
        d = dense_scored(part)
        d = d.select(KEYS).with_columns(pl.Series("p", ds.predict(d.select(DF).to_numpy()).astype(np.float32)))
        save(pl.concat([base, cast(d).join(base, on=KEYS, how="anti")]), B0_OUT[part])


def scores(model, part):
    if model == "v6cd_submitted":
        return cast(pl.read_parquet(WORK / f"runs/v6cd/{part}_scores_ce.parquet").select(*KEYS, "p")).join(
            PARTS[part].select("s1_idx"), on="s1_idx", how="semi")
    if model == "v6cd_rebuilt":
        return cast(pl.read_parquet(B0_OUT[part]))
    return scores_rr(part)   # "reranker", "reranker+empty"


MODELS = (["v6cd_submitted"] if HAVE_V6CD else []) + ["v6cd_rebuilt", "reranker", "reranker+empty"]
BASELINE = MODELS[0]
log("models:", MODELS, "| baseline:", BASELINE)

## 11. Decision grid on val, held-out-half gate, choice
The expected-F0.5 set selection of `er.decide.select` (exclusivity + one-to-one coherent posterior), with an optional
per-S1 P(empty). Every grid point is scored per S1 on the whole val set (targets compete across all S1, as on test);
the held-out estimate picks the grid point on one half of val S1 and scores the other half.

In [ ]:
def select_q(df, prm, q0=None):
    """er.decide.select, plus: if q0 (s1_idx, q0) is given, P(no match) = q0 instead of prod(1-p)."""
    d = coherent(df.select("s1_idx", "src", "t_idx", "p"), prm.coh)
    d = d.with_columns(
        pl.col("p").rank("ordinal", descending=True).over("src", "t_idx").alias("_tr"),
        pl.col("p").top_k(2).min().over("src", "t_idx").alias("_p2"),
        pl.len().over("src", "t_idx").alias("_tn"))
    lead_ = pl.when(pl.col("_tn") > 1).then(pl.col("p") - pl.col("_p2")).otherwise(pl.col("p"))
    d = d.with_columns(((pl.col("_tr") == 1) & (lead_ >= prm.margin)).alias("_ok"))
    d = d.with_columns(pl.when(pl.col("_ok")).then(pl.col("p")).otherwise(0.0).alias("_q"))
    d = d.sort(["s1_idx", "_q"], descending=[False, True]).with_columns(
        pl.col("_q").cum_sum().over("s1_idx").alias("_cs"),
        pl.int_range(1, pl.len() + 1).over("s1_idx").alias("_k"),
        pl.col("p").sum().over("s1_idx").alias("_tot"),
        (1 - pl.col("p")).log().sum().over("s1_idx").alias("_lp0"))
    d = d.with_columns(((1 + 0.25) * pl.col("_cs") / (0.25 * (pl.col("_tot") + prm.m0) + pl.col("_k"))).alias("_ef"))
    best = d.group_by("s1_idx").agg(pl.col("_ef").max().alias("_efmax"), pl.col("_k").get(pl.col("_ef").arg_max()).alias("_kbest"),
                                    pl.col("_lp0").first().exp().alias("_pe"))
    if q0 is not None:
        best = best.join(q0, on="s1_idx", how="left").with_columns(pl.coalesce("q0", "_pe").alias("_pe"))
    best = best.with_columns((pl.col("_pe") * prm.empty_bias).alias("_ef0"))
    d = d.join(best.select("s1_idx", "_efmax", "_kbest", "_ef0"), on="s1_idx").filter(
        (pl.col("_efmax") > pl.col("_ef0")) & (pl.col("_k") <= pl.col("_kbest")) & pl.col("_ok"))
    return d.select("s1_idx", "src", "t_idx")


VAL_S1 = PARTS["val"].sort("s1_idx").with_columns(
    ((((pl.col("s1_idx").cast(pl.UInt64) * 2654435761) % 1000) < 500)).cast(pl.Int8).alias("half"))
VAL_TRUTH = LAB.join(VAL_S1.select("s1_idx"), on="s1_idx", how="semi").select(KEYS)
VAL_NT = VAL_S1.select("s1_idx").join(VAL_TRUTH.group_by("s1_idx").len("nt"), on="s1_idx", how="left").fill_null(0)["nt"].to_numpy()


def per_s1_f(sel):
    """macro-F0.5 terms aligned with VAL_S1 (exact challenge rules: empty/empty = 1)."""
    npred = VAL_S1.select("s1_idx").join(sel.group_by("s1_idx").len("np"), on="s1_idx", how="left").fill_null(0)["np"].to_numpy()
    tp = VAL_S1.select("s1_idx").join(sel.join(VAL_TRUTH, on=KEYS).group_by("s1_idx").len("tp"), on="s1_idx", how="left").fill_null(0)["tp"].to_numpy()
    den = 0.25 * VAL_NT + npred
    f = np.where(den > 0, 1.25 * tp / np.maximum(den, 1e-9), 1.0)
    return f.astype(np.float32)


GRID = [DecisionParams(margin=mg, m0=m0, empty_bias=eb, coh=coh)
        for coh in (0.5, 1.0) for mg in (0.0, 0.05, 0.1) for m0 in (0.25, 0.5, 0.75, 1.0) for eb in (1.5, 2.0, 2.5, 3.0)]
if SMOKE:
    GRID = GRID[::8]
COUNTRY_V = VAL_S1["country"].to_numpy()
HALF = VAL_S1["half"].to_numpy()
EVAL = {}
for mname in MODELS:
    ef = NB / "eval" / f"grid_{mname}.npy"
    if ef.exists():
        Fm = np.load(ef)
    else:
        sv = scores(mname, "val")
        q0 = pl.read_parquet(Q0_OUT["val"]) if mname == "reranker+empty" else None
        Fm = np.stack([per_s1_f(select_q(sv, g, q0)) for g in GRID])
        np.save(ef, Fm)
    full = Fm.mean(1)
    gbest = int(full.argmax())
    held = np.empty(Fm.shape[1], dtype=np.float32)
    for h in (0, 1):
        gh = int(Fm[:, HALF != h].mean(1).argmax())
        held[HALF == h] = Fm[gh, HALF == h]
    EVAL[mname] = {"F": Fm, "gbest": gbest, "held": held}
    print(f"{mname:>16}: val F0.5 (tuned on all val) {full[gbest]:.5f} "
          + " ".join(f"{c} {Fm[gbest, COUNTRY_V == c].mean():.5f}" for c in COUNTRIES)
          + f" | held-out {held.mean():.5f} | params {GRID[gbest]}", flush=True)

In [ ]:
def boot(dlt, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    bs = np.array([dlt[rng.integers(0, len(dlt), len(dlt))].mean() for _ in range(n)])
    return float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


GATE = {}
for mname in MODELS:
    if mname == BASELINE:
        continue
    dlt = EVAL[mname]["held"] - EVAL[BASELINE]["held"]
    lo, hi = boot(dlt)
    byc = {c: float(dlt[COUNTRY_V == c].mean()) for c in COUNTRIES}
    GATE[mname] = {"delta": float(dlt.mean()), "ci95": [lo, hi], "by_country": byc,
                   "accept": lo > 0 and all(v >= 0 for v in byc.values())}
    print(f"gate {mname} vs {BASELINE}: {GATE[mname]}", flush=True)
acc = [m for m in GATE if GATE[m]["accept"]]
CHOSEN = max(acc, key=lambda m: GATE[m]["delta"]) if acc else BASELINE
log("CHOSEN:", CHOSEN)
json.dump({"models": {m: {"val_f05": float(EVAL[m]["F"][EVAL[m]["gbest"]].mean()), "heldout_f05": float(EVAL[m]["held"].mean()),
                          "by_country": {c: float(EVAL[m]["F"][EVAL[m]["gbest"], COUNTRY_V == c].mean()) for c in COUNTRIES},
                          "params": GRID[EVAL[m]["gbest"]].__dict__} for m in MODELS},
           "gate": GATE, "baseline": BASELINE, "chosen": CHOSEN}, open(NB / "eval" / "report.json", "w"), indent=1)

## 12. Test predictions → submission (US/India from the chosen model, France verbatim from the LB-0.988 file)

In [ ]:
def read_sub(path, col):
    raw = open(path, "rb").read()
    if raw[:2] == b"PK":
        z = zipfile.ZipFile(io.BytesIO(raw))
        raw = z.read(next(n for n in z.namelist() if n.endswith(".tsv")))
    d = pl.read_csv(io.BytesIO(raw), separator="\t", quote_char=None, infer_schema=False)
    d = d.rename({d.columns[0]: "s1", d.columns[1]: col})
    return d.with_columns(pl.col("s1", col).fill_null("").str.replace_all('"', "").str.strip_chars())


def explode_ids(d, col):
    return (d.with_columns(pl.col(col).str.split(",")).explode(col).filter(pl.col(col) != "")
             .select("s1", pl.col(col).alias("tid")))


prm = GRID[EVAL[CHOSEN]["gbest"]]
st = scores(CHOSEN, "test")
q0t = pl.read_parquet(Q0_OUT["test"]) if CHOSEN == "reranker+empty" else None
sel = select_q(st, prm, q0t)
s1_ids, tg_ids = P.id_maps("test")
s1_ids = s1_ids.with_columns(pl.col("s1_idx").cast(pl.UInt32))
tg_ids = tg_ids.with_columns(pl.col("t_idx").cast(pl.UInt32), pl.col("src").cast(pl.UInt8))
to_ids = lambda x: x.join(s1_ids.select("s1_idx", "s1"), on="s1_idx").join(tg_ids, on=["t_idx", "src"]).select("s1", "tid")
m_usin = to_ids(sel)
fr_s1 = TEST_S1_ALL.filter(pl.col("country") == "France").select(pl.col("entity_id").alias("s1"))
if FR_0988.exists():
    old = read_sub(FR_0988, "m")
    m_fr = explode_ids(old.join(fr_s1, on="s1", how="semi"), "m")
else:
    old = None
    fr_sc = cast(pl.read_parquet(WORK / "runs/v6ce/test_scores_ce.parquet").select(*KEYS, "p")).join(
        TEST_S1_ALL.filter(pl.col("country") == "France").select("s1_idx"), on="s1_idx", how="semi")
    m_fr = to_ids(select_q(fr_sc, prm))
matches = pl.concat([m_usin, m_fr]).unique()
cands = pl.concat([to_ids(cast(pl.read_parquet(WORK / "runs/v6ce/test_scores_ce.parquet").select(KEYS))),
                   to_ids(pl.read_parquet(DENSE_OUT["test"]).select(KEYS)), matches]).unique()
all_s1 = TEST_S1_ALL["entity_id"].alias("s1")
M = id_lists(matches, all_s1, "matched_entity_ids")
Cd = id_lists(cands, all_s1, "candidate_entity_ids")
check_invariants(M, Cd, all_s1)
OUT = NB / "sub" / f"SUB_{CHOSEN.replace('+', '_')}"
OUT.mkdir(parents=True, exist_ok=True)
M.write_csv(OUT / "matching_results.tsv", separator="\t", quote_style="never")
Cd.write_csv(OUT / "candidate_pairs.tsv", separator="\t", quote_style="never")
if VALIDATOR.exists():
    r = subprocess.run([sys.executable, str(VALIDATOR), "--matching", str(OUT / "matching_results.tsv"),
                        "--candidate", str(OUT / "candidate_pairs.tsv"), "--test-dir", str(DATA_DIR / "test")],
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(r.stdout[-1500:], r.stderr[-1500:])
    assert "PASS" in r.stdout, "official validator did not PASS"
for f in ("matching_results", "candidate_pairs"):
    with zipfile.ZipFile(OUT / f"{f}.zip", "w", zipfile.ZIP_DEFLATED) as zz:
        zz.write(OUT / f"{f}.tsv", f"{f}.tsv")

# ---- report: change vs the 0.988 file and a projected LB -----------------------------------------------------------
rep = {"chosen": CHOSEN, "params": prm.__dict__, "test_pairs": matches.height}
cnt = TEST_S1_ALL.group_by("country").len("n")
N = {c: int(n) for c, n in cnt.iter_rows()}
rep["test_s1_by_country"] = N
if old is not None:
    po = explode_ids(old, "m")
    rep["added_vs_0988"] = matches.join(po, on=["s1", "tid"], how="anti").height
    rep["removed_vs_0988"] = po.join(matches, on=["s1", "tid"], how="anti").height
vb = {c: float(EVAL[BASELINE]["F"][EVAL[BASELINE]["gbest"], COUNTRY_V == c].mean()) for c in COUNTRIES}
vn = {c: float(EVAL[CHOSEN]["held"][COUNTRY_V == c].mean()) + (vb[c] - float(EVAL[BASELINE]["held"][COUNTRY_V == c].mean())) for c in COUNTRIES}
n_us = sum(N.get(c, 0) for c in COUNTRIES)
n_fr = sum(N.values()) - n_us
if n_fr:
    proj = {}
    for lb in (0.9875, 0.988, 0.9885):      # the LB shows 3 decimals: 0.988 means [0.9875, 0.9885)
        f_fr = (lb * sum(N.values()) - sum(vb[c] * N.get(c, 0) for c in COUNTRIES)) / n_fr
        proj[str(lb)] = {"implied_france": round(f_fr, 5),
                         "projected_lb": round((sum(vn[c] * N.get(c, 0) for c in COUNTRIES) + f_fr * n_fr) / sum(N.values()), 5)}
    rep["lb_projection (assumes test US/India behave like val)"] = proj
    if not all(0 <= v["implied_france"] <= 1 for v in proj.values()):
        print("NOTE: implied France score outside [0, 1] -> test US/India do not behave like val; ignore the projection.")
json.dump(rep, open(OUT / "report.json", "w"), indent=1)
print(json.dumps(rep, indent=1))
log("SUBMISSION READY:", OUT / "matching_results.zip", OUT / "candidate_pairs.zip")